# P05 on PolyBench with a T4 GPU (Phase 3.2)

Runs on **Colab** or **Kaggle** (it works in `/content` or `/kaggle/working`, whichever exists).

- **Colab:** Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. Keep the
  tab open; if the session disconnects, reconnect and Run all again: the run resumes from the
  rows already in the CSV (unless the runtime was reset).
- **Kaggle:** File → Import Notebook (this file). In the right-hand panel, Session options:
  **Accelerator → GPU T4 x2** and **Internet on** (both need a phone-verified account). Then
  **Save Version → Save & Run All (Commit)**: it runs in the background (up to 12 hours) with
  the browser closed, and the CSV appears under the version's **Output**.

Installing the NVIDIA HPC SDK takes about 10 minutes; the benchmark run takes longer (30
kernels, 5 variants, 3 runs each at the LARGE size).

Variants per kernel: `seq` (nvc -O3, no OpenMP), `p05` (P05's own choices with the T4
machine model: GPU, host threads or sequential per loop), `p05-naive` (every loop P05 can
offload goes to the GPU), and both again as `-warm` (the device started before `main`, so
the timed kernel doesn't pay the ~0.16 s start-up). Correctness: exact hex dumps against
`seq`. `gpu_launches` counts kernel launches (0 = every guard chose the host).

**When it finishes, download `polybench-t4.csv` (Colab: the last cell offers it; Kaggle: the
version's Output) and give it to Claude**; it goes to `eval/polybench/results/t4-large.csv`.

Generated by `eval/polybench/make_gpu_notebook.py` at P05 9c975d5 (30
kernels, LARGE_DATASET, 37 GPU pragmas in the gated rewrites): edit that and `gpu_run.py`, not this
notebook.


In [ ]:
# Where to work: Colab's /content or Kaggle's /kaggle/working.
import os
ROOT = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content'
os.makedirs(ROOT, exist_ok=True)
os.environ['P05_ROOT'] = ROOT  # the %%bash cells cd here
os.chdir(ROOT)
open(os.path.join(ROOT, 'gpu_run.py'), 'w').write('#!/usr/bin/env python3\n"""Phase 3.2: P05 on PolyBench/C 4.2.1 with a GPU (Colab T4, nvc -mp=gpu).\n\n    python3 gpu_run.py --pb polybench --src rewrites [--runs 3] [--csv gpu.csv]\n\nSelf-contained (stdlib only): eval/polybench/make_gpu_notebook.py embeds it,\nand P05\'s rewrites, in eval/colab/polybench_t4.ipynb, since the notebook\ncan\'t clone the repository. --src holds P05\'s rewrites, made on the\ndevelopment machine with -machine machine/colab-t4.json:\n  src/gated/<kernel>.c   P05\'s own choices (GPU, CPU threads or sequential)\n  src/naive/<kernel>.c   -policy naive: every SAFE loop it can offload on the GPU\n\nPer kernel (LARGE_DATASET, -O3):\n  seq          the original, no OpenMP\n  p05          gated rewrite, OpenMP offload\n  p05-naive    naive rewrite\n  p05-warm, p05-naive-warm\n               the same, with an empty target region run before main (a\n               constructor in warm.c): the timed kernel no longer pays the\n               device\'s start-up (about 0.16 s on the T4, NOTES 2026-10-16),\n               which a long-running application pays once\nHost threads: OMP_NUM_THREADS = --threads (Colab: 2 vCPUs).\n\nCorrectness: a POLYBENCH_DUMP_ARRAYS build of each variant printing exact\nhex floats, compared with seq\'s ("exact" or the largest relative\ndifference). The same run counts GPU kernel launches (NVCOMPILER_ACC_NOTIFY;\nnvc only): 0 means every target region\'s if() chose the host.\n\nRows are appended to the CSV as they finish; a rerun skips rows already\nthere, so a disconnected Colab session resumes where it stopped.\n"""\nimport argparse\nimport csv\nimport os\nimport re\nimport shutil\nimport subprocess\nimport sys\n\nFIELDS = ["kernel", "variant", "threads", "status", "time_s", "times", "correct",\n          "max_rel_err", "gpu_launches", "target_pragmas", "cpu_pragmas", "detail"]\nVARIANTS = [("seq", "orig", False, False), ("p05", "gated", True, False),\n            ("p05-naive", "naive", True, False), ("p05-warm", "gated", True, True),\n            ("p05-naive-warm", "naive", True, True)]\nWARM_C = """#include <omp.h>\n__attribute__((constructor)) static void p05_warm(void) {\n  int x = 0;\n#pragma omp target map(tofrom: x)\n  x += 1;\n}\n"""\n\n\ndef run(cmd, env=None, timeout=None):\n    e = dict(os.environ)\n    e.update(env or {})\n    try:\n        r = subprocess.run(cmd, capture_output=True, text=True, env=e, timeout=timeout)\n        return r.returncode, r.stdout, r.stderr\n    except subprocess.TimeoutExpired:\n        return -1, "", "timeout"\n\n\ndef compare(ref, got):\n    """\'exact\', or the largest relative difference between the numbers."""\n    if ref == got:\n        return "exact", 0.0\n    a, b = ref.split(), got.split()\n    if len(a) != len(b):\n        return "differs", float("inf")\n    worst = 0.0\n    for x, y in zip(a, b):\n        if x == y:\n            continue\n        try:\n            fx, fy = float.fromhex(x), float.fromhex(y)\n        except ValueError:\n            try:\n                fx, fy = float(x), float(y)\n            except ValueError:\n                return "differs", float("inf")\n        if fx != fy:\n            if fx != fx or fy != fy:\n                return "differs", float("inf")\n            worst = max(worst, abs(fx - fy) / max(abs(fx), abs(fy), 1e-300))\n    return ("tolerance" if worst <= 1e-9 else "differs"), worst\n\n\nclass Bench:\n    def __init__(self, args):\n        self.a = args\n        self.pb = os.path.abspath(args.pb)\n        self.work = os.path.abspath(args.work)\n        os.makedirs(self.work, exist_ok=True)\n        self.warm = os.path.join(self.work, "warm.c")\n        open(self.warm, "w").write(WARM_C)\n        self.extra = []\n        # __INTPTR_TYPE__ appears in P05\'s guards; GCC and Clang predefine it.\n        probe = os.path.join(self.work, "probe.c")\n        open(probe, "w").write("int main(void) { return (int)(__INTPTR_TYPE__)0; }\\n")\n        if run([args.cc, probe, "-o", probe + ".out"])[0] != 0:\n            self.extra = ["-D__INTPTR_TYPE__=long"]\n\n    def kernels(self):\n        with open(os.path.join(self.pb, "utilities", "benchmark_list")) as f:\n            return [l.strip()[2:] for l in f if l.strip()]\n\n    def headers(self, kpath, exact):\n        """A copy of the kernel\'s headers in the work directory (exact: printing\n        hex floats); sources are compiled next to them, never in --pb."""\n        src = os.path.join(self.pb, os.path.dirname(kpath))\n        d = os.path.join(self.work, "hdr-hex" if exact else "hdr", os.path.basename(src))\n        if not os.path.isdir(d):\n            os.makedirs(d)\n            for f in os.listdir(src):\n                if f.endswith(".h"):\n                    t = open(os.path.join(src, f)).read()\n                    if exact:\n                        t = t.replace(\'"%0.2lf "\', \'"%a "\').replace(\'"%0.2f "\', \'"%a "\')\n                    open(os.path.join(d, f), "w").write(t)\n        return d\n\n    def build(self, name, kpath, src, omp, warm, exact):\n        d = os.path.join(self.work, "bin", os.path.basename(os.path.dirname(kpath)))\n        os.makedirs(d, exist_ok=True)\n        exe = os.path.join(d, name + ("-dump" if exact else ""))\n        # `#include "k.h"` searches the source\'s own directory first: compile\n        # a copy next to the headers wanted.\n        copy = os.path.join(self.headers(kpath, exact), f"{name}-{os.path.basename(src)}")\n        shutil.copy(src, copy)\n        defs = ["-D" + self.a.dataset,\n                "-DPOLYBENCH_DUMP_ARRAYS" if exact else "-DPOLYBENCH_TIME"]\n        flags = ["-O3"] + (self.a.omp_flags.split() if omp else []) + self.extra\n        cmd = ([self.a.cc] + flags + defs + ["-I", self.headers(kpath, exact), "-I",\n               os.path.join(self.pb, "utilities"), copy,\n               os.path.join(self.pb, "utilities", "polybench.c")]\n               + ([self.warm] if warm else []) + ["-lm", "-o", exe])\n        rc, _, se = run(cmd)\n        if rc != 0:\n            return None, (se.strip().splitlines() or ["build failed"])[-1]\n        return exe, ""\n\n    def env(self, omp):\n        e = {"OMP_NUM_THREADS": str(self.a.threads)}\n        if omp:\n            e["OMP_TARGET_OFFLOAD"] = self.a.offload\n        return e\n\n\ndef pragmas(path):\n    text = open(path).read()\n    t = len(re.findall(r"#pragma omp target teams", text))\n    c = len(re.findall(r"#pragma omp parallel for", text))\n    return t, c\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    ap.add_argument("--pb", required=True, help="PolyBench/C 4.2.1 directory")\n    ap.add_argument("--src", required=True, help="directory with gated/ and naive/")\n    ap.add_argument("--work", default="work")\n    ap.add_argument("--csv", default="polybench-t4.csv")\n    ap.add_argument("--cc", default="nvc")\n    ap.add_argument("--omp-flags", default="-mp=gpu -gpu=cc75")\n    ap.add_argument("--offload", default="DEFAULT",\n                    help="OMP_TARGET_OFFLOAD for the OpenMP variants")\n    ap.add_argument("--dataset", default="LARGE_DATASET")\n    ap.add_argument("--threads", type=int, default=2)\n    ap.add_argument("--runs", type=int, default=3)\n    ap.add_argument("--timeout", type=int, default=600, help="seconds per run")\n    ap.add_argument("--kernels", nargs="*")\n    ap.add_argument("--variants", nargs="*")\n    args = ap.parse_args()\n    b = Bench(args)\n\n    done, new = set(), True\n    try:\n        with open(args.csv) as f:\n            done = {(r["kernel"], r["variant"]) for r in csv.DictReader(f)}\n            new = False\n    except FileNotFoundError:\n        pass\n    out = open(args.csv, "a", newline="")\n    w = csv.DictWriter(out, FIELDS)\n    if new:\n        w.writeheader()\n\n    for kpath in b.kernels():\n        k = os.path.splitext(os.path.basename(kpath))[0]\n        if args.kernels and k not in args.kernels:\n            continue\n        ref = None\n        for name, source, omp, warm in VARIANTS:\n            if args.variants and name not in args.variants and name != "seq":\n                continue\n            if (k, name) in done and name != "seq":\n                continue\n            src = (os.path.join(b.pb, kpath) if source == "orig"\n                   else os.path.join(args.src, source, os.path.basename(kpath)))\n            tp, cp = pragmas(src)\n            row = dict(kernel=k, variant=name, threads=args.threads if omp else 1,\n                       target_pragmas=tp, cpu_pragmas=cp)\n            # Correctness and GPU launches.\n            check, err, launches = "", 0.0, ""\n            exe, log = b.build(name, kpath, src, omp, warm, exact=True)\n            if exe:\n                env = b.env(omp)\n                if omp:\n                    env["NVCOMPILER_ACC_NOTIFY"] = "1"\n                rc, _, se = run([exe], env=env, timeout=args.timeout)\n                lines = se.splitlines()\n                launches = sum(1 for l in lines if l.startswith("launch CUDA kernel"))\n                dump = "\\n".join(l for l in lines if "CUDA" not in l)\n                if rc != 0:\n                    check = "timeout" if rc == -1 else f"exit {rc}"\n                elif name == "seq":\n                    ref = dump\n                elif ref is not None:\n                    check, err = compare(ref, dump)\n            if (k, name) in done:\n                continue\n            exe, log = b.build(name, kpath, src, omp, warm, exact=False)\n            if not exe:\n                w.writerow(dict(row, status="build-failed", detail=log[:200]))\n                out.flush()\n                print(f"{k:16} {name:15} build failed: {log[:120]}", flush=True)\n                continue\n            times, status = [], "ok"\n            for _ in range(args.runs):\n                rc, so, se = run([exe], env=b.env(omp), timeout=args.timeout)\n                if rc != 0:\n                    status = "timeout" if rc == -1 else f"exit {rc}"\n                    break\n                times.append(float(so.strip().split()[-1]))\n            w.writerow(dict(row, status=status, correct=check,\n                            max_rel_err=f"{err:.3g}" if check else "",\n                            gpu_launches=launches if omp else "",\n                            time_s=f"{min(times):.6f}" if times else "",\n                            times=" ".join(f"{x:.6f}" for x in times),\n                            detail="" if status == "ok" else se.strip()[-200:]))\n            out.flush()\n            print(f"{k:16} {name:15} {status:8} "\n                  f"{min(times) if times else float(\'nan\'):10.4f}s  {check:9} "\n                  f"launches={launches}", flush=True)\n    out.close()\n\n\nif __name__ == "__main__":\n    main()\n')
print('working in', ROOT)


In [ ]:
%%bash
nvidia-smi --query-gpu=name,compute_cap,driver_version,memory.total --format=csv
nproc; grep -m1 'model name' /proc/cpuinfo


In [ ]:
%%bash
# NVIDIA HPC SDK (nvc -mp=gpu): the compiler the GPU evaluation uses (NOTES.md, 2026-10-02).
set -e
if ls /opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin/nvc >/dev/null 2>&1; then echo "already installed"; exit 0; fi
curl -s https://developer.download.nvidia.com/hpc-sdk/ubuntu/DEB-GPG-KEY-NVIDIA-HPC-SDK \
  | gpg --dearmor --yes -o /usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg
echo 'deb [signed-by=/usr/share/keyrings/nvidia-hpcsdk-archive-keyring.gpg] https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64 /' \
  > /etc/apt/sources.list.d/nvhpc.list
apt-get update -qq
PKG=$(apt-cache search --names-only '^nvhpc-[0-9]+-[0-9]+$' | awk '{print $1}' | sort -V | tail -1)
echo "Installing $PKG"; apt-get install -y -qq $PKG > /tmp/nvhpc.log 2>&1 || { tail -20 /tmp/nvhpc.log; exit 1; }
ls -d /opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin/nvc


In [ ]:
%%bash
# PolyBench/C 4.2.1 at the commit eval/llm_baseline/fetch_polybench.sh pins.
set -e
cd "$P05_ROOT"
[ -d polybench/.git ] || git clone -q https://github.com/MatthiasJReisinger/PolyBenchC-4.2.1 polybench
git -C polybench checkout -q 3e872547cef7e5c9909422ef1e6af03cf4e56072
test "$(git -C polybench rev-parse HEAD)" = 3e872547cef7e5c9909422ef1e6af03cf4e56072 && echo "polybench at 3e872547cef7e5c9909422ef1e6af03cf4e56072"


In [ ]:
# P05's rewrites (made with -machine machine/colab-t4.json).
import json, os
ROOT = os.environ['P05_ROOT']
FILES = json.loads('{"gated/correlation.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* correlation.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"correlation.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int m,\\n\\t\\t int n,\\n\\t\\t DATA_TYPE *float_n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(data,N,M,n,m))\\n{\\n  int i, j;\\n\\n  *float_n = (DATA_TYPE)N;\\n\\n  #pragma omp parallel for private(j)\\n  for (i = 0; i < N; i++)\\n    for (j = 0; j < M; j++)\\n      data[i][j] = (DATA_TYPE)(i*j)/M + i;\\n\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int m,\\n\\t\\t DATA_TYPE POLYBENCH_2D(corr,M,M,m,m))\\n\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"corr\\");\\n  for (i = 0; i < m; i++)\\n    for (j = 0; j < m; j++) {\\n      if ((i * m + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, corr[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"corr\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_correlation(int m, int n,\\n\\t\\t\\tDATA_TYPE float_n,\\n\\t\\t\\tDATA_TYPE POLYBENCH_2D(data,N,M,n,m),\\n\\t\\t\\tDATA_TYPE POLYBENCH_2D(corr,M,M,m,m),\\n\\t\\t\\tDATA_TYPE POLYBENCH_1D(mean,M,m),\\n\\t\\t\\tDATA_TYPE POLYBENCH_1D(stddev,M,m))\\n{\\n  int i, j, k;\\n\\n  DATA_TYPE eps = SCALAR_VAL(0.1);\\n\\n\\n#pragma scop\\n  for (j = 0; j < _PB_M; j++)\\n    {\\n      mean[j] = SCALAR_VAL(0.0);\\n      for (i = 0; i < _PB_N; i++)\\n\\tmean[j] += data[i][j];\\n      mean[j] /= float_n;\\n    }\\n\\n\\n   for (j = 0; j < _PB_M; j++)\\n    {\\n      stddev[j] = SCALAR_VAL(0.0);\\n      for (i = 0; i < _PB_N; i++)\\n        stddev[j] += (data[i][j] - mean[j]) * (data[i][j] - mean[j]);\\n      stddev[j] /= float_n;\\n      stddev[j] = SQRT_FUN(stddev[j]);\\n      /* The following in an inelegant but usual way to handle\\n         near-zero std. dev. values, which below would cause a zero-\\n         divide. */\\n      stddev[j] = stddev[j] <= eps ? SCALAR_VAL(1.0) : stddev[j];\\n    }\\n\\n  /* Center and reduce the column vectors. */\\n  for (i = 0; i < _PB_N; i++)\\n    #pragma omp parallel for if(parallel: (m >= 262144) && ((__INTPTR_TYPE__)data + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)m - 1) + 1) <= (__INTPTR_TYPE__)mean + 0 || (__INTPTR_TYPE__)mean + 8 * (((__INTPTR_TYPE__)m - 1) + 1) <= (__INTPTR_TYPE__)data + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)) && ((__INTPTR_TYPE__)data + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)m - 1) + 1) <= (__INTPTR_TYPE__)stddev + 0 || (__INTPTR_TYPE__)stddev + 8 * (((__INTPTR_TYPE__)m - 1) + 1) <= (__INTPTR_TYPE__)data + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)))\\n    for (j = 0; j < _PB_M; j++)\\n      {\\n        data[i][j] -= mean[j];\\n        data[i][j] /= SQRT_FUN(float_n) * stddev[j];\\n      }\\n\\n  /* Calculate the m * m correlation matrix. */\\n  for (i = 0; i < _PB_M-1; i++)\\n    {\\n      corr[i][i] = SCALAR_VAL(1.0);\\n      for (j = i+1; j < _PB_M; j++)\\n        {\\n          corr[i][j] = SCALAR_VAL(0.0);\\n          for (k = 0; k < _PB_N; k++)\\n            corr[i][j] += (data[k][i] * data[k][j]);\\n          corr[j][i] = corr[i][j];\\n        }\\n    }\\n  corr[_PB_M-1][_PB_M-1] = SCALAR_VAL(1.0);\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n  int m = M;\\n\\n  /* Variable declaration/allocation. */\\n  DATA_TYPE float_n;\\n  POLYBENCH_2D_ARRAY_DECL(data,DATA_TYPE,N,M,n,m);\\n  POLYBENCH_2D_ARRAY_DECL(corr,DATA_TYPE,M,M,m,m);\\n  POLYBENCH_1D_ARRAY_DECL(mean,DATA_TYPE,M,m);\\n  POLYBENCH_1D_ARRAY_DECL(stddev,DATA_TYPE,M,m);\\n\\n  /* Initialize array(s). */\\n  init_array (m, n, &float_n, POLYBENCH_ARRAY(data));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_correlation (m, n, float_n,\\n\\t\\t      POLYBENCH_ARRAY(data),\\n\\t\\t      POLYBENCH_ARRAY(corr),\\n\\t\\t      POLYBENCH_ARRAY(mean),\\n\\t\\t      POLYBENCH_ARRAY(stddev));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(m, POLYBENCH_ARRAY(corr)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(data);\\n  POLYBENCH_FREE_ARRAY(corr);\\n  POLYBENCH_FREE_ARRAY(mean);\\n  POLYBENCH_FREE_ARRAY(stddev);\\n\\n  return 0;\\n}\\n", "naive/correlation.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* correlation.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"correlation.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int m,\\n\\t\\t int n,\\n\\t\\t DATA_TYPE *float_n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(data,N,M,n,m))\\n{\\n  int i, j;\\n\\n  *float_n = (DATA_TYPE)N;\\n\\n  #pragma omp target teams distribute parallel for collapse(2) map(from: data[0:1400][0:1200]) private(j)\\n  for (i = 0; i < N; i++)\\n    for (j = 0; j < M; j++)\\n      data[i][j] = (DATA_TYPE)(i*j)/M + i;\\n\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int m,\\n\\t\\t DATA_TYPE POLYBENCH_2D(corr,M,M,m,m))\\n\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"corr\\");\\n  for (i = 0; i < m; i++)\\n    for (j = 0; j < m; j++) {\\n      if ((i * m + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, corr[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"corr\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_correlation(int m, int n,\\n\\t\\t\\tDATA_TYPE float_n,\\n\\t\\t\\tDATA_TYPE POLYBENCH_2D(data,N,M,n,m),\\n\\t\\t\\tDATA_TYPE POLYBENCH_2D(corr,M,M,m,m),\\n\\t\\t\\tDATA_TYPE POLYBENCH_1D(mean,M,m),\\n\\t\\t\\tDATA_TYPE POLYBENCH_1D(stddev,M,m))\\n{\\n  int i, j, k;\\n\\n  DATA_TYPE eps = SCALAR_VAL(0.1);\\n\\n\\n#pragma scop\\n  for (j = 0; j < _PB_M; j++)\\n    {\\n      mean[j] = SCALAR_VAL(0.0);\\n      for (i = 0; i < _PB_N; i++)\\n\\tmean[j] += data[i][j];\\n      mean[j] /= float_n;\\n    }\\n\\n\\n   for (j = 0; j < _PB_M; j++)\\n    {\\n      stddev[j] = SCALAR_VAL(0.0);\\n      for (i = 0; i < _PB_N; i++)\\n        stddev[j] += (data[i][j] - mean[j]) * (data[i][j] - mean[j]);\\n      stddev[j] /= float_n;\\n      stddev[j] = SQRT_FUN(stddev[j]);\\n      /* The following in an inelegant but usual way to handle\\n         near-zero std. dev. values, which below would cause a zero-\\n         divide. */\\n      stddev[j] = stddev[j] <= eps ? SCALAR_VAL(1.0) : stddev[j];\\n    }\\n\\n  /* Center and reduce the column vectors. */\\n  for (i = 0; i < _PB_N; i++)\\n    #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)data + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)m - 1) + 1) <= (__INTPTR_TYPE__)mean + 0 || (__INTPTR_TYPE__)mean + 8 * (((__INTPTR_TYPE__)m - 1) + 1) <= (__INTPTR_TYPE__)data + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)) && ((__INTPTR_TYPE__)data + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)m - 1) + 1) <= (__INTPTR_TYPE__)stddev + 0 || (__INTPTR_TYPE__)stddev + 8 * (((__INTPTR_TYPE__)m - 1) + 1) <= (__INTPTR_TYPE__)data + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)))\\n    for (j = 0; j < _PB_M; j++)\\n      {\\n        data[i][j] -= mean[j];\\n        data[i][j] /= SQRT_FUN(float_n) * stddev[j];\\n      }\\n\\n  /* Calculate the m * m correlation matrix. */\\n  for (i = 0; i < _PB_M-1; i++)\\n    {\\n      corr[i][i] = SCALAR_VAL(1.0);\\n      for (j = i+1; j < _PB_M; j++)\\n        {\\n          corr[i][j] = SCALAR_VAL(0.0);\\n          for (k = 0; k < _PB_N; k++)\\n            corr[i][j] += (data[k][i] * data[k][j]);\\n          corr[j][i] = corr[i][j];\\n        }\\n    }\\n  corr[_PB_M-1][_PB_M-1] = SCALAR_VAL(1.0);\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n  int m = M;\\n\\n  /* Variable declaration/allocation. */\\n  DATA_TYPE float_n;\\n  POLYBENCH_2D_ARRAY_DECL(data,DATA_TYPE,N,M,n,m);\\n  POLYBENCH_2D_ARRAY_DECL(corr,DATA_TYPE,M,M,m,m);\\n  POLYBENCH_1D_ARRAY_DECL(mean,DATA_TYPE,M,m);\\n  POLYBENCH_1D_ARRAY_DECL(stddev,DATA_TYPE,M,m);\\n\\n  /* Initialize array(s). */\\n  init_array (m, n, &float_n, POLYBENCH_ARRAY(data));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_correlation (m, n, float_n,\\n\\t\\t      POLYBENCH_ARRAY(data),\\n\\t\\t      POLYBENCH_ARRAY(corr),\\n\\t\\t      POLYBENCH_ARRAY(mean),\\n\\t\\t      POLYBENCH_ARRAY(stddev));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(m, POLYBENCH_ARRAY(corr)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(data);\\n  POLYBENCH_FREE_ARRAY(corr);\\n  POLYBENCH_FREE_ARRAY(mean);\\n  POLYBENCH_FREE_ARRAY(stddev);\\n\\n  return 0;\\n}\\n", "gated/covariance.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* covariance.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"covariance.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int m, int n,\\n\\t\\t DATA_TYPE *float_n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(data,N,M,n,m))\\n{\\n  int i, j;\\n\\n  *float_n = (DATA_TYPE)n;\\n\\n  #pragma omp parallel for private(j)\\n  for (i = 0; i < N; i++)\\n    for (j = 0; j < M; j++)\\n      data[i][j] = ((DATA_TYPE) i*j) / M;\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int m,\\n\\t\\t DATA_TYPE POLYBENCH_2D(cov,M,M,m,m))\\n\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"cov\\");\\n  for (i = 0; i < m; i++)\\n    for (j = 0; j < m; j++) {\\n      if ((i * m + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, cov[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"cov\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_covariance(int m, int n,\\n\\t\\t       DATA_TYPE float_n,\\n\\t\\t       DATA_TYPE POLYBENCH_2D(data,N,M,n,m),\\n\\t\\t       DATA_TYPE POLYBENCH_2D(cov,M,M,m,m),\\n\\t\\t       DATA_TYPE POLYBENCH_1D(mean,M,m))\\n{\\n  int i, j, k;\\n\\n#pragma scop\\n  for (j = 0; j < _PB_M; j++)\\n    {\\n      mean[j] = SCALAR_VAL(0.0);\\n      for (i = 0; i < _PB_N; i++)\\n        mean[j] += data[i][j];\\n      mean[j] /= float_n;\\n    }\\n\\n  for (i = 0; i < _PB_N; i++)\\n    #pragma omp parallel for if(parallel: (m >= 524288) && ((__INTPTR_TYPE__)data + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)m - 1) + 1) <= (__INTPTR_TYPE__)mean + 0 || (__INTPTR_TYPE__)mean + 8 * (((__INTPTR_TYPE__)m - 1) + 1) <= (__INTPTR_TYPE__)data + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)))\\n    for (j = 0; j < _PB_M; j++)\\n      data[i][j] -= mean[j];\\n\\n  for (i = 0; i < _PB_M; i++)\\n    for (j = i; j < _PB_M; j++)\\n      {\\n        cov[i][j] = SCALAR_VAL(0.0);\\n        for (k = 0; k < _PB_N; k++)\\n\\t  cov[i][j] += data[k][i] * data[k][j];\\n        cov[i][j] /= (float_n - SCALAR_VAL(1.0));\\n        cov[j][i] = cov[i][j];\\n      }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n  int m = M;\\n\\n  /* Variable declaration/allocation. */\\n  DATA_TYPE float_n;\\n  POLYBENCH_2D_ARRAY_DECL(data,DATA_TYPE,N,M,n,m);\\n  POLYBENCH_2D_ARRAY_DECL(cov,DATA_TYPE,M,M,m,m);\\n  POLYBENCH_1D_ARRAY_DECL(mean,DATA_TYPE,M,m);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (m, n, &float_n, POLYBENCH_ARRAY(data));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_covariance (m, n, float_n,\\n\\t\\t     POLYBENCH_ARRAY(data),\\n\\t\\t     POLYBENCH_ARRAY(cov),\\n\\t\\t     POLYBENCH_ARRAY(mean));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(m, POLYBENCH_ARRAY(cov)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(data);\\n  POLYBENCH_FREE_ARRAY(cov);\\n  POLYBENCH_FREE_ARRAY(mean);\\n\\n  return 0;\\n}\\n", "naive/covariance.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* covariance.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"covariance.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int m, int n,\\n\\t\\t DATA_TYPE *float_n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(data,N,M,n,m))\\n{\\n  int i, j;\\n\\n  *float_n = (DATA_TYPE)n;\\n\\n  #pragma omp target teams distribute parallel for collapse(2) map(from: data[0:1400][0:1200]) private(j)\\n  for (i = 0; i < N; i++)\\n    for (j = 0; j < M; j++)\\n      data[i][j] = ((DATA_TYPE) i*j) / M;\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int m,\\n\\t\\t DATA_TYPE POLYBENCH_2D(cov,M,M,m,m))\\n\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"cov\\");\\n  for (i = 0; i < m; i++)\\n    for (j = 0; j < m; j++) {\\n      if ((i * m + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, cov[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"cov\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_covariance(int m, int n,\\n\\t\\t       DATA_TYPE float_n,\\n\\t\\t       DATA_TYPE POLYBENCH_2D(data,N,M,n,m),\\n\\t\\t       DATA_TYPE POLYBENCH_2D(cov,M,M,m,m),\\n\\t\\t       DATA_TYPE POLYBENCH_1D(mean,M,m))\\n{\\n  int i, j, k;\\n\\n#pragma scop\\n  for (j = 0; j < _PB_M; j++)\\n    {\\n      mean[j] = SCALAR_VAL(0.0);\\n      for (i = 0; i < _PB_N; i++)\\n        mean[j] += data[i][j];\\n      mean[j] /= float_n;\\n    }\\n\\n  for (i = 0; i < _PB_N; i++)\\n    #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)data + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)m - 1) + 1) <= (__INTPTR_TYPE__)mean + 0 || (__INTPTR_TYPE__)mean + 8 * (((__INTPTR_TYPE__)m - 1) + 1) <= (__INTPTR_TYPE__)data + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)))\\n    for (j = 0; j < _PB_M; j++)\\n      data[i][j] -= mean[j];\\n\\n  for (i = 0; i < _PB_M; i++)\\n    for (j = i; j < _PB_M; j++)\\n      {\\n        cov[i][j] = SCALAR_VAL(0.0);\\n        for (k = 0; k < _PB_N; k++)\\n\\t  cov[i][j] += data[k][i] * data[k][j];\\n        cov[i][j] /= (float_n - SCALAR_VAL(1.0));\\n        cov[j][i] = cov[i][j];\\n      }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n  int m = M;\\n\\n  /* Variable declaration/allocation. */\\n  DATA_TYPE float_n;\\n  POLYBENCH_2D_ARRAY_DECL(data,DATA_TYPE,N,M,n,m);\\n  POLYBENCH_2D_ARRAY_DECL(cov,DATA_TYPE,M,M,m,m);\\n  POLYBENCH_1D_ARRAY_DECL(mean,DATA_TYPE,M,m);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (m, n, &float_n, POLYBENCH_ARRAY(data));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_covariance (m, n, float_n,\\n\\t\\t     POLYBENCH_ARRAY(data),\\n\\t\\t     POLYBENCH_ARRAY(cov),\\n\\t\\t     POLYBENCH_ARRAY(mean));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(m, POLYBENCH_ARRAY(cov)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(data);\\n  POLYBENCH_FREE_ARRAY(cov);\\n  POLYBENCH_FREE_ARRAY(mean);\\n\\n  return 0;\\n}\\n", "gated/2mm.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* 2mm.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"2mm.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int ni, int nj, int nk, int nl,\\n\\t\\tDATA_TYPE *alpha,\\n\\t\\tDATA_TYPE *beta,\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,NI,NK,ni,nk),\\n\\t\\tDATA_TYPE POLYBENCH_2D(B,NK,NJ,nk,nj),\\n\\t\\tDATA_TYPE POLYBENCH_2D(C,NJ,NL,nj,nl),\\n\\t\\tDATA_TYPE POLYBENCH_2D(D,NI,NL,ni,nl))\\n{\\n  int i, j;\\n\\n  *alpha = 1.5;\\n  *beta = 1.2;\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ((double)(ni) * ((5 * (double)(nk)) - 27063) >= 1073741824) && ni >= 1 && nk >= 1) map(tofrom: A[0:ni][0:1100]) private(j)\\n  for (i = 0; i < ni; i++)\\n    for (j = 0; j < nk; j++)\\n      A[i][j] = (DATA_TYPE) ((i*j+1) % ni) / ni;\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ((double)(nk) * ((5 * (double)(nj)) - 22142) >= 1073741824) && nk >= 1 && nj >= 1) map(tofrom: B[0:nk][0:900]) private(j)\\n  for (i = 0; i < nk; i++)\\n    for (j = 0; j < nj; j++)\\n      B[i][j] = (DATA_TYPE) (i*(j+1) % nj) / nj;\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ((double)(nj) * ((6 * (double)(nl)) - 29523) >= 1073741824) && nj >= 1 && nl >= 1) map(tofrom: C[0:nj][0:1200]) private(j)\\n  for (i = 0; i < nj; i++)\\n    for (j = 0; j < nl; j++)\\n      C[i][j] = (DATA_TYPE) ((i*(j+3)+1) % nl) / nl;\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ((double)(ni) * ((5 * (double)(nl)) - 29523) >= 1073741824) && ni >= 1 && nl >= 1) map(tofrom: D[0:ni][0:1200]) private(j)\\n  for (i = 0; i < ni; i++)\\n    for (j = 0; j < nl; j++)\\n      D[i][j] = (DATA_TYPE) (i*(j+2) % nk) / nk;\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int ni, int nl,\\n\\t\\t DATA_TYPE POLYBENCH_2D(D,NI,NL,ni,nl))\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"D\\");\\n  for (i = 0; i < ni; i++)\\n    for (j = 0; j < nl; j++) {\\n\\tif ((i * ni + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n\\tfprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, D[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"D\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_2mm(int ni, int nj, int nk, int nl,\\n\\t\\tDATA_TYPE alpha,\\n\\t\\tDATA_TYPE beta,\\n\\t\\tDATA_TYPE POLYBENCH_2D(tmp,NI,NJ,ni,nj),\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,NI,NK,ni,nk),\\n\\t\\tDATA_TYPE POLYBENCH_2D(B,NK,NJ,nk,nj),\\n\\t\\tDATA_TYPE POLYBENCH_2D(C,NJ,NL,nj,nl),\\n\\t\\tDATA_TYPE POLYBENCH_2D(D,NI,NL,ni,nl))\\n{\\n  int i, j, k;\\n\\n#pragma scop\\n  /* D := alpha*A*B*C + beta*D */\\n  for (i = 0; i < _PB_NI; i++)\\n    for (j = 0; j < _PB_NJ; j++)\\n      {\\n\\ttmp[i][j] = SCALAR_VAL(0.0);\\n\\tfor (k = 0; k < _PB_NK; ++k)\\n\\t  tmp[i][j] += alpha * A[i][k] * B[k][j];\\n      }\\n  for (i = 0; i < _PB_NI; i++)\\n    for (j = 0; j < _PB_NL; j++)\\n      {\\n\\tD[i][j] *= beta;\\n\\tfor (k = 0; k < _PB_NJ; ++k)\\n\\t  D[i][j] += tmp[i][k] * C[k][j];\\n      }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int ni = NI;\\n  int nj = NJ;\\n  int nk = NK;\\n  int nl = NL;\\n\\n  /* Variable declaration/allocation. */\\n  DATA_TYPE alpha;\\n  DATA_TYPE beta;\\n  POLYBENCH_2D_ARRAY_DECL(tmp,DATA_TYPE,NI,NJ,ni,nj);\\n  POLYBENCH_2D_ARRAY_DECL(A,DATA_TYPE,NI,NK,ni,nk);\\n  POLYBENCH_2D_ARRAY_DECL(B,DATA_TYPE,NK,NJ,nk,nj);\\n  POLYBENCH_2D_ARRAY_DECL(C,DATA_TYPE,NJ,NL,nj,nl);\\n  POLYBENCH_2D_ARRAY_DECL(D,DATA_TYPE,NI,NL,ni,nl);\\n\\n  /* Initialize array(s). */\\n  init_array (ni, nj, nk, nl, &alpha, &beta,\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(B),\\n\\t      POLYBENCH_ARRAY(C),\\n\\t      POLYBENCH_ARRAY(D));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_2mm (ni, nj, nk, nl,\\n\\t      alpha, beta,\\n\\t      POLYBENCH_ARRAY(tmp),\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(B),\\n\\t      POLYBENCH_ARRAY(C),\\n\\t      POLYBENCH_ARRAY(D));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(ni, nl,  POLYBENCH_ARRAY(D)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(tmp);\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(B);\\n  POLYBENCH_FREE_ARRAY(C);\\n  POLYBENCH_FREE_ARRAY(D);\\n\\n  return 0;\\n}\\n", "naive/2mm.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* 2mm.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"2mm.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int ni, int nj, int nk, int nl,\\n\\t\\tDATA_TYPE *alpha,\\n\\t\\tDATA_TYPE *beta,\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,NI,NK,ni,nk),\\n\\t\\tDATA_TYPE POLYBENCH_2D(B,NK,NJ,nk,nj),\\n\\t\\tDATA_TYPE POLYBENCH_2D(C,NJ,NL,nj,nl),\\n\\t\\tDATA_TYPE POLYBENCH_2D(D,NI,NL,ni,nl))\\n{\\n  int i, j;\\n\\n  *alpha = 1.5;\\n  *beta = 1.2;\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ni >= 1 && nk >= 1) map(tofrom: A[0:ni][0:1100]) private(j)\\n  for (i = 0; i < ni; i++)\\n    for (j = 0; j < nk; j++)\\n      A[i][j] = (DATA_TYPE) ((i*j+1) % ni) / ni;\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: nk >= 1 && nj >= 1) map(tofrom: B[0:nk][0:900]) private(j)\\n  for (i = 0; i < nk; i++)\\n    for (j = 0; j < nj; j++)\\n      B[i][j] = (DATA_TYPE) (i*(j+1) % nj) / nj;\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: nj >= 1 && nl >= 1) map(tofrom: C[0:nj][0:1200]) private(j)\\n  for (i = 0; i < nj; i++)\\n    for (j = 0; j < nl; j++)\\n      C[i][j] = (DATA_TYPE) ((i*(j+3)+1) % nl) / nl;\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ni >= 1 && nl >= 1) map(tofrom: D[0:ni][0:1200]) private(j)\\n  for (i = 0; i < ni; i++)\\n    for (j = 0; j < nl; j++)\\n      D[i][j] = (DATA_TYPE) (i*(j+2) % nk) / nk;\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int ni, int nl,\\n\\t\\t DATA_TYPE POLYBENCH_2D(D,NI,NL,ni,nl))\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"D\\");\\n  for (i = 0; i < ni; i++)\\n    for (j = 0; j < nl; j++) {\\n\\tif ((i * ni + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n\\tfprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, D[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"D\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_2mm(int ni, int nj, int nk, int nl,\\n\\t\\tDATA_TYPE alpha,\\n\\t\\tDATA_TYPE beta,\\n\\t\\tDATA_TYPE POLYBENCH_2D(tmp,NI,NJ,ni,nj),\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,NI,NK,ni,nk),\\n\\t\\tDATA_TYPE POLYBENCH_2D(B,NK,NJ,nk,nj),\\n\\t\\tDATA_TYPE POLYBENCH_2D(C,NJ,NL,nj,nl),\\n\\t\\tDATA_TYPE POLYBENCH_2D(D,NI,NL,ni,nl))\\n{\\n  int i, j, k;\\n\\n#pragma scop\\n  /* D := alpha*A*B*C + beta*D */\\n  for (i = 0; i < _PB_NI; i++)\\n    for (j = 0; j < _PB_NJ; j++)\\n      {\\n\\ttmp[i][j] = SCALAR_VAL(0.0);\\n\\tfor (k = 0; k < _PB_NK; ++k)\\n\\t  tmp[i][j] += alpha * A[i][k] * B[k][j];\\n      }\\n  for (i = 0; i < _PB_NI; i++)\\n    for (j = 0; j < _PB_NL; j++)\\n      {\\n\\tD[i][j] *= beta;\\n\\tfor (k = 0; k < _PB_NJ; ++k)\\n\\t  D[i][j] += tmp[i][k] * C[k][j];\\n      }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int ni = NI;\\n  int nj = NJ;\\n  int nk = NK;\\n  int nl = NL;\\n\\n  /* Variable declaration/allocation. */\\n  DATA_TYPE alpha;\\n  DATA_TYPE beta;\\n  POLYBENCH_2D_ARRAY_DECL(tmp,DATA_TYPE,NI,NJ,ni,nj);\\n  POLYBENCH_2D_ARRAY_DECL(A,DATA_TYPE,NI,NK,ni,nk);\\n  POLYBENCH_2D_ARRAY_DECL(B,DATA_TYPE,NK,NJ,nk,nj);\\n  POLYBENCH_2D_ARRAY_DECL(C,DATA_TYPE,NJ,NL,nj,nl);\\n  POLYBENCH_2D_ARRAY_DECL(D,DATA_TYPE,NI,NL,ni,nl);\\n\\n  /* Initialize array(s). */\\n  init_array (ni, nj, nk, nl, &alpha, &beta,\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(B),\\n\\t      POLYBENCH_ARRAY(C),\\n\\t      POLYBENCH_ARRAY(D));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_2mm (ni, nj, nk, nl,\\n\\t      alpha, beta,\\n\\t      POLYBENCH_ARRAY(tmp),\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(B),\\n\\t      POLYBENCH_ARRAY(C),\\n\\t      POLYBENCH_ARRAY(D));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(ni, nl,  POLYBENCH_ARRAY(D)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(tmp);\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(B);\\n  POLYBENCH_FREE_ARRAY(C);\\n  POLYBENCH_FREE_ARRAY(D);\\n\\n  return 0;\\n}\\n", "gated/3mm.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* 3mm.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"3mm.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int ni, int nj, int nk, int nl, int nm,\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,NI,NK,ni,nk),\\n\\t\\tDATA_TYPE POLYBENCH_2D(B,NK,NJ,nk,nj),\\n\\t\\tDATA_TYPE POLYBENCH_2D(C,NJ,NM,nj,nm),\\n\\t\\tDATA_TYPE POLYBENCH_2D(D,NM,NL,nm,nl))\\n{\\n  int i, j;\\n\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ((double)(ni) * ((6 * (double)(nk)) - 24603) >= 1073741824) && ni >= 1 && nk >= 1) map(tofrom: A[0:ni][0:1000]) private(j)\\n  for (i = 0; i < ni; i++)\\n    for (j = 0; j < nk; j++)\\n      A[i][j] = (DATA_TYPE) ((i*j+1) % ni) / (5*ni);\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ((double)(nk) * ((7 * (double)(nj)) - 22142) >= 1073741824) && nk >= 1 && nj >= 1) map(tofrom: B[0:nk][0:900]) private(j)\\n  for (i = 0; i < nk; i++)\\n    for (j = 0; j < nj; j++)\\n      B[i][j] = (DATA_TYPE) ((i*(j+1)+2) % nj) / (5*nj);\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ((double)(nj) * ((6 * (double)(nm)) - 29523) >= 1073741824) && nj >= 1 && nm >= 1) map(tofrom: C[0:nj][0:1200]) private(j)\\n  for (i = 0; i < nj; i++)\\n    for (j = 0; j < nm; j++)\\n      C[i][j] = (DATA_TYPE) (i*(j+3) % nl) / (5*nl);\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ((double)(nm) * ((7 * (double)(nl)) - 27063) >= 1073741824) && nm >= 1 && nl >= 1) map(tofrom: D[0:nm][0:1100]) private(j)\\n  for (i = 0; i < nm; i++)\\n    for (j = 0; j < nl; j++)\\n      D[i][j] = (DATA_TYPE) ((i*(j+2)+2) % nk) / (5*nk);\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int ni, int nl,\\n\\t\\t DATA_TYPE POLYBENCH_2D(G,NI,NL,ni,nl))\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"G\\");\\n  for (i = 0; i < ni; i++)\\n    for (j = 0; j < nl; j++) {\\n\\tif ((i * ni + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n\\tfprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, G[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"G\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_3mm(int ni, int nj, int nk, int nl, int nm,\\n\\t\\tDATA_TYPE POLYBENCH_2D(E,NI,NJ,ni,nj),\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,NI,NK,ni,nk),\\n\\t\\tDATA_TYPE POLYBENCH_2D(B,NK,NJ,nk,nj),\\n\\t\\tDATA_TYPE POLYBENCH_2D(F,NJ,NL,nj,nl),\\n\\t\\tDATA_TYPE POLYBENCH_2D(C,NJ,NM,nj,nm),\\n\\t\\tDATA_TYPE POLYBENCH_2D(D,NM,NL,nm,nl),\\n\\t\\tDATA_TYPE POLYBENCH_2D(G,NI,NL,ni,nl))\\n{\\n  int i, j, k;\\n\\n#pragma scop\\n  /* E := A*B */\\n  for (i = 0; i < _PB_NI; i++)\\n    for (j = 0; j < _PB_NJ; j++)\\n      {\\n\\tE[i][j] = SCALAR_VAL(0.0);\\n\\tfor (k = 0; k < _PB_NK; ++k)\\n\\t  E[i][j] += A[i][k] * B[k][j];\\n      }\\n  /* F := C*D */\\n  for (i = 0; i < _PB_NJ; i++)\\n    for (j = 0; j < _PB_NL; j++)\\n      {\\n\\tF[i][j] = SCALAR_VAL(0.0);\\n\\tfor (k = 0; k < _PB_NM; ++k)\\n\\t  F[i][j] += C[i][k] * D[k][j];\\n      }\\n  /* G := E*F */\\n  for (i = 0; i < _PB_NI; i++)\\n    for (j = 0; j < _PB_NL; j++)\\n      {\\n\\tG[i][j] = SCALAR_VAL(0.0);\\n\\tfor (k = 0; k < _PB_NJ; ++k)\\n\\t  G[i][j] += E[i][k] * F[k][j];\\n      }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int ni = NI;\\n  int nj = NJ;\\n  int nk = NK;\\n  int nl = NL;\\n  int nm = NM;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(E, DATA_TYPE, NI, NJ, ni, nj);\\n  POLYBENCH_2D_ARRAY_DECL(A, DATA_TYPE, NI, NK, ni, nk);\\n  POLYBENCH_2D_ARRAY_DECL(B, DATA_TYPE, NK, NJ, nk, nj);\\n  POLYBENCH_2D_ARRAY_DECL(F, DATA_TYPE, NJ, NL, nj, nl);\\n  POLYBENCH_2D_ARRAY_DECL(C, DATA_TYPE, NJ, NM, nj, nm);\\n  POLYBENCH_2D_ARRAY_DECL(D, DATA_TYPE, NM, NL, nm, nl);\\n  POLYBENCH_2D_ARRAY_DECL(G, DATA_TYPE, NI, NL, ni, nl);\\n\\n  /* Initialize array(s). */\\n  init_array (ni, nj, nk, nl, nm,\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(B),\\n\\t      POLYBENCH_ARRAY(C),\\n\\t      POLYBENCH_ARRAY(D));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_3mm (ni, nj, nk, nl, nm,\\n\\t      POLYBENCH_ARRAY(E),\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(B),\\n\\t      POLYBENCH_ARRAY(F),\\n\\t      POLYBENCH_ARRAY(C),\\n\\t      POLYBENCH_ARRAY(D),\\n\\t      POLYBENCH_ARRAY(G));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(ni, nl,  POLYBENCH_ARRAY(G)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(E);\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(B);\\n  POLYBENCH_FREE_ARRAY(F);\\n  POLYBENCH_FREE_ARRAY(C);\\n  POLYBENCH_FREE_ARRAY(D);\\n  POLYBENCH_FREE_ARRAY(G);\\n\\n  return 0;\\n}\\n", "naive/3mm.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* 3mm.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"3mm.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int ni, int nj, int nk, int nl, int nm,\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,NI,NK,ni,nk),\\n\\t\\tDATA_TYPE POLYBENCH_2D(B,NK,NJ,nk,nj),\\n\\t\\tDATA_TYPE POLYBENCH_2D(C,NJ,NM,nj,nm),\\n\\t\\tDATA_TYPE POLYBENCH_2D(D,NM,NL,nm,nl))\\n{\\n  int i, j;\\n\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ni >= 1 && nk >= 1) map(tofrom: A[0:ni][0:1000]) private(j)\\n  for (i = 0; i < ni; i++)\\n    for (j = 0; j < nk; j++)\\n      A[i][j] = (DATA_TYPE) ((i*j+1) % ni) / (5*ni);\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: nk >= 1 && nj >= 1) map(tofrom: B[0:nk][0:900]) private(j)\\n  for (i = 0; i < nk; i++)\\n    for (j = 0; j < nj; j++)\\n      B[i][j] = (DATA_TYPE) ((i*(j+1)+2) % nj) / (5*nj);\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: nj >= 1 && nm >= 1) map(tofrom: C[0:nj][0:1200]) private(j)\\n  for (i = 0; i < nj; i++)\\n    for (j = 0; j < nm; j++)\\n      C[i][j] = (DATA_TYPE) (i*(j+3) % nl) / (5*nl);\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: nm >= 1 && nl >= 1) map(tofrom: D[0:nm][0:1100]) private(j)\\n  for (i = 0; i < nm; i++)\\n    for (j = 0; j < nl; j++)\\n      D[i][j] = (DATA_TYPE) ((i*(j+2)+2) % nk) / (5*nk);\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int ni, int nl,\\n\\t\\t DATA_TYPE POLYBENCH_2D(G,NI,NL,ni,nl))\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"G\\");\\n  for (i = 0; i < ni; i++)\\n    for (j = 0; j < nl; j++) {\\n\\tif ((i * ni + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n\\tfprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, G[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"G\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_3mm(int ni, int nj, int nk, int nl, int nm,\\n\\t\\tDATA_TYPE POLYBENCH_2D(E,NI,NJ,ni,nj),\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,NI,NK,ni,nk),\\n\\t\\tDATA_TYPE POLYBENCH_2D(B,NK,NJ,nk,nj),\\n\\t\\tDATA_TYPE POLYBENCH_2D(F,NJ,NL,nj,nl),\\n\\t\\tDATA_TYPE POLYBENCH_2D(C,NJ,NM,nj,nm),\\n\\t\\tDATA_TYPE POLYBENCH_2D(D,NM,NL,nm,nl),\\n\\t\\tDATA_TYPE POLYBENCH_2D(G,NI,NL,ni,nl))\\n{\\n  int i, j, k;\\n\\n#pragma scop\\n  /* E := A*B */\\n  for (i = 0; i < _PB_NI; i++)\\n    for (j = 0; j < _PB_NJ; j++)\\n      {\\n\\tE[i][j] = SCALAR_VAL(0.0);\\n\\tfor (k = 0; k < _PB_NK; ++k)\\n\\t  E[i][j] += A[i][k] * B[k][j];\\n      }\\n  /* F := C*D */\\n  for (i = 0; i < _PB_NJ; i++)\\n    for (j = 0; j < _PB_NL; j++)\\n      {\\n\\tF[i][j] = SCALAR_VAL(0.0);\\n\\tfor (k = 0; k < _PB_NM; ++k)\\n\\t  F[i][j] += C[i][k] * D[k][j];\\n      }\\n  /* G := E*F */\\n  for (i = 0; i < _PB_NI; i++)\\n    for (j = 0; j < _PB_NL; j++)\\n      {\\n\\tG[i][j] = SCALAR_VAL(0.0);\\n\\tfor (k = 0; k < _PB_NJ; ++k)\\n\\t  G[i][j] += E[i][k] * F[k][j];\\n      }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int ni = NI;\\n  int nj = NJ;\\n  int nk = NK;\\n  int nl = NL;\\n  int nm = NM;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(E, DATA_TYPE, NI, NJ, ni, nj);\\n  POLYBENCH_2D_ARRAY_DECL(A, DATA_TYPE, NI, NK, ni, nk);\\n  POLYBENCH_2D_ARRAY_DECL(B, DATA_TYPE, NK, NJ, nk, nj);\\n  POLYBENCH_2D_ARRAY_DECL(F, DATA_TYPE, NJ, NL, nj, nl);\\n  POLYBENCH_2D_ARRAY_DECL(C, DATA_TYPE, NJ, NM, nj, nm);\\n  POLYBENCH_2D_ARRAY_DECL(D, DATA_TYPE, NM, NL, nm, nl);\\n  POLYBENCH_2D_ARRAY_DECL(G, DATA_TYPE, NI, NL, ni, nl);\\n\\n  /* Initialize array(s). */\\n  init_array (ni, nj, nk, nl, nm,\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(B),\\n\\t      POLYBENCH_ARRAY(C),\\n\\t      POLYBENCH_ARRAY(D));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_3mm (ni, nj, nk, nl, nm,\\n\\t      POLYBENCH_ARRAY(E),\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(B),\\n\\t      POLYBENCH_ARRAY(F),\\n\\t      POLYBENCH_ARRAY(C),\\n\\t      POLYBENCH_ARRAY(D),\\n\\t      POLYBENCH_ARRAY(G));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(ni, nl,  POLYBENCH_ARRAY(G)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(E);\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(B);\\n  POLYBENCH_FREE_ARRAY(F);\\n  POLYBENCH_FREE_ARRAY(C);\\n  POLYBENCH_FREE_ARRAY(D);\\n  POLYBENCH_FREE_ARRAY(G);\\n\\n  return 0;\\n}\\n", "gated/atax.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* atax.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"atax.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int m, int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,M,N,m,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(x,N,n))\\n{\\n  int i, j;\\n  DATA_TYPE fn;\\n  fn = (DATA_TYPE)n;\\n\\n  #pragma omp parallel for if(parallel: n >= 1048576)\\n  for (i = 0; i < n; i++)\\n      x[i] = 1 + (i / fn);\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ((double)(m) * ((5 * (double)(n)) - 51665) >= 4294967296) && m >= 1 && n >= 1) map(tofrom: A[0:m][0:2100]) private(j)\\n  for (i = 0; i < m; i++)\\n    for (j = 0; j < n; j++)\\n      A[i][j] = (DATA_TYPE) ((i+j) % n) / (5*m);\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_1D(y,N,n))\\n\\n{\\n  int i;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"y\\");\\n  for (i = 0; i < n; i++) {\\n    if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, y[i]);\\n  }\\n  POLYBENCH_DUMP_END(\\"y\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_atax(int m, int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,M,N,m,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(x,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(y,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(tmp,M,m))\\n{\\n  int i, j;\\n\\n#pragma scop\\n  #pragma omp parallel for if(parallel: n >= 1048576)\\n  for (i = 0; i < _PB_N; i++)\\n    y[i] = 0;\\n  for (i = 0; i < _PB_M; i++)\\n    {\\n      tmp[i] = SCALAR_VAL(0.0);\\n      for (j = 0; j < _PB_N; j++)\\n\\ttmp[i] = tmp[i] + A[i][j] * x[j];\\n      #pragma omp parallel for if(parallel: (n >= 262144) && ((__INTPTR_TYPE__)y + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 2100 + 0) || (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 2100 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)y + 0) && ((__INTPTR_TYPE__)y + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)tmp + 8 * (__INTPTR_TYPE__)i || (__INTPTR_TYPE__)tmp + 8 * ((__INTPTR_TYPE__)i + 1) <= (__INTPTR_TYPE__)y + 0))\\n      for (j = 0; j < _PB_N; j++)\\n\\ty[j] = y[j] + A[i][j] * tmp[i];\\n    }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int m = M;\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(A, DATA_TYPE, M, N, m, n);\\n  POLYBENCH_1D_ARRAY_DECL(x, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(y, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(tmp, DATA_TYPE, M, m);\\n\\n  /* Initialize array(s). */\\n  init_array (m, n, POLYBENCH_ARRAY(A), POLYBENCH_ARRAY(x));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_atax (m, n,\\n\\t       POLYBENCH_ARRAY(A),\\n\\t       POLYBENCH_ARRAY(x),\\n\\t       POLYBENCH_ARRAY(y),\\n\\t       POLYBENCH_ARRAY(tmp));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(y)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(x);\\n  POLYBENCH_FREE_ARRAY(y);\\n  POLYBENCH_FREE_ARRAY(tmp);\\n\\n  return 0;\\n}\\n", "naive/atax.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* atax.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"atax.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int m, int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,M,N,m,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(x,N,n))\\n{\\n  int i, j;\\n  DATA_TYPE fn;\\n  fn = (DATA_TYPE)n;\\n\\n  #pragma omp target teams distribute parallel for if(target: n >= 1) map(from: x[0:n])\\n  for (i = 0; i < n; i++)\\n      x[i] = 1 + (i / fn);\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: m >= 1 && n >= 1) map(tofrom: A[0:m][0:2100]) private(j)\\n  for (i = 0; i < m; i++)\\n    for (j = 0; j < n; j++)\\n      A[i][j] = (DATA_TYPE) ((i+j) % n) / (5*m);\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_1D(y,N,n))\\n\\n{\\n  int i;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"y\\");\\n  for (i = 0; i < n; i++) {\\n    if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, y[i]);\\n  }\\n  POLYBENCH_DUMP_END(\\"y\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_atax(int m, int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,M,N,m,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(x,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(y,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(tmp,M,m))\\n{\\n  int i, j;\\n\\n#pragma scop\\n  #pragma omp target teams distribute parallel for if(target: n >= 1) map(from: y[0:n])\\n  for (i = 0; i < _PB_N; i++)\\n    y[i] = 0;\\n  for (i = 0; i < _PB_M; i++)\\n    {\\n      tmp[i] = SCALAR_VAL(0.0);\\n      for (j = 0; j < _PB_N; j++)\\n\\ttmp[i] = tmp[i] + A[i][j] * x[j];\\n      #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)y + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 2100 + 0) || (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 2100 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)y + 0) && ((__INTPTR_TYPE__)y + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)tmp + 8 * (__INTPTR_TYPE__)i || (__INTPTR_TYPE__)tmp + 8 * ((__INTPTR_TYPE__)i + 1) <= (__INTPTR_TYPE__)y + 0))\\n      for (j = 0; j < _PB_N; j++)\\n\\ty[j] = y[j] + A[i][j] * tmp[i];\\n    }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int m = M;\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(A, DATA_TYPE, M, N, m, n);\\n  POLYBENCH_1D_ARRAY_DECL(x, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(y, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(tmp, DATA_TYPE, M, m);\\n\\n  /* Initialize array(s). */\\n  init_array (m, n, POLYBENCH_ARRAY(A), POLYBENCH_ARRAY(x));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_atax (m, n,\\n\\t       POLYBENCH_ARRAY(A),\\n\\t       POLYBENCH_ARRAY(x),\\n\\t       POLYBENCH_ARRAY(y),\\n\\t       POLYBENCH_ARRAY(tmp));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(y)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(x);\\n  POLYBENCH_FREE_ARRAY(y);\\n  POLYBENCH_FREE_ARRAY(tmp);\\n\\n  return 0;\\n}\\n", "gated/bicg.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* bicg.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"bicg.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int m, int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,N,M,n,m),\\n\\t\\t DATA_TYPE POLYBENCH_1D(r,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(p,M,m))\\n{\\n  int i, j;\\n\\n  #pragma omp parallel for if(parallel: m >= 1048576)\\n  for (i = 0; i < m; i++)\\n    p[i] = (DATA_TYPE)(i % m) / m;\\n  for (i = 0; i < n; i++) {\\n    r[i] = (DATA_TYPE)(i % n) / n;\\n    #pragma omp target teams distribute parallel for if(target: (m >= 536870912) && i >= 0 && m >= 1) map(tofrom: A[i:1][0:1900])\\n    for (j = 0; j < m; j++)\\n      A[i][j] = (DATA_TYPE) (i*(j+1) % n)/n;\\n  }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int m, int n,\\n\\t\\t DATA_TYPE POLYBENCH_1D(s,M,m),\\n\\t\\t DATA_TYPE POLYBENCH_1D(q,N,n))\\n\\n{\\n  int i;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"s\\");\\n  for (i = 0; i < m; i++) {\\n    if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, s[i]);\\n  }\\n  POLYBENCH_DUMP_END(\\"s\\");\\n  POLYBENCH_DUMP_BEGIN(\\"q\\");\\n  for (i = 0; i < n; i++) {\\n    if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, q[i]);\\n  }\\n  POLYBENCH_DUMP_END(\\"q\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_bicg(int m, int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,N,M,n,m),\\n\\t\\t DATA_TYPE POLYBENCH_1D(s,M,m),\\n\\t\\t DATA_TYPE POLYBENCH_1D(q,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(p,M,m),\\n\\t\\t DATA_TYPE POLYBENCH_1D(r,N,n))\\n{\\n  int i, j;\\n\\n#pragma scop\\n  #pragma omp parallel for if(parallel: m >= 1048576)\\n  for (i = 0; i < _PB_M; i++)\\n    s[i] = 0;\\n  for (i = 0; i < _PB_N; i++)\\n    {\\n      q[i] = SCALAR_VAL(0.0);\\n      for (j = 0; j < _PB_M; j++)\\n\\t{\\n\\t  s[j] = s[j] + r[i] * A[i][j];\\n\\t  q[i] = q[i] + A[i][j] * p[j];\\n\\t}\\n    }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n  int m = M;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(A, DATA_TYPE, N, M, n, m);\\n  POLYBENCH_1D_ARRAY_DECL(s, DATA_TYPE, M, m);\\n  POLYBENCH_1D_ARRAY_DECL(q, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(p, DATA_TYPE, M, m);\\n  POLYBENCH_1D_ARRAY_DECL(r, DATA_TYPE, N, n);\\n\\n  /* Initialize array(s). */\\n  init_array (m, n,\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(r),\\n\\t      POLYBENCH_ARRAY(p));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_bicg (m, n,\\n\\t       POLYBENCH_ARRAY(A),\\n\\t       POLYBENCH_ARRAY(s),\\n\\t       POLYBENCH_ARRAY(q),\\n\\t       POLYBENCH_ARRAY(p),\\n\\t       POLYBENCH_ARRAY(r));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(m, n, POLYBENCH_ARRAY(s), POLYBENCH_ARRAY(q)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(s);\\n  POLYBENCH_FREE_ARRAY(q);\\n  POLYBENCH_FREE_ARRAY(p);\\n  POLYBENCH_FREE_ARRAY(r);\\n\\n  return 0;\\n}\\n", "naive/bicg.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* bicg.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"bicg.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int m, int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,N,M,n,m),\\n\\t\\t DATA_TYPE POLYBENCH_1D(r,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(p,M,m))\\n{\\n  int i, j;\\n\\n  #pragma omp target teams distribute parallel for if(target: m >= 1) map(from: p[0:m])\\n  for (i = 0; i < m; i++)\\n    p[i] = (DATA_TYPE)(i % m) / m;\\n  for (i = 0; i < n; i++) {\\n    r[i] = (DATA_TYPE)(i % n) / n;\\n    #pragma omp target teams distribute parallel for if(target: i >= 0 && m >= 1) map(tofrom: A[i:1][0:1900])\\n    for (j = 0; j < m; j++)\\n      A[i][j] = (DATA_TYPE) (i*(j+1) % n)/n;\\n  }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int m, int n,\\n\\t\\t DATA_TYPE POLYBENCH_1D(s,M,m),\\n\\t\\t DATA_TYPE POLYBENCH_1D(q,N,n))\\n\\n{\\n  int i;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"s\\");\\n  for (i = 0; i < m; i++) {\\n    if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, s[i]);\\n  }\\n  POLYBENCH_DUMP_END(\\"s\\");\\n  POLYBENCH_DUMP_BEGIN(\\"q\\");\\n  for (i = 0; i < n; i++) {\\n    if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, q[i]);\\n  }\\n  POLYBENCH_DUMP_END(\\"q\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_bicg(int m, int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,N,M,n,m),\\n\\t\\t DATA_TYPE POLYBENCH_1D(s,M,m),\\n\\t\\t DATA_TYPE POLYBENCH_1D(q,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(p,M,m),\\n\\t\\t DATA_TYPE POLYBENCH_1D(r,N,n))\\n{\\n  int i, j;\\n\\n#pragma scop\\n  #pragma omp target teams distribute parallel for if(target: m >= 1) map(from: s[0:m])\\n  for (i = 0; i < _PB_M; i++)\\n    s[i] = 0;\\n  for (i = 0; i < _PB_N; i++)\\n    {\\n      q[i] = SCALAR_VAL(0.0);\\n      for (j = 0; j < _PB_M; j++)\\n\\t{\\n\\t  s[j] = s[j] + r[i] * A[i][j];\\n\\t  q[i] = q[i] + A[i][j] * p[j];\\n\\t}\\n    }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n  int m = M;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(A, DATA_TYPE, N, M, n, m);\\n  POLYBENCH_1D_ARRAY_DECL(s, DATA_TYPE, M, m);\\n  POLYBENCH_1D_ARRAY_DECL(q, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(p, DATA_TYPE, M, m);\\n  POLYBENCH_1D_ARRAY_DECL(r, DATA_TYPE, N, n);\\n\\n  /* Initialize array(s). */\\n  init_array (m, n,\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(r),\\n\\t      POLYBENCH_ARRAY(p));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_bicg (m, n,\\n\\t       POLYBENCH_ARRAY(A),\\n\\t       POLYBENCH_ARRAY(s),\\n\\t       POLYBENCH_ARRAY(q),\\n\\t       POLYBENCH_ARRAY(p),\\n\\t       POLYBENCH_ARRAY(r));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(m, n, POLYBENCH_ARRAY(s), POLYBENCH_ARRAY(q)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(s);\\n  POLYBENCH_FREE_ARRAY(q);\\n  POLYBENCH_FREE_ARRAY(p);\\n  POLYBENCH_FREE_ARRAY(r);\\n\\n  return 0;\\n}\\n", "gated/doitgen.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* doitgen.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"doitgen.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int nr, int nq, int np,\\n\\t\\tDATA_TYPE POLYBENCH_3D(A,NR,NQ,NP,nr,nq,np),\\n\\t\\tDATA_TYPE POLYBENCH_2D(C4,NP,NP,np,np))\\n{\\n  int i, j, k;\\n\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ((double)(nr) * ((5 * (double)(np) * (double)(nq) + 1 * (double)(nq)) - 551088) >= 2147483648) && nr >= 1 && nq >= 1 && np >= 1) map(tofrom: A[0:nr][0:140][0:160]) private(j, k)\\n  for (i = 0; i < nr; i++)\\n    for (j = 0; j < nq; j++)\\n      for (k = 0; k < np; k++)\\n\\tA[i][j][k] = (DATA_TYPE) ((i*j + k)%np) / np;\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ((double)(np) * ((4 * (double)(np)) - 3937) >= 2147483648) && np >= 1) map(tofrom: C4[0:np][0:160]) private(j)\\n  for (i = 0; i < np; i++)\\n    for (j = 0; j < np; j++)\\n      C4[i][j] = (DATA_TYPE) (i*j % np) / np;\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int nr, int nq, int np,\\n\\t\\t DATA_TYPE POLYBENCH_3D(A,NR,NQ,NP,nr,nq,np))\\n{\\n  int i, j, k;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"A\\");\\n  for (i = 0; i < nr; i++)\\n    for (j = 0; j < nq; j++)\\n      for (k = 0; k < np; k++) {\\n\\tif ((i*nq*np+j*np+k) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n\\tfprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j][k]);\\n      }\\n  POLYBENCH_DUMP_END(\\"A\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nvoid kernel_doitgen(int nr, int nq, int np,\\n\\t\\t    DATA_TYPE POLYBENCH_3D(A,NR,NQ,NP,nr,nq,np),\\n\\t\\t    DATA_TYPE POLYBENCH_2D(C4,NP,NP,np,np),\\n\\t\\t    DATA_TYPE POLYBENCH_1D(sum,NP,np))\\n{\\n  int r, q, p, s;\\n\\n#pragma scop\\n  for (r = 0; r < _PB_NR; r++)\\n    for (q = 0; q < _PB_NQ; q++)  {\\n      for (p = 0; p < _PB_NP; p++)  {\\n\\tsum[p] = SCALAR_VAL(0.0);\\n\\tfor (s = 0; s < _PB_NP; s++)\\n\\t  sum[p] += A[r][q][s] * C4[s][p];\\n      }\\n      #pragma omp parallel for if(parallel: (np >= 1048576) && ((__INTPTR_TYPE__)A + 8 * ((((__INTPTR_TYPE__)r) * 140 + (__INTPTR_TYPE__)q) * 160 + ((__INTPTR_TYPE__)np - 1) + 1) <= (__INTPTR_TYPE__)sum + 0 || (__INTPTR_TYPE__)sum + 8 * (((__INTPTR_TYPE__)np - 1) + 1) <= (__INTPTR_TYPE__)A + 8 * ((((__INTPTR_TYPE__)r) * 140 + (__INTPTR_TYPE__)q) * 160 + 0)))\\n      for (p = 0; p < _PB_NP; p++)\\n\\tA[r][q][p] = sum[p];\\n    }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int nr = NR;\\n  int nq = NQ;\\n  int np = NP;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_3D_ARRAY_DECL(A,DATA_TYPE,NR,NQ,NP,nr,nq,np);\\n  POLYBENCH_1D_ARRAY_DECL(sum,DATA_TYPE,NP,np);\\n  POLYBENCH_2D_ARRAY_DECL(C4,DATA_TYPE,NP,NP,np,np);\\n\\n  /* Initialize array(s). */\\n  init_array (nr, nq, np,\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(C4));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_doitgen (nr, nq, np,\\n\\t\\t  POLYBENCH_ARRAY(A),\\n\\t\\t  POLYBENCH_ARRAY(C4),\\n\\t\\t  POLYBENCH_ARRAY(sum));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(nr, nq, np,  POLYBENCH_ARRAY(A)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(sum);\\n  POLYBENCH_FREE_ARRAY(C4);\\n\\n  return 0;\\n}\\n", "naive/doitgen.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* doitgen.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"doitgen.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int nr, int nq, int np,\\n\\t\\tDATA_TYPE POLYBENCH_3D(A,NR,NQ,NP,nr,nq,np),\\n\\t\\tDATA_TYPE POLYBENCH_2D(C4,NP,NP,np,np))\\n{\\n  int i, j, k;\\n\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: nr >= 1 && nq >= 1 && np >= 1) map(tofrom: A[0:nr][0:140][0:160]) private(j, k)\\n  for (i = 0; i < nr; i++)\\n    for (j = 0; j < nq; j++)\\n      for (k = 0; k < np; k++)\\n\\tA[i][j][k] = (DATA_TYPE) ((i*j + k)%np) / np;\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: np >= 1) map(tofrom: C4[0:np][0:160]) private(j)\\n  for (i = 0; i < np; i++)\\n    for (j = 0; j < np; j++)\\n      C4[i][j] = (DATA_TYPE) (i*j % np) / np;\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int nr, int nq, int np,\\n\\t\\t DATA_TYPE POLYBENCH_3D(A,NR,NQ,NP,nr,nq,np))\\n{\\n  int i, j, k;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"A\\");\\n  for (i = 0; i < nr; i++)\\n    for (j = 0; j < nq; j++)\\n      for (k = 0; k < np; k++) {\\n\\tif ((i*nq*np+j*np+k) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n\\tfprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j][k]);\\n      }\\n  POLYBENCH_DUMP_END(\\"A\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nvoid kernel_doitgen(int nr, int nq, int np,\\n\\t\\t    DATA_TYPE POLYBENCH_3D(A,NR,NQ,NP,nr,nq,np),\\n\\t\\t    DATA_TYPE POLYBENCH_2D(C4,NP,NP,np,np),\\n\\t\\t    DATA_TYPE POLYBENCH_1D(sum,NP,np))\\n{\\n  int r, q, p, s;\\n\\n#pragma scop\\n  for (r = 0; r < _PB_NR; r++)\\n    for (q = 0; q < _PB_NQ; q++)  {\\n      for (p = 0; p < _PB_NP; p++)  {\\n\\tsum[p] = SCALAR_VAL(0.0);\\n\\tfor (s = 0; s < _PB_NP; s++)\\n\\t  sum[p] += A[r][q][s] * C4[s][p];\\n      }\\n      #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)A + 8 * ((((__INTPTR_TYPE__)r) * 140 + (__INTPTR_TYPE__)q) * 160 + ((__INTPTR_TYPE__)np - 1) + 1) <= (__INTPTR_TYPE__)sum + 0 || (__INTPTR_TYPE__)sum + 8 * (((__INTPTR_TYPE__)np - 1) + 1) <= (__INTPTR_TYPE__)A + 8 * ((((__INTPTR_TYPE__)r) * 140 + (__INTPTR_TYPE__)q) * 160 + 0)))\\n      for (p = 0; p < _PB_NP; p++)\\n\\tA[r][q][p] = sum[p];\\n    }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int nr = NR;\\n  int nq = NQ;\\n  int np = NP;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_3D_ARRAY_DECL(A,DATA_TYPE,NR,NQ,NP,nr,nq,np);\\n  POLYBENCH_1D_ARRAY_DECL(sum,DATA_TYPE,NP,np);\\n  POLYBENCH_2D_ARRAY_DECL(C4,DATA_TYPE,NP,NP,np,np);\\n\\n  /* Initialize array(s). */\\n  init_array (nr, nq, np,\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(C4));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_doitgen (nr, nq, np,\\n\\t\\t  POLYBENCH_ARRAY(A),\\n\\t\\t  POLYBENCH_ARRAY(C4),\\n\\t\\t  POLYBENCH_ARRAY(sum));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(nr, nq, np,  POLYBENCH_ARRAY(A)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(sum);\\n  POLYBENCH_FREE_ARRAY(C4);\\n\\n  return 0;\\n}\\n", "gated/mvt.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* mvt.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"mvt.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int n,\\n\\t\\tDATA_TYPE POLYBENCH_1D(x1,N,n),\\n\\t\\tDATA_TYPE POLYBENCH_1D(x2,N,n),\\n\\t\\tDATA_TYPE POLYBENCH_1D(y_1,N,n),\\n\\t\\tDATA_TYPE POLYBENCH_1D(y_2,N,n),\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,N,N,n,n))\\n{\\n  int i, j;\\n\\n  for (i = 0; i < n; i++)\\n    {\\n      x1[i] = (DATA_TYPE) (i % n) / n;\\n      x2[i] = (DATA_TYPE) ((i + 1) % n) / n;\\n      y_1[i] = (DATA_TYPE) ((i + 3) % n) / n;\\n      y_2[i] = (DATA_TYPE) ((i + 4) % n) / n;\\n      #pragma omp target teams distribute parallel for if(target: (n >= 536870912) && i >= 0 && n >= 1) map(tofrom: A[i:1][0:2000])\\n      for (j = 0; j < n; j++)\\n\\tA[i][j] = (DATA_TYPE) (i*j % n) / n;\\n    }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_1D(x1,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(x2,N,n))\\n\\n{\\n  int i;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"x1\\");\\n  for (i = 0; i < n; i++) {\\n    if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, x1[i]);\\n  }\\n  POLYBENCH_DUMP_END(\\"x1\\");\\n\\n  POLYBENCH_DUMP_BEGIN(\\"x2\\");\\n  for (i = 0; i < n; i++) {\\n    if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, x2[i]);\\n  }\\n  POLYBENCH_DUMP_END(\\"x2\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_mvt(int n,\\n\\t\\tDATA_TYPE POLYBENCH_1D(x1,N,n),\\n\\t\\tDATA_TYPE POLYBENCH_1D(x2,N,n),\\n\\t\\tDATA_TYPE POLYBENCH_1D(y_1,N,n),\\n\\t\\tDATA_TYPE POLYBENCH_1D(y_2,N,n),\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,N,N,n,n))\\n{\\n  int i, j;\\n\\n#pragma scop\\n  for (i = 0; i < _PB_N; i++)\\n    for (j = 0; j < _PB_N; j++)\\n      x1[i] = x1[i] + A[i][j] * y_1[j];\\n  for (i = 0; i < _PB_N; i++)\\n    for (j = 0; j < _PB_N; j++)\\n      x2[i] = x2[i] + A[j][i] * y_2[j];\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(A, DATA_TYPE, N, N, n, n);\\n  POLYBENCH_1D_ARRAY_DECL(x1, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(x2, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(y_1, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(y_2, DATA_TYPE, N, n);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (n,\\n\\t      POLYBENCH_ARRAY(x1),\\n\\t      POLYBENCH_ARRAY(x2),\\n\\t      POLYBENCH_ARRAY(y_1),\\n\\t      POLYBENCH_ARRAY(y_2),\\n\\t      POLYBENCH_ARRAY(A));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_mvt (n,\\n\\t      POLYBENCH_ARRAY(x1),\\n\\t      POLYBENCH_ARRAY(x2),\\n\\t      POLYBENCH_ARRAY(y_1),\\n\\t      POLYBENCH_ARRAY(y_2),\\n\\t      POLYBENCH_ARRAY(A));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(x1), POLYBENCH_ARRAY(x2)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(x1);\\n  POLYBENCH_FREE_ARRAY(x2);\\n  POLYBENCH_FREE_ARRAY(y_1);\\n  POLYBENCH_FREE_ARRAY(y_2);\\n\\n  return 0;\\n}\\n", "naive/mvt.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* mvt.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"mvt.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int n,\\n\\t\\tDATA_TYPE POLYBENCH_1D(x1,N,n),\\n\\t\\tDATA_TYPE POLYBENCH_1D(x2,N,n),\\n\\t\\tDATA_TYPE POLYBENCH_1D(y_1,N,n),\\n\\t\\tDATA_TYPE POLYBENCH_1D(y_2,N,n),\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,N,N,n,n))\\n{\\n  int i, j;\\n\\n  for (i = 0; i < n; i++)\\n    {\\n      x1[i] = (DATA_TYPE) (i % n) / n;\\n      x2[i] = (DATA_TYPE) ((i + 1) % n) / n;\\n      y_1[i] = (DATA_TYPE) ((i + 3) % n) / n;\\n      y_2[i] = (DATA_TYPE) ((i + 4) % n) / n;\\n      #pragma omp target teams distribute parallel for if(target: i >= 0 && n >= 1) map(tofrom: A[i:1][0:2000])\\n      for (j = 0; j < n; j++)\\n\\tA[i][j] = (DATA_TYPE) (i*j % n) / n;\\n    }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_1D(x1,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(x2,N,n))\\n\\n{\\n  int i;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"x1\\");\\n  for (i = 0; i < n; i++) {\\n    if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, x1[i]);\\n  }\\n  POLYBENCH_DUMP_END(\\"x1\\");\\n\\n  POLYBENCH_DUMP_BEGIN(\\"x2\\");\\n  for (i = 0; i < n; i++) {\\n    if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, x2[i]);\\n  }\\n  POLYBENCH_DUMP_END(\\"x2\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_mvt(int n,\\n\\t\\tDATA_TYPE POLYBENCH_1D(x1,N,n),\\n\\t\\tDATA_TYPE POLYBENCH_1D(x2,N,n),\\n\\t\\tDATA_TYPE POLYBENCH_1D(y_1,N,n),\\n\\t\\tDATA_TYPE POLYBENCH_1D(y_2,N,n),\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,N,N,n,n))\\n{\\n  int i, j;\\n\\n#pragma scop\\n  for (i = 0; i < _PB_N; i++)\\n    for (j = 0; j < _PB_N; j++)\\n      x1[i] = x1[i] + A[i][j] * y_1[j];\\n  for (i = 0; i < _PB_N; i++)\\n    for (j = 0; j < _PB_N; j++)\\n      x2[i] = x2[i] + A[j][i] * y_2[j];\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(A, DATA_TYPE, N, N, n, n);\\n  POLYBENCH_1D_ARRAY_DECL(x1, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(x2, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(y_1, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(y_2, DATA_TYPE, N, n);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (n,\\n\\t      POLYBENCH_ARRAY(x1),\\n\\t      POLYBENCH_ARRAY(x2),\\n\\t      POLYBENCH_ARRAY(y_1),\\n\\t      POLYBENCH_ARRAY(y_2),\\n\\t      POLYBENCH_ARRAY(A));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_mvt (n,\\n\\t      POLYBENCH_ARRAY(x1),\\n\\t      POLYBENCH_ARRAY(x2),\\n\\t      POLYBENCH_ARRAY(y_1),\\n\\t      POLYBENCH_ARRAY(y_2),\\n\\t      POLYBENCH_ARRAY(A));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(x1), POLYBENCH_ARRAY(x2)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(x1);\\n  POLYBENCH_FREE_ARRAY(x2);\\n  POLYBENCH_FREE_ARRAY(y_1);\\n  POLYBENCH_FREE_ARRAY(y_2);\\n\\n  return 0;\\n}\\n", "gated/gemm.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* gemm.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"gemm.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int ni, int nj, int nk,\\n\\t\\tDATA_TYPE *alpha,\\n\\t\\tDATA_TYPE *beta,\\n\\t\\tDATA_TYPE POLYBENCH_2D(C,NI,NJ,ni,nj),\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,NI,NK,ni,nk),\\n\\t\\tDATA_TYPE POLYBENCH_2D(B,NK,NJ,nk,nj))\\n{\\n  int i, j;\\n\\n  *alpha = 1.5;\\n  *beta = 1.2;\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ((double)(ni) * ((5 * (double)(nj)) - 27063) >= 1073741824) && ni >= 1 && nj >= 1) map(tofrom: C[0:ni][0:1100]) private(j)\\n  for (i = 0; i < ni; i++)\\n    for (j = 0; j < nj; j++)\\n      C[i][j] = (DATA_TYPE) ((i*j+1) % ni) / ni;\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ((double)(ni) * ((5 * (double)(nk)) - 29523) >= 1073741824) && ni >= 1 && nk >= 1) map(tofrom: A[0:ni][0:1200]) private(j)\\n  for (i = 0; i < ni; i++)\\n    for (j = 0; j < nk; j++)\\n      A[i][j] = (DATA_TYPE) (i*(j+1) % nk) / nk;\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ((double)(nk) * ((5 * (double)(nj)) - 27063) >= 1073741824) && nk >= 1 && nj >= 1) map(tofrom: B[0:nk][0:1100]) private(j)\\n  for (i = 0; i < nk; i++)\\n    for (j = 0; j < nj; j++)\\n      B[i][j] = (DATA_TYPE) (i*(j+2) % nj) / nj;\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int ni, int nj,\\n\\t\\t DATA_TYPE POLYBENCH_2D(C,NI,NJ,ni,nj))\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"C\\");\\n  for (i = 0; i < ni; i++)\\n    for (j = 0; j < nj; j++) {\\n\\tif ((i * ni + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n\\tfprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, C[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"C\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_gemm(int ni, int nj, int nk,\\n\\t\\t DATA_TYPE alpha,\\n\\t\\t DATA_TYPE beta,\\n\\t\\t DATA_TYPE POLYBENCH_2D(C,NI,NJ,ni,nj),\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,NI,NK,ni,nk),\\n\\t\\t DATA_TYPE POLYBENCH_2D(B,NK,NJ,nk,nj))\\n{\\n  int i, j, k;\\n\\n//BLAS PARAMS\\n//TRANSA = \'N\'\\n//TRANSB = \'N\'\\n// => Form C := alpha*A*B + beta*C,\\n//A is NIxNK\\n//B is NKxNJ\\n//C is NIxNJ\\n#pragma scop\\n  for (i = 0; i < _PB_NI; i++) {\\n    #pragma omp target teams distribute parallel for if(target: (nj >= 67108864) && i >= 0 && nj >= 1) map(tofrom: C[i:1][0:1100])\\n    for (j = 0; j < _PB_NJ; j++)\\n\\tC[i][j] *= beta;\\n    for (k = 0; k < _PB_NK; k++) {\\n       #pragma omp parallel for if(parallel: (nj >= 524288) && ((__INTPTR_TYPE__)C + 8 * (((__INTPTR_TYPE__)i) * 1100 + ((__INTPTR_TYPE__)nj - 1) + 1) <= (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1200 + (__INTPTR_TYPE__)k) || (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1200 + (__INTPTR_TYPE__)k + 1) <= (__INTPTR_TYPE__)C + 8 * (((__INTPTR_TYPE__)i) * 1100 + 0)) && ((__INTPTR_TYPE__)C + 8 * (((__INTPTR_TYPE__)i) * 1100 + ((__INTPTR_TYPE__)nj - 1) + 1) <= (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)k) * 1100 + 0) || (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)k) * 1100 + ((__INTPTR_TYPE__)nj - 1) + 1) <= (__INTPTR_TYPE__)C + 8 * (((__INTPTR_TYPE__)i) * 1100 + 0)))\\n       for (j = 0; j < _PB_NJ; j++)\\n\\t  C[i][j] += alpha * A[i][k] * B[k][j];\\n    }\\n  }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int ni = NI;\\n  int nj = NJ;\\n  int nk = NK;\\n\\n  /* Variable declaration/allocation. */\\n  DATA_TYPE alpha;\\n  DATA_TYPE beta;\\n  POLYBENCH_2D_ARRAY_DECL(C,DATA_TYPE,NI,NJ,ni,nj);\\n  POLYBENCH_2D_ARRAY_DECL(A,DATA_TYPE,NI,NK,ni,nk);\\n  POLYBENCH_2D_ARRAY_DECL(B,DATA_TYPE,NK,NJ,nk,nj);\\n\\n  /* Initialize array(s). */\\n  init_array (ni, nj, nk, &alpha, &beta,\\n\\t      POLYBENCH_ARRAY(C),\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(B));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_gemm (ni, nj, nk,\\n\\t       alpha, beta,\\n\\t       POLYBENCH_ARRAY(C),\\n\\t       POLYBENCH_ARRAY(A),\\n\\t       POLYBENCH_ARRAY(B));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(ni, nj,  POLYBENCH_ARRAY(C)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(C);\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(B);\\n\\n  return 0;\\n}\\n", "naive/gemm.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* gemm.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"gemm.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int ni, int nj, int nk,\\n\\t\\tDATA_TYPE *alpha,\\n\\t\\tDATA_TYPE *beta,\\n\\t\\tDATA_TYPE POLYBENCH_2D(C,NI,NJ,ni,nj),\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,NI,NK,ni,nk),\\n\\t\\tDATA_TYPE POLYBENCH_2D(B,NK,NJ,nk,nj))\\n{\\n  int i, j;\\n\\n  *alpha = 1.5;\\n  *beta = 1.2;\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ni >= 1 && nj >= 1) map(tofrom: C[0:ni][0:1100]) private(j)\\n  for (i = 0; i < ni; i++)\\n    for (j = 0; j < nj; j++)\\n      C[i][j] = (DATA_TYPE) ((i*j+1) % ni) / ni;\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ni >= 1 && nk >= 1) map(tofrom: A[0:ni][0:1200]) private(j)\\n  for (i = 0; i < ni; i++)\\n    for (j = 0; j < nk; j++)\\n      A[i][j] = (DATA_TYPE) (i*(j+1) % nk) / nk;\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: nk >= 1 && nj >= 1) map(tofrom: B[0:nk][0:1100]) private(j)\\n  for (i = 0; i < nk; i++)\\n    for (j = 0; j < nj; j++)\\n      B[i][j] = (DATA_TYPE) (i*(j+2) % nj) / nj;\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int ni, int nj,\\n\\t\\t DATA_TYPE POLYBENCH_2D(C,NI,NJ,ni,nj))\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"C\\");\\n  for (i = 0; i < ni; i++)\\n    for (j = 0; j < nj; j++) {\\n\\tif ((i * ni + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n\\tfprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, C[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"C\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_gemm(int ni, int nj, int nk,\\n\\t\\t DATA_TYPE alpha,\\n\\t\\t DATA_TYPE beta,\\n\\t\\t DATA_TYPE POLYBENCH_2D(C,NI,NJ,ni,nj),\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,NI,NK,ni,nk),\\n\\t\\t DATA_TYPE POLYBENCH_2D(B,NK,NJ,nk,nj))\\n{\\n  int i, j, k;\\n\\n//BLAS PARAMS\\n//TRANSA = \'N\'\\n//TRANSB = \'N\'\\n// => Form C := alpha*A*B + beta*C,\\n//A is NIxNK\\n//B is NKxNJ\\n//C is NIxNJ\\n#pragma scop\\n  for (i = 0; i < _PB_NI; i++) {\\n    #pragma omp target teams distribute parallel for if(target: i >= 0 && nj >= 1) map(tofrom: C[i:1][0:1100])\\n    for (j = 0; j < _PB_NJ; j++)\\n\\tC[i][j] *= beta;\\n    for (k = 0; k < _PB_NK; k++) {\\n       #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)C + 8 * (((__INTPTR_TYPE__)i) * 1100 + ((__INTPTR_TYPE__)nj - 1) + 1) <= (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1200 + (__INTPTR_TYPE__)k) || (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1200 + (__INTPTR_TYPE__)k + 1) <= (__INTPTR_TYPE__)C + 8 * (((__INTPTR_TYPE__)i) * 1100 + 0)) && ((__INTPTR_TYPE__)C + 8 * (((__INTPTR_TYPE__)i) * 1100 + ((__INTPTR_TYPE__)nj - 1) + 1) <= (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)k) * 1100 + 0) || (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)k) * 1100 + ((__INTPTR_TYPE__)nj - 1) + 1) <= (__INTPTR_TYPE__)C + 8 * (((__INTPTR_TYPE__)i) * 1100 + 0)))\\n       for (j = 0; j < _PB_NJ; j++)\\n\\t  C[i][j] += alpha * A[i][k] * B[k][j];\\n    }\\n  }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int ni = NI;\\n  int nj = NJ;\\n  int nk = NK;\\n\\n  /* Variable declaration/allocation. */\\n  DATA_TYPE alpha;\\n  DATA_TYPE beta;\\n  POLYBENCH_2D_ARRAY_DECL(C,DATA_TYPE,NI,NJ,ni,nj);\\n  POLYBENCH_2D_ARRAY_DECL(A,DATA_TYPE,NI,NK,ni,nk);\\n  POLYBENCH_2D_ARRAY_DECL(B,DATA_TYPE,NK,NJ,nk,nj);\\n\\n  /* Initialize array(s). */\\n  init_array (ni, nj, nk, &alpha, &beta,\\n\\t      POLYBENCH_ARRAY(C),\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(B));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_gemm (ni, nj, nk,\\n\\t       alpha, beta,\\n\\t       POLYBENCH_ARRAY(C),\\n\\t       POLYBENCH_ARRAY(A),\\n\\t       POLYBENCH_ARRAY(B));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(ni, nj,  POLYBENCH_ARRAY(C)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(C);\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(B);\\n\\n  return 0;\\n}\\n", "gated/gemver.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* gemver.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"gemver.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int n,\\n\\t\\t DATA_TYPE *alpha,\\n\\t\\t DATA_TYPE *beta,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,N,N,n,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(u1,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(v1,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(u2,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(v2,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(w,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(x,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(y,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(z,N,n))\\n{\\n  int i, j;\\n\\n  *alpha = 1.5;\\n  *beta = 1.2;\\n\\n  DATA_TYPE fn = (DATA_TYPE)n;\\n\\n  for (i = 0; i < n; i++)\\n    {\\n      u1[i] = i;\\n      u2[i] = ((i+1)/fn)/2.0;\\n      v1[i] = ((i+1)/fn)/4.0;\\n      v2[i] = ((i+1)/fn)/6.0;\\n      y[i] = ((i+1)/fn)/8.0;\\n      z[i] = ((i+1)/fn)/9.0;\\n      x[i] = 0.0;\\n      w[i] = 0.0;\\n      #pragma omp target teams distribute parallel for if(target: (n >= 536870912) && i >= 0 && n >= 1) map(tofrom: A[i:1][0:2000])\\n      for (j = 0; j < n; j++)\\n        A[i][j] = (DATA_TYPE) (i*j % n) / n;\\n    }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_1D(w,N,n))\\n{\\n  int i;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"w\\");\\n  for (i = 0; i < n; i++) {\\n    if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, w[i]);\\n  }\\n  POLYBENCH_DUMP_END(\\"w\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_gemver(int n,\\n\\t\\t   DATA_TYPE alpha,\\n\\t\\t   DATA_TYPE beta,\\n\\t\\t   DATA_TYPE POLYBENCH_2D(A,N,N,n,n),\\n\\t\\t   DATA_TYPE POLYBENCH_1D(u1,N,n),\\n\\t\\t   DATA_TYPE POLYBENCH_1D(v1,N,n),\\n\\t\\t   DATA_TYPE POLYBENCH_1D(u2,N,n),\\n\\t\\t   DATA_TYPE POLYBENCH_1D(v2,N,n),\\n\\t\\t   DATA_TYPE POLYBENCH_1D(w,N,n),\\n\\t\\t   DATA_TYPE POLYBENCH_1D(x,N,n),\\n\\t\\t   DATA_TYPE POLYBENCH_1D(y,N,n),\\n\\t\\t   DATA_TYPE POLYBENCH_1D(z,N,n))\\n{\\n  int i, j;\\n\\n#pragma scop\\n\\n  for (i = 0; i < _PB_N; i++)\\n    #pragma omp parallel for if(parallel: (n >= 262144) && ((__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 2000 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)u1 + 8 * (__INTPTR_TYPE__)i || (__INTPTR_TYPE__)u1 + 8 * ((__INTPTR_TYPE__)i + 1) <= (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 2000 + 0)) && ((__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 2000 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)v1 + 0 || (__INTPTR_TYPE__)v1 + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 2000 + 0)) && ((__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 2000 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)u2 + 8 * (__INTPTR_TYPE__)i || (__INTPTR_TYPE__)u2 + 8 * ((__INTPTR_TYPE__)i + 1) <= (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 2000 + 0)) && ((__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 2000 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)v2 + 0 || (__INTPTR_TYPE__)v2 + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 2000 + 0)))\\n    for (j = 0; j < _PB_N; j++)\\n      A[i][j] = A[i][j] + u1[i] * v1[j] + u2[i] * v2[j];\\n\\n  for (i = 0; i < _PB_N; i++)\\n    for (j = 0; j < _PB_N; j++)\\n      x[i] = x[i] + beta * A[j][i] * y[j];\\n\\n  #pragma omp parallel for if(parallel: (n >= 524288) && ((__INTPTR_TYPE__)x + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)z + 0 || (__INTPTR_TYPE__)z + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)x + 0))\\n  for (i = 0; i < _PB_N; i++)\\n    x[i] = x[i] + z[i];\\n\\n  for (i = 0; i < _PB_N; i++)\\n    for (j = 0; j < _PB_N; j++)\\n      w[i] = w[i] +  alpha * A[i][j] * x[j];\\n\\n#pragma endscop\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  DATA_TYPE alpha;\\n  DATA_TYPE beta;\\n  POLYBENCH_2D_ARRAY_DECL(A, DATA_TYPE, N, N, n, n);\\n  POLYBENCH_1D_ARRAY_DECL(u1, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(v1, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(u2, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(v2, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(w, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(x, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(y, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(z, DATA_TYPE, N, n);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (n, &alpha, &beta,\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(u1),\\n\\t      POLYBENCH_ARRAY(v1),\\n\\t      POLYBENCH_ARRAY(u2),\\n\\t      POLYBENCH_ARRAY(v2),\\n\\t      POLYBENCH_ARRAY(w),\\n\\t      POLYBENCH_ARRAY(x),\\n\\t      POLYBENCH_ARRAY(y),\\n\\t      POLYBENCH_ARRAY(z));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_gemver (n, alpha, beta,\\n\\t\\t POLYBENCH_ARRAY(A),\\n\\t\\t POLYBENCH_ARRAY(u1),\\n\\t\\t POLYBENCH_ARRAY(v1),\\n\\t\\t POLYBENCH_ARRAY(u2),\\n\\t\\t POLYBENCH_ARRAY(v2),\\n\\t\\t POLYBENCH_ARRAY(w),\\n\\t\\t POLYBENCH_ARRAY(x),\\n\\t\\t POLYBENCH_ARRAY(y),\\n\\t\\t POLYBENCH_ARRAY(z));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(w)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(u1);\\n  POLYBENCH_FREE_ARRAY(v1);\\n  POLYBENCH_FREE_ARRAY(u2);\\n  POLYBENCH_FREE_ARRAY(v2);\\n  POLYBENCH_FREE_ARRAY(w);\\n  POLYBENCH_FREE_ARRAY(x);\\n  POLYBENCH_FREE_ARRAY(y);\\n  POLYBENCH_FREE_ARRAY(z);\\n\\n  return 0;\\n}\\n", "naive/gemver.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* gemver.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"gemver.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int n,\\n\\t\\t DATA_TYPE *alpha,\\n\\t\\t DATA_TYPE *beta,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,N,N,n,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(u1,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(v1,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(u2,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(v2,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(w,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(x,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(y,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(z,N,n))\\n{\\n  int i, j;\\n\\n  *alpha = 1.5;\\n  *beta = 1.2;\\n\\n  DATA_TYPE fn = (DATA_TYPE)n;\\n\\n  for (i = 0; i < n; i++)\\n    {\\n      u1[i] = i;\\n      u2[i] = ((i+1)/fn)/2.0;\\n      v1[i] = ((i+1)/fn)/4.0;\\n      v2[i] = ((i+1)/fn)/6.0;\\n      y[i] = ((i+1)/fn)/8.0;\\n      z[i] = ((i+1)/fn)/9.0;\\n      x[i] = 0.0;\\n      w[i] = 0.0;\\n      #pragma omp target teams distribute parallel for if(target: i >= 0 && n >= 1) map(tofrom: A[i:1][0:2000])\\n      for (j = 0; j < n; j++)\\n        A[i][j] = (DATA_TYPE) (i*j % n) / n;\\n    }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_1D(w,N,n))\\n{\\n  int i;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"w\\");\\n  for (i = 0; i < n; i++) {\\n    if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, w[i]);\\n  }\\n  POLYBENCH_DUMP_END(\\"w\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_gemver(int n,\\n\\t\\t   DATA_TYPE alpha,\\n\\t\\t   DATA_TYPE beta,\\n\\t\\t   DATA_TYPE POLYBENCH_2D(A,N,N,n,n),\\n\\t\\t   DATA_TYPE POLYBENCH_1D(u1,N,n),\\n\\t\\t   DATA_TYPE POLYBENCH_1D(v1,N,n),\\n\\t\\t   DATA_TYPE POLYBENCH_1D(u2,N,n),\\n\\t\\t   DATA_TYPE POLYBENCH_1D(v2,N,n),\\n\\t\\t   DATA_TYPE POLYBENCH_1D(w,N,n),\\n\\t\\t   DATA_TYPE POLYBENCH_1D(x,N,n),\\n\\t\\t   DATA_TYPE POLYBENCH_1D(y,N,n),\\n\\t\\t   DATA_TYPE POLYBENCH_1D(z,N,n))\\n{\\n  int i, j;\\n\\n#pragma scop\\n\\n  for (i = 0; i < _PB_N; i++)\\n    #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 2000 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)u1 + 8 * (__INTPTR_TYPE__)i || (__INTPTR_TYPE__)u1 + 8 * ((__INTPTR_TYPE__)i + 1) <= (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 2000 + 0)) && ((__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 2000 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)v1 + 0 || (__INTPTR_TYPE__)v1 + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 2000 + 0)) && ((__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 2000 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)u2 + 8 * (__INTPTR_TYPE__)i || (__INTPTR_TYPE__)u2 + 8 * ((__INTPTR_TYPE__)i + 1) <= (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 2000 + 0)) && ((__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 2000 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)v2 + 0 || (__INTPTR_TYPE__)v2 + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 2000 + 0)))\\n    for (j = 0; j < _PB_N; j++)\\n      A[i][j] = A[i][j] + u1[i] * v1[j] + u2[i] * v2[j];\\n\\n  for (i = 0; i < _PB_N; i++)\\n    for (j = 0; j < _PB_N; j++)\\n      x[i] = x[i] + beta * A[j][i] * y[j];\\n\\n  #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)x + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)z + 0 || (__INTPTR_TYPE__)z + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)x + 0))\\n  for (i = 0; i < _PB_N; i++)\\n    x[i] = x[i] + z[i];\\n\\n  for (i = 0; i < _PB_N; i++)\\n    for (j = 0; j < _PB_N; j++)\\n      w[i] = w[i] +  alpha * A[i][j] * x[j];\\n\\n#pragma endscop\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  DATA_TYPE alpha;\\n  DATA_TYPE beta;\\n  POLYBENCH_2D_ARRAY_DECL(A, DATA_TYPE, N, N, n, n);\\n  POLYBENCH_1D_ARRAY_DECL(u1, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(v1, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(u2, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(v2, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(w, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(x, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(y, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(z, DATA_TYPE, N, n);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (n, &alpha, &beta,\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(u1),\\n\\t      POLYBENCH_ARRAY(v1),\\n\\t      POLYBENCH_ARRAY(u2),\\n\\t      POLYBENCH_ARRAY(v2),\\n\\t      POLYBENCH_ARRAY(w),\\n\\t      POLYBENCH_ARRAY(x),\\n\\t      POLYBENCH_ARRAY(y),\\n\\t      POLYBENCH_ARRAY(z));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_gemver (n, alpha, beta,\\n\\t\\t POLYBENCH_ARRAY(A),\\n\\t\\t POLYBENCH_ARRAY(u1),\\n\\t\\t POLYBENCH_ARRAY(v1),\\n\\t\\t POLYBENCH_ARRAY(u2),\\n\\t\\t POLYBENCH_ARRAY(v2),\\n\\t\\t POLYBENCH_ARRAY(w),\\n\\t\\t POLYBENCH_ARRAY(x),\\n\\t\\t POLYBENCH_ARRAY(y),\\n\\t\\t POLYBENCH_ARRAY(z));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(w)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(u1);\\n  POLYBENCH_FREE_ARRAY(v1);\\n  POLYBENCH_FREE_ARRAY(u2);\\n  POLYBENCH_FREE_ARRAY(v2);\\n  POLYBENCH_FREE_ARRAY(w);\\n  POLYBENCH_FREE_ARRAY(x);\\n  POLYBENCH_FREE_ARRAY(y);\\n  POLYBENCH_FREE_ARRAY(z);\\n\\n  return 0;\\n}\\n", "gated/gesummv.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* gesummv.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"gesummv.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int n,\\n\\t\\tDATA_TYPE *alpha,\\n\\t\\tDATA_TYPE *beta,\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,N,N,n,n),\\n\\t\\tDATA_TYPE POLYBENCH_2D(B,N,N,n,n),\\n\\t\\tDATA_TYPE POLYBENCH_1D(x,N,n))\\n{\\n  int i, j;\\n\\n  *alpha = 1.5;\\n  *beta = 1.2;\\n  for (i = 0; i < n; i++)\\n    {\\n      x[i] = (DATA_TYPE)( i % n) / n;\\n      #pragma omp parallel for if(parallel: (n >= 1048576) && ((__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) || (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)))\\n      for (j = 0; j < n; j++) {\\n\\tA[i][j] = (DATA_TYPE) ((i*j+1) % n) / n;\\n\\tB[i][j] = (DATA_TYPE) ((i*j+2) % n) / n;\\n      }\\n    }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_1D(y,N,n))\\n\\n{\\n  int i;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"y\\");\\n  for (i = 0; i < n; i++) {\\n    if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, y[i]);\\n  }\\n  POLYBENCH_DUMP_END(\\"y\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_gesummv(int n,\\n\\t\\t    DATA_TYPE alpha,\\n\\t\\t    DATA_TYPE beta,\\n\\t\\t    DATA_TYPE POLYBENCH_2D(A,N,N,n,n),\\n\\t\\t    DATA_TYPE POLYBENCH_2D(B,N,N,n,n),\\n\\t\\t    DATA_TYPE POLYBENCH_1D(tmp,N,n),\\n\\t\\t    DATA_TYPE POLYBENCH_1D(x,N,n),\\n\\t\\t    DATA_TYPE POLYBENCH_1D(y,N,n))\\n{\\n  int i, j;\\n\\n#pragma scop\\n  for (i = 0; i < _PB_N; i++)\\n    {\\n      tmp[i] = SCALAR_VAL(0.0);\\n      y[i] = SCALAR_VAL(0.0);\\n      for (j = 0; j < _PB_N; j++)\\n\\t{\\n\\t  tmp[i] = A[i][j] * x[j] + tmp[i];\\n\\t  y[i] = B[i][j] * x[j] + y[i];\\n\\t}\\n      y[i] = alpha * tmp[i] + beta * y[i];\\n    }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  DATA_TYPE alpha;\\n  DATA_TYPE beta;\\n  POLYBENCH_2D_ARRAY_DECL(A, DATA_TYPE, N, N, n, n);\\n  POLYBENCH_2D_ARRAY_DECL(B, DATA_TYPE, N, N, n, n);\\n  POLYBENCH_1D_ARRAY_DECL(tmp, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(x, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(y, DATA_TYPE, N, n);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (n, &alpha, &beta,\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(B),\\n\\t      POLYBENCH_ARRAY(x));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_gesummv (n, alpha, beta,\\n\\t\\t  POLYBENCH_ARRAY(A),\\n\\t\\t  POLYBENCH_ARRAY(B),\\n\\t\\t  POLYBENCH_ARRAY(tmp),\\n\\t\\t  POLYBENCH_ARRAY(x),\\n\\t\\t  POLYBENCH_ARRAY(y));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(y)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(B);\\n  POLYBENCH_FREE_ARRAY(tmp);\\n  POLYBENCH_FREE_ARRAY(x);\\n  POLYBENCH_FREE_ARRAY(y);\\n\\n  return 0;\\n}\\n", "naive/gesummv.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* gesummv.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"gesummv.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int n,\\n\\t\\tDATA_TYPE *alpha,\\n\\t\\tDATA_TYPE *beta,\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,N,N,n,n),\\n\\t\\tDATA_TYPE POLYBENCH_2D(B,N,N,n,n),\\n\\t\\tDATA_TYPE POLYBENCH_1D(x,N,n))\\n{\\n  int i, j;\\n\\n  *alpha = 1.5;\\n  *beta = 1.2;\\n  for (i = 0; i < n; i++)\\n    {\\n      x[i] = (DATA_TYPE)( i % n) / n;\\n      #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) || (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)))\\n      for (j = 0; j < n; j++) {\\n\\tA[i][j] = (DATA_TYPE) ((i*j+1) % n) / n;\\n\\tB[i][j] = (DATA_TYPE) ((i*j+2) % n) / n;\\n      }\\n    }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_1D(y,N,n))\\n\\n{\\n  int i;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"y\\");\\n  for (i = 0; i < n; i++) {\\n    if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, y[i]);\\n  }\\n  POLYBENCH_DUMP_END(\\"y\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_gesummv(int n,\\n\\t\\t    DATA_TYPE alpha,\\n\\t\\t    DATA_TYPE beta,\\n\\t\\t    DATA_TYPE POLYBENCH_2D(A,N,N,n,n),\\n\\t\\t    DATA_TYPE POLYBENCH_2D(B,N,N,n,n),\\n\\t\\t    DATA_TYPE POLYBENCH_1D(tmp,N,n),\\n\\t\\t    DATA_TYPE POLYBENCH_1D(x,N,n),\\n\\t\\t    DATA_TYPE POLYBENCH_1D(y,N,n))\\n{\\n  int i, j;\\n\\n#pragma scop\\n  for (i = 0; i < _PB_N; i++)\\n    {\\n      tmp[i] = SCALAR_VAL(0.0);\\n      y[i] = SCALAR_VAL(0.0);\\n      for (j = 0; j < _PB_N; j++)\\n\\t{\\n\\t  tmp[i] = A[i][j] * x[j] + tmp[i];\\n\\t  y[i] = B[i][j] * x[j] + y[i];\\n\\t}\\n      y[i] = alpha * tmp[i] + beta * y[i];\\n    }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  DATA_TYPE alpha;\\n  DATA_TYPE beta;\\n  POLYBENCH_2D_ARRAY_DECL(A, DATA_TYPE, N, N, n, n);\\n  POLYBENCH_2D_ARRAY_DECL(B, DATA_TYPE, N, N, n, n);\\n  POLYBENCH_1D_ARRAY_DECL(tmp, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(x, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(y, DATA_TYPE, N, n);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (n, &alpha, &beta,\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(B),\\n\\t      POLYBENCH_ARRAY(x));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_gesummv (n, alpha, beta,\\n\\t\\t  POLYBENCH_ARRAY(A),\\n\\t\\t  POLYBENCH_ARRAY(B),\\n\\t\\t  POLYBENCH_ARRAY(tmp),\\n\\t\\t  POLYBENCH_ARRAY(x),\\n\\t\\t  POLYBENCH_ARRAY(y));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(y)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(B);\\n  POLYBENCH_FREE_ARRAY(tmp);\\n  POLYBENCH_FREE_ARRAY(x);\\n  POLYBENCH_FREE_ARRAY(y);\\n\\n  return 0;\\n}\\n", "gated/symm.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* symm.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"symm.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int m, int n,\\n\\t\\tDATA_TYPE *alpha,\\n\\t\\tDATA_TYPE *beta,\\n\\t\\tDATA_TYPE POLYBENCH_2D(C,M,N,m,n),\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,M,M,m,m),\\n\\t\\tDATA_TYPE POLYBENCH_2D(B,M,N,m,n))\\n{\\n  int i, j;\\n\\n  *alpha = 1.5;\\n  *beta = 1.2;\\n  for (i = 0; i < m; i++)\\n    #pragma omp parallel for if(parallel: (n >= 1048576) && ((__INTPTR_TYPE__)C + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0) || (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)C + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)))\\n    for (j = 0; j < n; j++) {\\n      C[i][j] = (DATA_TYPE) ((i+j) % 100) / m;\\n      B[i][j] = (DATA_TYPE) ((n+i-j) % 100) / m;\\n    }\\n  for (i = 0; i < m; i++) {\\n    #pragma omp target data map(tofrom: A[i:1][0:1000]) if(i >= 0 && i >= 268435456 && i >= 0 && (-(__INTPTR_TYPE__)i + (__INTPTR_TYPE__)m) >= 2 && m >= 268435456)\\n    {\\n    #pragma omp target teams distribute parallel for if(target: i >= 0 && i >= 268435456 && i >= 0 && (-(__INTPTR_TYPE__)i + (__INTPTR_TYPE__)m) >= 2 && m >= 268435456) map(tofrom: A[i:1][0:1000])\\n    for (j = 0; j <=i; j++)\\n      A[i][j] = (DATA_TYPE) ((i+j) % 100) / m;\\n    #pragma omp target teams distribute parallel for if(target: i >= 0 && i >= 268435456 && i >= 0 && (-(__INTPTR_TYPE__)i + (__INTPTR_TYPE__)m) >= 2 && m >= 268435456) map(tofrom: A[i:1][0:1000])\\n    for (j = i+1; j < m; j++)\\n      A[i][j] = -999;\\n    }\\n     //regions of arrays that should not be used\\n  }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int m, int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(C,M,N,m,n))\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"C\\");\\n  for (i = 0; i < m; i++)\\n    for (j = 0; j < n; j++) {\\n\\tif ((i * m + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n\\tfprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, C[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"C\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_symm(int m, int n,\\n\\t\\t DATA_TYPE alpha,\\n\\t\\t DATA_TYPE beta,\\n\\t\\t DATA_TYPE POLYBENCH_2D(C,M,N,m,n),\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,M,M,m,m),\\n\\t\\t DATA_TYPE POLYBENCH_2D(B,M,N,m,n))\\n{\\n  int i, j, k;\\n  DATA_TYPE temp2;\\n\\n//BLAS PARAMS\\n//SIDE = \'L\'\\n//UPLO = \'L\'\\n// =>  Form  C := alpha*A*B + beta*C\\n// A is MxM\\n// B is MxN\\n// C is MxN\\n//note that due to Fortran array layout, the code below more closely resembles upper triangular case in BLAS\\n#pragma scop\\n   for (i = 0; i < _PB_M; i++)\\n      for (j = 0; j < _PB_N; j++ )\\n      {\\n        temp2 = 0;\\n        for (k = 0; k < i; k++) {\\n           C[k][j] += alpha*B[i][j] * A[i][k];\\n           temp2 += B[k][j] * A[i][k];\\n        }\\n        C[i][j] = beta * C[i][j] + alpha*B[i][j] * A[i][i] + alpha * temp2;\\n     }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int m = M;\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  DATA_TYPE alpha;\\n  DATA_TYPE beta;\\n  POLYBENCH_2D_ARRAY_DECL(C,DATA_TYPE,M,N,m,n);\\n  POLYBENCH_2D_ARRAY_DECL(A,DATA_TYPE,M,M,m,m);\\n  POLYBENCH_2D_ARRAY_DECL(B,DATA_TYPE,M,N,m,n);\\n\\n  /* Initialize array(s). */\\n  init_array (m, n, &alpha, &beta,\\n\\t      POLYBENCH_ARRAY(C),\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(B));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_symm (m, n,\\n\\t       alpha, beta,\\n\\t       POLYBENCH_ARRAY(C),\\n\\t       POLYBENCH_ARRAY(A),\\n\\t       POLYBENCH_ARRAY(B));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(m, n, POLYBENCH_ARRAY(C)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(C);\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(B);\\n\\n  return 0;\\n}\\n", "naive/symm.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* symm.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"symm.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int m, int n,\\n\\t\\tDATA_TYPE *alpha,\\n\\t\\tDATA_TYPE *beta,\\n\\t\\tDATA_TYPE POLYBENCH_2D(C,M,N,m,n),\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,M,M,m,m),\\n\\t\\tDATA_TYPE POLYBENCH_2D(B,M,N,m,n))\\n{\\n  int i, j;\\n\\n  *alpha = 1.5;\\n  *beta = 1.2;\\n  for (i = 0; i < m; i++)\\n    #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)C + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0) || (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)C + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)))\\n    for (j = 0; j < n; j++) {\\n      C[i][j] = (DATA_TYPE) ((i+j) % 100) / m;\\n      B[i][j] = (DATA_TYPE) ((n+i-j) % 100) / m;\\n    }\\n  for (i = 0; i < m; i++) {\\n    #pragma omp target data map(tofrom: A[i:1][0:1000]) if(i >= 0 && i >= 0 && (-(__INTPTR_TYPE__)i + (__INTPTR_TYPE__)m) >= 2)\\n    {\\n    #pragma omp target teams distribute parallel for if(target: i >= 0 && i >= 0 && (-(__INTPTR_TYPE__)i + (__INTPTR_TYPE__)m) >= 2) map(tofrom: A[i:1][0:1000])\\n    for (j = 0; j <=i; j++)\\n      A[i][j] = (DATA_TYPE) ((i+j) % 100) / m;\\n    #pragma omp target teams distribute parallel for if(target: i >= 0 && i >= 0 && (-(__INTPTR_TYPE__)i + (__INTPTR_TYPE__)m) >= 2) map(tofrom: A[i:1][0:1000])\\n    for (j = i+1; j < m; j++)\\n      A[i][j] = -999;\\n    }\\n     //regions of arrays that should not be used\\n  }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int m, int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(C,M,N,m,n))\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"C\\");\\n  for (i = 0; i < m; i++)\\n    for (j = 0; j < n; j++) {\\n\\tif ((i * m + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n\\tfprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, C[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"C\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_symm(int m, int n,\\n\\t\\t DATA_TYPE alpha,\\n\\t\\t DATA_TYPE beta,\\n\\t\\t DATA_TYPE POLYBENCH_2D(C,M,N,m,n),\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,M,M,m,m),\\n\\t\\t DATA_TYPE POLYBENCH_2D(B,M,N,m,n))\\n{\\n  int i, j, k;\\n  DATA_TYPE temp2;\\n\\n//BLAS PARAMS\\n//SIDE = \'L\'\\n//UPLO = \'L\'\\n// =>  Form  C := alpha*A*B + beta*C\\n// A is MxM\\n// B is MxN\\n// C is MxN\\n//note that due to Fortran array layout, the code below more closely resembles upper triangular case in BLAS\\n#pragma scop\\n   for (i = 0; i < _PB_M; i++)\\n      for (j = 0; j < _PB_N; j++ )\\n      {\\n        temp2 = 0;\\n        for (k = 0; k < i; k++) {\\n           C[k][j] += alpha*B[i][j] * A[i][k];\\n           temp2 += B[k][j] * A[i][k];\\n        }\\n        C[i][j] = beta * C[i][j] + alpha*B[i][j] * A[i][i] + alpha * temp2;\\n     }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int m = M;\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  DATA_TYPE alpha;\\n  DATA_TYPE beta;\\n  POLYBENCH_2D_ARRAY_DECL(C,DATA_TYPE,M,N,m,n);\\n  POLYBENCH_2D_ARRAY_DECL(A,DATA_TYPE,M,M,m,m);\\n  POLYBENCH_2D_ARRAY_DECL(B,DATA_TYPE,M,N,m,n);\\n\\n  /* Initialize array(s). */\\n  init_array (m, n, &alpha, &beta,\\n\\t      POLYBENCH_ARRAY(C),\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(B));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_symm (m, n,\\n\\t       alpha, beta,\\n\\t       POLYBENCH_ARRAY(C),\\n\\t       POLYBENCH_ARRAY(A),\\n\\t       POLYBENCH_ARRAY(B));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(m, n, POLYBENCH_ARRAY(C)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(C);\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(B);\\n\\n  return 0;\\n}\\n", "gated/syr2k.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* syr2k.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"syr2k.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int n, int m,\\n\\t\\tDATA_TYPE *alpha,\\n\\t\\tDATA_TYPE *beta,\\n\\t\\tDATA_TYPE POLYBENCH_2D(C,N,N,n,n),\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,N,M,n,m),\\n\\t\\tDATA_TYPE POLYBENCH_2D(B,N,M,n,m))\\n{\\n  int i, j;\\n\\n  *alpha = 1.5;\\n  *beta = 1.2;\\n  for (i = 0; i < n; i++)\\n    #pragma omp parallel for if(parallel: (m >= 524288) && ((__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1000 + ((__INTPTR_TYPE__)m - 1) + 1) <= (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)i) * 1000 + 0) || (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)i) * 1000 + ((__INTPTR_TYPE__)m - 1) + 1) <= (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1000 + 0)))\\n    for (j = 0; j < m; j++) {\\n      A[i][j] = (DATA_TYPE) ((i*j+1)%n) / n;\\n      B[i][j] = (DATA_TYPE) ((i*j+2)%m) / m;\\n    }\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ((double)(n) * ((5 * (double)(n)) - 29523) >= 2147483648) && n >= 1) map(tofrom: C[0:n][0:1200]) private(j)\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++) {\\n      C[i][j] = (DATA_TYPE) ((i*j+3)%n) / m;\\n    }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(C,N,N,n,n))\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"C\\");\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++) {\\n\\tif ((i * n + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n\\tfprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, C[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"C\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_syr2k(int n, int m,\\n\\t\\t  DATA_TYPE alpha,\\n\\t\\t  DATA_TYPE beta,\\n\\t\\t  DATA_TYPE POLYBENCH_2D(C,N,N,n,n),\\n\\t\\t  DATA_TYPE POLYBENCH_2D(A,N,M,n,m),\\n\\t\\t  DATA_TYPE POLYBENCH_2D(B,N,M,n,m))\\n{\\n  int i, j, k;\\n\\n//BLAS PARAMS\\n//UPLO  = \'L\'\\n//TRANS = \'N\'\\n//A is NxM\\n//B is NxM\\n//C is NxN\\n#pragma scop\\n  for (i = 0; i < _PB_N; i++) {\\n    #pragma omp target teams distribute parallel for if(target: (i >= 134217728) && i >= 0) map(tofrom: C[i:1][0:1200])\\n    for (j = 0; j <= i; j++)\\n      C[i][j] *= beta;\\n    for (k = 0; k < _PB_M; k++)\\n      #pragma omp parallel for if(parallel: (i >= 512) && ((__INTPTR_TYPE__)C + 8 * (((__INTPTR_TYPE__)i) * 1200 + (__INTPTR_TYPE__)i + 1) <= (__INTPTR_TYPE__)A + (8 * ((0) * 1000 + (__INTPTR_TYPE__)k) < 8 * (((__INTPTR_TYPE__)i) * 1000 + (__INTPTR_TYPE__)k) ? 8 * ((0) * 1000 + (__INTPTR_TYPE__)k) : 8 * (((__INTPTR_TYPE__)i) * 1000 + (__INTPTR_TYPE__)k)) || (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1000 + (__INTPTR_TYPE__)k + 1) <= (__INTPTR_TYPE__)C + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)) && ((__INTPTR_TYPE__)C + 8 * (((__INTPTR_TYPE__)i) * 1200 + (__INTPTR_TYPE__)i + 1) <= (__INTPTR_TYPE__)B + (8 * (((__INTPTR_TYPE__)i) * 1000 + (__INTPTR_TYPE__)k) < 8 * ((0) * 1000 + (__INTPTR_TYPE__)k) ? 8 * (((__INTPTR_TYPE__)i) * 1000 + (__INTPTR_TYPE__)k) : 8 * ((0) * 1000 + (__INTPTR_TYPE__)k)) || (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)i) * 1000 + (__INTPTR_TYPE__)k + 1) <= (__INTPTR_TYPE__)C + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)))\\n      for (j = 0; j <= i; j++)\\n\\t{\\n\\t  C[i][j] += A[j][k]*alpha*B[i][k] + B[j][k]*alpha*A[i][k];\\n\\t}\\n  }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n  int m = M;\\n\\n  /* Variable declaration/allocation. */\\n  DATA_TYPE alpha;\\n  DATA_TYPE beta;\\n  POLYBENCH_2D_ARRAY_DECL(C,DATA_TYPE,N,N,n,n);\\n  POLYBENCH_2D_ARRAY_DECL(A,DATA_TYPE,N,M,n,m);\\n  POLYBENCH_2D_ARRAY_DECL(B,DATA_TYPE,N,M,n,m);\\n\\n  /* Initialize array(s). */\\n  init_array (n, m, &alpha, &beta,\\n\\t      POLYBENCH_ARRAY(C),\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(B));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_syr2k (n, m,\\n\\t\\talpha, beta,\\n\\t\\tPOLYBENCH_ARRAY(C),\\n\\t\\tPOLYBENCH_ARRAY(A),\\n\\t\\tPOLYBENCH_ARRAY(B));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(C)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(C);\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(B);\\n\\n  return 0;\\n}\\n", "naive/syr2k.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* syr2k.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"syr2k.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int n, int m,\\n\\t\\tDATA_TYPE *alpha,\\n\\t\\tDATA_TYPE *beta,\\n\\t\\tDATA_TYPE POLYBENCH_2D(C,N,N,n,n),\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,N,M,n,m),\\n\\t\\tDATA_TYPE POLYBENCH_2D(B,N,M,n,m))\\n{\\n  int i, j;\\n\\n  *alpha = 1.5;\\n  *beta = 1.2;\\n  for (i = 0; i < n; i++)\\n    #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1000 + ((__INTPTR_TYPE__)m - 1) + 1) <= (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)i) * 1000 + 0) || (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)i) * 1000 + ((__INTPTR_TYPE__)m - 1) + 1) <= (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1000 + 0)))\\n    for (j = 0; j < m; j++) {\\n      A[i][j] = (DATA_TYPE) ((i*j+1)%n) / n;\\n      B[i][j] = (DATA_TYPE) ((i*j+2)%m) / m;\\n    }\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: n >= 1) map(tofrom: C[0:n][0:1200]) private(j)\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++) {\\n      C[i][j] = (DATA_TYPE) ((i*j+3)%n) / m;\\n    }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(C,N,N,n,n))\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"C\\");\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++) {\\n\\tif ((i * n + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n\\tfprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, C[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"C\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_syr2k(int n, int m,\\n\\t\\t  DATA_TYPE alpha,\\n\\t\\t  DATA_TYPE beta,\\n\\t\\t  DATA_TYPE POLYBENCH_2D(C,N,N,n,n),\\n\\t\\t  DATA_TYPE POLYBENCH_2D(A,N,M,n,m),\\n\\t\\t  DATA_TYPE POLYBENCH_2D(B,N,M,n,m))\\n{\\n  int i, j, k;\\n\\n//BLAS PARAMS\\n//UPLO  = \'L\'\\n//TRANS = \'N\'\\n//A is NxM\\n//B is NxM\\n//C is NxN\\n#pragma scop\\n  for (i = 0; i < _PB_N; i++) {\\n    #pragma omp target teams distribute parallel for if(target: i >= 0) map(tofrom: C[i:1][0:1200])\\n    for (j = 0; j <= i; j++)\\n      C[i][j] *= beta;\\n    for (k = 0; k < _PB_M; k++)\\n      #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)C + 8 * (((__INTPTR_TYPE__)i) * 1200 + (__INTPTR_TYPE__)i + 1) <= (__INTPTR_TYPE__)A + (8 * ((0) * 1000 + (__INTPTR_TYPE__)k) < 8 * (((__INTPTR_TYPE__)i) * 1000 + (__INTPTR_TYPE__)k) ? 8 * ((0) * 1000 + (__INTPTR_TYPE__)k) : 8 * (((__INTPTR_TYPE__)i) * 1000 + (__INTPTR_TYPE__)k)) || (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1000 + (__INTPTR_TYPE__)k + 1) <= (__INTPTR_TYPE__)C + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)) && ((__INTPTR_TYPE__)C + 8 * (((__INTPTR_TYPE__)i) * 1200 + (__INTPTR_TYPE__)i + 1) <= (__INTPTR_TYPE__)B + (8 * (((__INTPTR_TYPE__)i) * 1000 + (__INTPTR_TYPE__)k) < 8 * ((0) * 1000 + (__INTPTR_TYPE__)k) ? 8 * (((__INTPTR_TYPE__)i) * 1000 + (__INTPTR_TYPE__)k) : 8 * ((0) * 1000 + (__INTPTR_TYPE__)k)) || (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)i) * 1000 + (__INTPTR_TYPE__)k + 1) <= (__INTPTR_TYPE__)C + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)))\\n      for (j = 0; j <= i; j++)\\n\\t{\\n\\t  C[i][j] += A[j][k]*alpha*B[i][k] + B[j][k]*alpha*A[i][k];\\n\\t}\\n  }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n  int m = M;\\n\\n  /* Variable declaration/allocation. */\\n  DATA_TYPE alpha;\\n  DATA_TYPE beta;\\n  POLYBENCH_2D_ARRAY_DECL(C,DATA_TYPE,N,N,n,n);\\n  POLYBENCH_2D_ARRAY_DECL(A,DATA_TYPE,N,M,n,m);\\n  POLYBENCH_2D_ARRAY_DECL(B,DATA_TYPE,N,M,n,m);\\n\\n  /* Initialize array(s). */\\n  init_array (n, m, &alpha, &beta,\\n\\t      POLYBENCH_ARRAY(C),\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(B));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_syr2k (n, m,\\n\\t\\talpha, beta,\\n\\t\\tPOLYBENCH_ARRAY(C),\\n\\t\\tPOLYBENCH_ARRAY(A),\\n\\t\\tPOLYBENCH_ARRAY(B));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(C)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(C);\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(B);\\n\\n  return 0;\\n}\\n", "gated/syrk.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* syrk.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"syrk.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int n, int m,\\n\\t\\tDATA_TYPE *alpha,\\n\\t\\tDATA_TYPE *beta,\\n\\t\\tDATA_TYPE POLYBENCH_2D(C,N,N,n,n),\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,N,M,n,m))\\n{\\n  int i, j;\\n\\n  *alpha = 1.5;\\n  *beta = 1.2;\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ((double)(n) * ((5 * (double)(m)) - 24603) >= 1073741824) && n >= 1 && m >= 1) map(tofrom: A[0:n][0:1000]) private(j)\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < m; j++)\\n      A[i][j] = (DATA_TYPE) ((i*j+1)%n) / n;\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ((double)(n) * ((5 * (double)(n)) - 29523) >= 1073741824) && n >= 1) map(tofrom: C[0:n][0:1200]) private(j)\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++)\\n      C[i][j] = (DATA_TYPE) ((i*j+2)%m) / m;\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(C,N,N,n,n))\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"C\\");\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++) {\\n\\tif ((i * n + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n\\tfprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, C[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"C\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_syrk(int n, int m,\\n\\t\\t DATA_TYPE alpha,\\n\\t\\t DATA_TYPE beta,\\n\\t\\t DATA_TYPE POLYBENCH_2D(C,N,N,n,n),\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,N,M,n,m))\\n{\\n  int i, j, k;\\n\\n//BLAS PARAMS\\n//TRANS = \'N\'\\n//UPLO  = \'L\'\\n// =>  Form  C := alpha*A*A**T + beta*C.\\n//A is NxM\\n//C is NxN\\n#pragma scop\\n  for (i = 0; i < _PB_N; i++) {\\n    #pragma omp target teams distribute parallel for if(target: (i >= 67108864) && i >= 0) map(tofrom: C[i:1][0:1200])\\n    for (j = 0; j <= i; j++)\\n      C[i][j] *= beta;\\n    for (k = 0; k < _PB_M; k++) {\\n      #pragma omp parallel for if(parallel: (i >= 2048) && ((__INTPTR_TYPE__)C + 8 * (((__INTPTR_TYPE__)i) * 1200 + (__INTPTR_TYPE__)i + 1) <= (__INTPTR_TYPE__)A + (8 * (((__INTPTR_TYPE__)i) * 1000 + (__INTPTR_TYPE__)k) < 8 * ((0) * 1000 + (__INTPTR_TYPE__)k) ? 8 * (((__INTPTR_TYPE__)i) * 1000 + (__INTPTR_TYPE__)k) : 8 * ((0) * 1000 + (__INTPTR_TYPE__)k)) || (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1000 + (__INTPTR_TYPE__)k + 1) <= (__INTPTR_TYPE__)C + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)))\\n      for (j = 0; j <= i; j++)\\n        C[i][j] += alpha * A[i][k] * A[j][k];\\n    }\\n  }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n  int m = M;\\n\\n  /* Variable declaration/allocation. */\\n  DATA_TYPE alpha;\\n  DATA_TYPE beta;\\n  POLYBENCH_2D_ARRAY_DECL(C,DATA_TYPE,N,N,n,n);\\n  POLYBENCH_2D_ARRAY_DECL(A,DATA_TYPE,N,M,n,m);\\n\\n  /* Initialize array(s). */\\n  init_array (n, m, &alpha, &beta, POLYBENCH_ARRAY(C), POLYBENCH_ARRAY(A));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_syrk (n, m, alpha, beta, POLYBENCH_ARRAY(C), POLYBENCH_ARRAY(A));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(C)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(C);\\n  POLYBENCH_FREE_ARRAY(A);\\n\\n  return 0;\\n}\\n", "naive/syrk.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* syrk.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"syrk.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int n, int m,\\n\\t\\tDATA_TYPE *alpha,\\n\\t\\tDATA_TYPE *beta,\\n\\t\\tDATA_TYPE POLYBENCH_2D(C,N,N,n,n),\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,N,M,n,m))\\n{\\n  int i, j;\\n\\n  *alpha = 1.5;\\n  *beta = 1.2;\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: n >= 1 && m >= 1) map(tofrom: A[0:n][0:1000]) private(j)\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < m; j++)\\n      A[i][j] = (DATA_TYPE) ((i*j+1)%n) / n;\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: n >= 1) map(tofrom: C[0:n][0:1200]) private(j)\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++)\\n      C[i][j] = (DATA_TYPE) ((i*j+2)%m) / m;\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(C,N,N,n,n))\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"C\\");\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++) {\\n\\tif ((i * n + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n\\tfprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, C[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"C\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_syrk(int n, int m,\\n\\t\\t DATA_TYPE alpha,\\n\\t\\t DATA_TYPE beta,\\n\\t\\t DATA_TYPE POLYBENCH_2D(C,N,N,n,n),\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,N,M,n,m))\\n{\\n  int i, j, k;\\n\\n//BLAS PARAMS\\n//TRANS = \'N\'\\n//UPLO  = \'L\'\\n// =>  Form  C := alpha*A*A**T + beta*C.\\n//A is NxM\\n//C is NxN\\n#pragma scop\\n  for (i = 0; i < _PB_N; i++) {\\n    #pragma omp target teams distribute parallel for if(target: i >= 0) map(tofrom: C[i:1][0:1200])\\n    for (j = 0; j <= i; j++)\\n      C[i][j] *= beta;\\n    for (k = 0; k < _PB_M; k++) {\\n      #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)C + 8 * (((__INTPTR_TYPE__)i) * 1200 + (__INTPTR_TYPE__)i + 1) <= (__INTPTR_TYPE__)A + (8 * (((__INTPTR_TYPE__)i) * 1000 + (__INTPTR_TYPE__)k) < 8 * ((0) * 1000 + (__INTPTR_TYPE__)k) ? 8 * (((__INTPTR_TYPE__)i) * 1000 + (__INTPTR_TYPE__)k) : 8 * ((0) * 1000 + (__INTPTR_TYPE__)k)) || (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1000 + (__INTPTR_TYPE__)k + 1) <= (__INTPTR_TYPE__)C + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)))\\n      for (j = 0; j <= i; j++)\\n        C[i][j] += alpha * A[i][k] * A[j][k];\\n    }\\n  }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n  int m = M;\\n\\n  /* Variable declaration/allocation. */\\n  DATA_TYPE alpha;\\n  DATA_TYPE beta;\\n  POLYBENCH_2D_ARRAY_DECL(C,DATA_TYPE,N,N,n,n);\\n  POLYBENCH_2D_ARRAY_DECL(A,DATA_TYPE,N,M,n,m);\\n\\n  /* Initialize array(s). */\\n  init_array (n, m, &alpha, &beta, POLYBENCH_ARRAY(C), POLYBENCH_ARRAY(A));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_syrk (n, m, alpha, beta, POLYBENCH_ARRAY(C), POLYBENCH_ARRAY(A));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(C)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(C);\\n  POLYBENCH_FREE_ARRAY(A);\\n\\n  return 0;\\n}\\n", "gated/trmm.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* trmm.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"trmm.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int m, int n,\\n\\t\\tDATA_TYPE *alpha,\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,M,M,m,m),\\n\\t\\tDATA_TYPE POLYBENCH_2D(B,M,N,m,n))\\n{\\n  int i, j;\\n\\n  *alpha = 1.5;\\n  #pragma omp parallel for if(parallel: (m >= 2048) && ((__INTPTR_TYPE__)A + (8 * ((((__INTPTR_TYPE__)m - 1)) * 1000 + 999 + 1) > 8 * ((((__INTPTR_TYPE__)m - 1)) * 1000 + ((__INTPTR_TYPE__)m - 1) + 1) ? 8 * ((((__INTPTR_TYPE__)m - 1)) * 1000 + 999 + 1) : 8 * ((((__INTPTR_TYPE__)m - 1)) * 1000 + ((__INTPTR_TYPE__)m - 1) + 1)) <= (__INTPTR_TYPE__)B + 8 * ((0) * 1200 + 0) || (__INTPTR_TYPE__)B + 8 * ((((__INTPTR_TYPE__)m - 1)) * 1200 + 1199 + 1) <= (__INTPTR_TYPE__)A + 8 * ((0) * 1000 + 0))) private(j)\\n  for (i = 0; i < m; i++) {\\n    for (j = 0; j < i; j++) {\\n      A[i][j] = (DATA_TYPE)((i+j) % m)/m;\\n    }\\n    A[i][i] = 1.0;\\n    for (j = 0; j < n; j++) {\\n      B[i][j] = (DATA_TYPE)((n+(i-j)) % n)/n;\\n    }\\n }\\n\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int m, int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(B,M,N,m,n))\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"B\\");\\n  for (i = 0; i < m; i++)\\n    for (j = 0; j < n; j++) {\\n\\tif ((i * m + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n\\tfprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, B[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"B\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_trmm(int m, int n,\\n\\t\\t DATA_TYPE alpha,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,M,M,m,m),\\n\\t\\t DATA_TYPE POLYBENCH_2D(B,M,N,m,n))\\n{\\n  int i, j, k;\\n\\n//BLAS parameters\\n//SIDE   = \'L\'\\n//UPLO   = \'L\'\\n//TRANSA = \'T\'\\n//DIAG   = \'U\'\\n// => Form  B := alpha*A**T*B.\\n// A is MxM\\n// B is MxN\\n#pragma scop\\n  for (i = 0; i < _PB_M; i++)\\n     for (j = 0; j < _PB_N; j++) {\\n        for (k = i+1; k < _PB_M; k++)\\n           B[i][j] += A[k][i] * B[k][j];\\n        B[i][j] = alpha * B[i][j];\\n     }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int m = M;\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  DATA_TYPE alpha;\\n  POLYBENCH_2D_ARRAY_DECL(A,DATA_TYPE,M,M,m,m);\\n  POLYBENCH_2D_ARRAY_DECL(B,DATA_TYPE,M,N,m,n);\\n\\n  /* Initialize array(s). */\\n  init_array (m, n, &alpha, POLYBENCH_ARRAY(A), POLYBENCH_ARRAY(B));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_trmm (m, n, alpha, POLYBENCH_ARRAY(A), POLYBENCH_ARRAY(B));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(m, n, POLYBENCH_ARRAY(B)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(B);\\n\\n  return 0;\\n}\\n", "naive/trmm.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* trmm.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"trmm.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int m, int n,\\n\\t\\tDATA_TYPE *alpha,\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,M,M,m,m),\\n\\t\\tDATA_TYPE POLYBENCH_2D(B,M,N,m,n))\\n{\\n  int i, j;\\n\\n  *alpha = 1.5;\\n  #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)A + (8 * ((((__INTPTR_TYPE__)m - 1)) * 1000 + 999 + 1) > 8 * ((((__INTPTR_TYPE__)m - 1)) * 1000 + ((__INTPTR_TYPE__)m - 1) + 1) ? 8 * ((((__INTPTR_TYPE__)m - 1)) * 1000 + 999 + 1) : 8 * ((((__INTPTR_TYPE__)m - 1)) * 1000 + ((__INTPTR_TYPE__)m - 1) + 1)) <= (__INTPTR_TYPE__)B + 8 * ((0) * 1200 + 0) || (__INTPTR_TYPE__)B + 8 * ((((__INTPTR_TYPE__)m - 1)) * 1200 + 1199 + 1) <= (__INTPTR_TYPE__)A + 8 * ((0) * 1000 + 0))) private(j)\\n  for (i = 0; i < m; i++) {\\n    for (j = 0; j < i; j++) {\\n      A[i][j] = (DATA_TYPE)((i+j) % m)/m;\\n    }\\n    A[i][i] = 1.0;\\n    for (j = 0; j < n; j++) {\\n      B[i][j] = (DATA_TYPE)((n+(i-j)) % n)/n;\\n    }\\n }\\n\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int m, int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(B,M,N,m,n))\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"B\\");\\n  for (i = 0; i < m; i++)\\n    for (j = 0; j < n; j++) {\\n\\tif ((i * m + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n\\tfprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, B[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"B\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_trmm(int m, int n,\\n\\t\\t DATA_TYPE alpha,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,M,M,m,m),\\n\\t\\t DATA_TYPE POLYBENCH_2D(B,M,N,m,n))\\n{\\n  int i, j, k;\\n\\n//BLAS parameters\\n//SIDE   = \'L\'\\n//UPLO   = \'L\'\\n//TRANSA = \'T\'\\n//DIAG   = \'U\'\\n// => Form  B := alpha*A**T*B.\\n// A is MxM\\n// B is MxN\\n#pragma scop\\n  for (i = 0; i < _PB_M; i++)\\n     for (j = 0; j < _PB_N; j++) {\\n        for (k = i+1; k < _PB_M; k++)\\n           B[i][j] += A[k][i] * B[k][j];\\n        B[i][j] = alpha * B[i][j];\\n     }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int m = M;\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  DATA_TYPE alpha;\\n  POLYBENCH_2D_ARRAY_DECL(A,DATA_TYPE,M,M,m,m);\\n  POLYBENCH_2D_ARRAY_DECL(B,DATA_TYPE,M,N,m,n);\\n\\n  /* Initialize array(s). */\\n  init_array (m, n, &alpha, POLYBENCH_ARRAY(A), POLYBENCH_ARRAY(B));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_trmm (m, n, alpha, POLYBENCH_ARRAY(A), POLYBENCH_ARRAY(B));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(m, n, POLYBENCH_ARRAY(B)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(B);\\n\\n  return 0;\\n}\\n", "gated/cholesky.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* cholesky.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"cholesky.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int n,\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,N,N,n,n))\\n{\\n  int i, j;\\n\\n  #pragma omp target teams distribute parallel for if(target: ((double)(n) * ((1 + 6 * (double)(n)) - 49205) >= 4294967296) && n >= 1) map(tofrom: A[0:n][0:2000]) private(j)\\n  for (i = 0; i < n; i++)\\n    {\\n      for (j = 0; j <= i; j++)\\n\\tA[i][j] = (DATA_TYPE)(-j % n) / n + 1;\\n      for (j = i+1; j < n; j++) {\\n\\tA[i][j] = 0;\\n      }\\n      A[i][i] = 1;\\n    }\\n\\n  /* Make the matrix positive semi-definite. */\\n  int r,s,t;\\n  POLYBENCH_2D_ARRAY_DECL(B, DATA_TYPE, N, N, n, n);\\n  for (r = 0; r < n; ++r)\\n    for (s = 0; s < n; ++s)\\n      (POLYBENCH_ARRAY(B))[r][s] = 0;\\n  for (t = 0; t < n; ++t)\\n    for (r = 0; r < n; ++r)\\n      for (s = 0; s < n; ++s)\\n\\t(POLYBENCH_ARRAY(B))[r][s] += A[r][t] * A[s][t];\\n    for (r = 0; r < n; ++r)\\n      for (s = 0; s < n; ++s)\\n\\tA[r][s] = (POLYBENCH_ARRAY(B))[r][s];\\n  POLYBENCH_FREE_ARRAY(B);\\n\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,N,N,n,n))\\n\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"A\\");\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j <= i; j++) {\\n    if ((i * n + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j]);\\n  }\\n  POLYBENCH_DUMP_END(\\"A\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_cholesky(int n,\\n\\t\\t     DATA_TYPE POLYBENCH_2D(A,N,N,n,n))\\n{\\n  int i, j, k;\\n\\n\\n#pragma scop\\n  for (i = 0; i < _PB_N; i++) {\\n     //j<i\\n     for (j = 0; j < i; j++) {\\n        for (k = 0; k < j; k++) {\\n           A[i][j] -= A[i][k] * A[j][k];\\n        }\\n        A[i][j] /= A[j][j];\\n     }\\n     // i==j case\\n     for (k = 0; k < i; k++) {\\n        A[i][i] -= A[i][k] * A[i][k];\\n     }\\n     A[i][i] = SQRT_FUN(A[i][i]);\\n  }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(A, DATA_TYPE, N, N, n, n);\\n\\n  /* Initialize array(s). */\\n  init_array (n, POLYBENCH_ARRAY(A));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_cholesky (n, POLYBENCH_ARRAY(A));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(A)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n\\n  return 0;\\n}\\n", "naive/cholesky.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* cholesky.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"cholesky.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int n,\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,N,N,n,n))\\n{\\n  int i, j;\\n\\n  #pragma omp target teams distribute parallel for if(target: n >= 1) map(tofrom: A[0:n][0:2000]) private(j)\\n  for (i = 0; i < n; i++)\\n    {\\n      for (j = 0; j <= i; j++)\\n\\tA[i][j] = (DATA_TYPE)(-j % n) / n + 1;\\n      for (j = i+1; j < n; j++) {\\n\\tA[i][j] = 0;\\n      }\\n      A[i][i] = 1;\\n    }\\n\\n  /* Make the matrix positive semi-definite. */\\n  int r,s,t;\\n  POLYBENCH_2D_ARRAY_DECL(B, DATA_TYPE, N, N, n, n);\\n  for (r = 0; r < n; ++r)\\n    for (s = 0; s < n; ++s)\\n      (POLYBENCH_ARRAY(B))[r][s] = 0;\\n  for (t = 0; t < n; ++t)\\n    for (r = 0; r < n; ++r)\\n      for (s = 0; s < n; ++s)\\n\\t(POLYBENCH_ARRAY(B))[r][s] += A[r][t] * A[s][t];\\n    for (r = 0; r < n; ++r)\\n      for (s = 0; s < n; ++s)\\n\\tA[r][s] = (POLYBENCH_ARRAY(B))[r][s];\\n  POLYBENCH_FREE_ARRAY(B);\\n\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,N,N,n,n))\\n\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"A\\");\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j <= i; j++) {\\n    if ((i * n + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j]);\\n  }\\n  POLYBENCH_DUMP_END(\\"A\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_cholesky(int n,\\n\\t\\t     DATA_TYPE POLYBENCH_2D(A,N,N,n,n))\\n{\\n  int i, j, k;\\n\\n\\n#pragma scop\\n  for (i = 0; i < _PB_N; i++) {\\n     //j<i\\n     for (j = 0; j < i; j++) {\\n        for (k = 0; k < j; k++) {\\n           A[i][j] -= A[i][k] * A[j][k];\\n        }\\n        A[i][j] /= A[j][j];\\n     }\\n     // i==j case\\n     for (k = 0; k < i; k++) {\\n        A[i][i] -= A[i][k] * A[i][k];\\n     }\\n     A[i][i] = SQRT_FUN(A[i][i]);\\n  }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(A, DATA_TYPE, N, N, n, n);\\n\\n  /* Initialize array(s). */\\n  init_array (n, POLYBENCH_ARRAY(A));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_cholesky (n, POLYBENCH_ARRAY(A));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(A)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n\\n  return 0;\\n}\\n", "gated/durbin.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* durbin.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"durbin.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int n,\\n\\t\\t DATA_TYPE POLYBENCH_1D(r,N,n))\\n{\\n  int i, j;\\n\\n  #pragma omp parallel for if(parallel: n >= 1048576)\\n  for (i = 0; i < n; i++)\\n    {\\n      r[i] = (n+1-i);\\n    }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_1D(y,N,n))\\n\\n{\\n  int i;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"y\\");\\n  for (i = 0; i < n; i++) {\\n    if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, y[i]);\\n  }\\n  POLYBENCH_DUMP_END(\\"y\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_durbin(int n,\\n\\t\\t   DATA_TYPE POLYBENCH_1D(r,N,n),\\n\\t\\t   DATA_TYPE POLYBENCH_1D(y,N,n))\\n{\\n DATA_TYPE z[N];\\n DATA_TYPE alpha;\\n DATA_TYPE beta;\\n DATA_TYPE sum;\\n\\n int i,k;\\n\\n#pragma scop\\n y[0] = -r[0];\\n beta = SCALAR_VAL(1.0);\\n alpha = -r[0];\\n\\n for (k = 1; k < _PB_N; k++) {\\n   beta = (1-alpha*alpha)*beta;\\n   sum = SCALAR_VAL(0.0);\\n   for (i=0; i<k; i++) {\\n      sum += r[k-i-1]*y[i];\\n   }\\n   alpha = - (r[k] + sum)/beta;\\n\\n   #pragma omp parallel for if(parallel: k >= 524288)\\n   for (i=0; i<k; i++) {\\n      z[i] = y[i] + alpha*y[k-i-1];\\n   }\\n   #pragma omp parallel for if(parallel: k >= 524288)\\n   for (i=0; i<k; i++) {\\n     y[i] = z[i];\\n   }\\n   y[k] = alpha;\\n }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_1D_ARRAY_DECL(r, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(y, DATA_TYPE, N, n);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (n, POLYBENCH_ARRAY(r));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_durbin (n,\\n\\t\\t POLYBENCH_ARRAY(r),\\n\\t\\t POLYBENCH_ARRAY(y));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(y)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(r);\\n  POLYBENCH_FREE_ARRAY(y);\\n\\n  return 0;\\n}\\n", "naive/durbin.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* durbin.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"durbin.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int n,\\n\\t\\t DATA_TYPE POLYBENCH_1D(r,N,n))\\n{\\n  int i, j;\\n\\n  #pragma omp target teams distribute parallel for if(target: n >= 1) map(from: r[0:n])\\n  for (i = 0; i < n; i++)\\n    {\\n      r[i] = (n+1-i);\\n    }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_1D(y,N,n))\\n\\n{\\n  int i;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"y\\");\\n  for (i = 0; i < n; i++) {\\n    if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, y[i]);\\n  }\\n  POLYBENCH_DUMP_END(\\"y\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_durbin(int n,\\n\\t\\t   DATA_TYPE POLYBENCH_1D(r,N,n),\\n\\t\\t   DATA_TYPE POLYBENCH_1D(y,N,n))\\n{\\n DATA_TYPE z[N];\\n DATA_TYPE alpha;\\n DATA_TYPE beta;\\n DATA_TYPE sum;\\n\\n int i,k;\\n\\n#pragma scop\\n y[0] = -r[0];\\n beta = SCALAR_VAL(1.0);\\n alpha = -r[0];\\n\\n for (k = 1; k < _PB_N; k++) {\\n   beta = (1-alpha*alpha)*beta;\\n   sum = SCALAR_VAL(0.0);\\n   for (i=0; i<k; i++) {\\n      sum += r[k-i-1]*y[i];\\n   }\\n   alpha = - (r[k] + sum)/beta;\\n\\n   #pragma omp target data map(tofrom: z[0:2000]) map(tofrom: y[0:k]) if(k >= 1)\\n   {\\n   #pragma omp target teams distribute parallel for if(target: k >= 1) map(from: z[0:(((((__INTPTR_TYPE__)k - 1) > 1999 ? 1999 : ((__INTPTR_TYPE__)k - 1))) + 1 > 0 ? ((((__INTPTR_TYPE__)k - 1) > 1999 ? 1999 : ((__INTPTR_TYPE__)k - 1))) + 1 : 0)]) map(to: y[0:k])\\n   for (i=0; i<k; i++) {\\n      z[i] = y[i] + alpha*y[k-i-1];\\n   }\\n   #pragma omp target teams distribute parallel for if(target: k >= 1) map(from: y[0:k]) map(to: z[0:(((((__INTPTR_TYPE__)k - 1) > 1999 ? 1999 : ((__INTPTR_TYPE__)k - 1))) + 1 > 0 ? ((((__INTPTR_TYPE__)k - 1) > 1999 ? 1999 : ((__INTPTR_TYPE__)k - 1))) + 1 : 0)])\\n   for (i=0; i<k; i++) {\\n     y[i] = z[i];\\n   }\\n   }\\n   y[k] = alpha;\\n }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_1D_ARRAY_DECL(r, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(y, DATA_TYPE, N, n);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (n, POLYBENCH_ARRAY(r));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_durbin (n,\\n\\t\\t POLYBENCH_ARRAY(r),\\n\\t\\t POLYBENCH_ARRAY(y));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(y)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(r);\\n  POLYBENCH_FREE_ARRAY(y);\\n\\n  return 0;\\n}\\n", "gated/gramschmidt.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* gramschmidt.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"gramschmidt.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int m, int n,\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,M,N,m,n),\\n\\t\\tDATA_TYPE POLYBENCH_2D(R,N,N,n,n),\\n\\t\\tDATA_TYPE POLYBENCH_2D(Q,M,N,m,n))\\n{\\n  int i, j;\\n\\n  for (i = 0; i < m; i++)\\n    #pragma omp parallel for if(parallel: (n >= 524288) && ((__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)Q + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0) || (__INTPTR_TYPE__)Q + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)))\\n    for (j = 0; j < n; j++) {\\n      A[i][j] = (((DATA_TYPE) ((i*j) % m) / m )*100) + 10;\\n      Q[i][j] = 0.0;\\n    }\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ((double)(n) * ((1 * (double)(n)) - 29523) >= 4294967296) && n >= 1) map(tofrom: R[0:n][0:1200]) private(j)\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++)\\n      R[i][j] = 0.0;\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int m, int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,M,N,m,n),\\n\\t\\t DATA_TYPE POLYBENCH_2D(R,N,N,n,n),\\n\\t\\t DATA_TYPE POLYBENCH_2D(Q,M,N,m,n))\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"R\\");\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++) {\\n\\tif ((i*n+j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n\\tfprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, R[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"R\\");\\n\\n  POLYBENCH_DUMP_BEGIN(\\"Q\\");\\n  for (i = 0; i < m; i++)\\n    for (j = 0; j < n; j++) {\\n\\tif ((i*n+j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n\\tfprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, Q[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"Q\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\n/* QR Decomposition with Modified Gram Schmidt:\\n http://www.inf.ethz.ch/personal/gander/ */\\nstatic\\nvoid kernel_gramschmidt(int m, int n,\\n\\t\\t\\tDATA_TYPE POLYBENCH_2D(A,M,N,m,n),\\n\\t\\t\\tDATA_TYPE POLYBENCH_2D(R,N,N,n,n),\\n\\t\\t\\tDATA_TYPE POLYBENCH_2D(Q,M,N,m,n))\\n{\\n  int i, j, k;\\n\\n  DATA_TYPE nrm;\\n\\n#pragma scop\\n  for (k = 0; k < _PB_N; k++)\\n    {\\n      nrm = SCALAR_VAL(0.0);\\n      for (i = 0; i < _PB_M; i++)\\n        nrm += A[i][k] * A[i][k];\\n      R[k][k] = SQRT_FUN(nrm);\\n      #pragma omp parallel for if(parallel: (m >= 512) && ((__INTPTR_TYPE__)Q + 8 * ((((__INTPTR_TYPE__)m - 1)) * 1200 + (__INTPTR_TYPE__)k + 1) <= (__INTPTR_TYPE__)A + 8 * ((0) * 1200 + (__INTPTR_TYPE__)k) || (__INTPTR_TYPE__)A + 8 * ((((__INTPTR_TYPE__)m - 1)) * 1200 + (__INTPTR_TYPE__)k + 1) <= (__INTPTR_TYPE__)Q + 8 * ((0) * 1200 + (__INTPTR_TYPE__)k)) && ((__INTPTR_TYPE__)Q + 8 * ((((__INTPTR_TYPE__)m - 1)) * 1200 + (__INTPTR_TYPE__)k + 1) <= (__INTPTR_TYPE__)R + 8 * (((__INTPTR_TYPE__)k) * 1200 + (__INTPTR_TYPE__)k) || (__INTPTR_TYPE__)R + 8 * (((__INTPTR_TYPE__)k) * 1200 + (__INTPTR_TYPE__)k + 1) <= (__INTPTR_TYPE__)Q + 8 * ((0) * 1200 + (__INTPTR_TYPE__)k)))\\n      for (i = 0; i < _PB_M; i++)\\n        Q[i][k] = A[i][k] / R[k][k];\\n      for (j = k + 1; j < _PB_N; j++)\\n\\t{\\n\\t  R[k][j] = SCALAR_VAL(0.0);\\n\\t  for (i = 0; i < _PB_M; i++)\\n\\t    R[k][j] += Q[i][k] * A[i][j];\\n\\t  #pragma omp parallel for if(parallel: (m >= 256) && ((__INTPTR_TYPE__)A + 8 * ((((__INTPTR_TYPE__)m - 1)) * 1200 + (__INTPTR_TYPE__)j + 1) <= (__INTPTR_TYPE__)Q + 8 * ((0) * 1200 + (__INTPTR_TYPE__)k) || (__INTPTR_TYPE__)Q + 8 * ((((__INTPTR_TYPE__)m - 1)) * 1200 + (__INTPTR_TYPE__)k + 1) <= (__INTPTR_TYPE__)A + 8 * ((0) * 1200 + (__INTPTR_TYPE__)j)) && ((__INTPTR_TYPE__)A + 8 * ((((__INTPTR_TYPE__)m - 1)) * 1200 + (__INTPTR_TYPE__)j + 1) <= (__INTPTR_TYPE__)R + 8 * (((__INTPTR_TYPE__)k) * 1200 + (__INTPTR_TYPE__)j) || (__INTPTR_TYPE__)R + 8 * (((__INTPTR_TYPE__)k) * 1200 + (__INTPTR_TYPE__)j + 1) <= (__INTPTR_TYPE__)A + 8 * ((0) * 1200 + (__INTPTR_TYPE__)j)))\\n   for (i = 0; i < _PB_M; i++)\\n\\t    A[i][j] = A[i][j] - Q[i][k] * R[k][j];\\n\\t}\\n    }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int m = M;\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(A,DATA_TYPE,M,N,m,n);\\n  POLYBENCH_2D_ARRAY_DECL(R,DATA_TYPE,N,N,n,n);\\n  POLYBENCH_2D_ARRAY_DECL(Q,DATA_TYPE,M,N,m,n);\\n\\n  /* Initialize array(s). */\\n  init_array (m, n,\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(R),\\n\\t      POLYBENCH_ARRAY(Q));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_gramschmidt (m, n,\\n\\t\\t      POLYBENCH_ARRAY(A),\\n\\t\\t      POLYBENCH_ARRAY(R),\\n\\t\\t      POLYBENCH_ARRAY(Q));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(m, n, POLYBENCH_ARRAY(A), POLYBENCH_ARRAY(R), POLYBENCH_ARRAY(Q)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(R);\\n  POLYBENCH_FREE_ARRAY(Q);\\n\\n  return 0;\\n}\\n", "naive/gramschmidt.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* gramschmidt.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"gramschmidt.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int m, int n,\\n\\t\\tDATA_TYPE POLYBENCH_2D(A,M,N,m,n),\\n\\t\\tDATA_TYPE POLYBENCH_2D(R,N,N,n,n),\\n\\t\\tDATA_TYPE POLYBENCH_2D(Q,M,N,m,n))\\n{\\n  int i, j;\\n\\n  for (i = 0; i < m; i++)\\n    #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)Q + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0) || (__INTPTR_TYPE__)Q + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)))\\n    for (j = 0; j < n; j++) {\\n      A[i][j] = (((DATA_TYPE) ((i*j) % m) / m )*100) + 10;\\n      Q[i][j] = 0.0;\\n    }\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: n >= 1) map(tofrom: R[0:n][0:1200]) private(j)\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++)\\n      R[i][j] = 0.0;\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int m, int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,M,N,m,n),\\n\\t\\t DATA_TYPE POLYBENCH_2D(R,N,N,n,n),\\n\\t\\t DATA_TYPE POLYBENCH_2D(Q,M,N,m,n))\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"R\\");\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++) {\\n\\tif ((i*n+j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n\\tfprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, R[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"R\\");\\n\\n  POLYBENCH_DUMP_BEGIN(\\"Q\\");\\n  for (i = 0; i < m; i++)\\n    for (j = 0; j < n; j++) {\\n\\tif ((i*n+j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n\\tfprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, Q[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"Q\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\n/* QR Decomposition with Modified Gram Schmidt:\\n http://www.inf.ethz.ch/personal/gander/ */\\nstatic\\nvoid kernel_gramschmidt(int m, int n,\\n\\t\\t\\tDATA_TYPE POLYBENCH_2D(A,M,N,m,n),\\n\\t\\t\\tDATA_TYPE POLYBENCH_2D(R,N,N,n,n),\\n\\t\\t\\tDATA_TYPE POLYBENCH_2D(Q,M,N,m,n))\\n{\\n  int i, j, k;\\n\\n  DATA_TYPE nrm;\\n\\n#pragma scop\\n  for (k = 0; k < _PB_N; k++)\\n    {\\n      nrm = SCALAR_VAL(0.0);\\n      for (i = 0; i < _PB_M; i++)\\n        nrm += A[i][k] * A[i][k];\\n      R[k][k] = SQRT_FUN(nrm);\\n      #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)Q + 8 * ((((__INTPTR_TYPE__)m - 1)) * 1200 + (__INTPTR_TYPE__)k + 1) <= (__INTPTR_TYPE__)A + 8 * ((0) * 1200 + (__INTPTR_TYPE__)k) || (__INTPTR_TYPE__)A + 8 * ((((__INTPTR_TYPE__)m - 1)) * 1200 + (__INTPTR_TYPE__)k + 1) <= (__INTPTR_TYPE__)Q + 8 * ((0) * 1200 + (__INTPTR_TYPE__)k)) && ((__INTPTR_TYPE__)Q + 8 * ((((__INTPTR_TYPE__)m - 1)) * 1200 + (__INTPTR_TYPE__)k + 1) <= (__INTPTR_TYPE__)R + 8 * (((__INTPTR_TYPE__)k) * 1200 + (__INTPTR_TYPE__)k) || (__INTPTR_TYPE__)R + 8 * (((__INTPTR_TYPE__)k) * 1200 + (__INTPTR_TYPE__)k + 1) <= (__INTPTR_TYPE__)Q + 8 * ((0) * 1200 + (__INTPTR_TYPE__)k)))\\n      for (i = 0; i < _PB_M; i++)\\n        Q[i][k] = A[i][k] / R[k][k];\\n      for (j = k + 1; j < _PB_N; j++)\\n\\t{\\n\\t  R[k][j] = SCALAR_VAL(0.0);\\n\\t  for (i = 0; i < _PB_M; i++)\\n\\t    R[k][j] += Q[i][k] * A[i][j];\\n\\t  #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)A + 8 * ((((__INTPTR_TYPE__)m - 1)) * 1200 + (__INTPTR_TYPE__)j + 1) <= (__INTPTR_TYPE__)Q + 8 * ((0) * 1200 + (__INTPTR_TYPE__)k) || (__INTPTR_TYPE__)Q + 8 * ((((__INTPTR_TYPE__)m - 1)) * 1200 + (__INTPTR_TYPE__)k + 1) <= (__INTPTR_TYPE__)A + 8 * ((0) * 1200 + (__INTPTR_TYPE__)j)) && ((__INTPTR_TYPE__)A + 8 * ((((__INTPTR_TYPE__)m - 1)) * 1200 + (__INTPTR_TYPE__)j + 1) <= (__INTPTR_TYPE__)R + 8 * (((__INTPTR_TYPE__)k) * 1200 + (__INTPTR_TYPE__)j) || (__INTPTR_TYPE__)R + 8 * (((__INTPTR_TYPE__)k) * 1200 + (__INTPTR_TYPE__)j + 1) <= (__INTPTR_TYPE__)A + 8 * ((0) * 1200 + (__INTPTR_TYPE__)j)))\\n   for (i = 0; i < _PB_M; i++)\\n\\t    A[i][j] = A[i][j] - Q[i][k] * R[k][j];\\n\\t}\\n    }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int m = M;\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(A,DATA_TYPE,M,N,m,n);\\n  POLYBENCH_2D_ARRAY_DECL(R,DATA_TYPE,N,N,n,n);\\n  POLYBENCH_2D_ARRAY_DECL(Q,DATA_TYPE,M,N,m,n);\\n\\n  /* Initialize array(s). */\\n  init_array (m, n,\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(R),\\n\\t      POLYBENCH_ARRAY(Q));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_gramschmidt (m, n,\\n\\t\\t      POLYBENCH_ARRAY(A),\\n\\t\\t      POLYBENCH_ARRAY(R),\\n\\t\\t      POLYBENCH_ARRAY(Q));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(m, n, POLYBENCH_ARRAY(A), POLYBENCH_ARRAY(R), POLYBENCH_ARRAY(Q)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(R);\\n  POLYBENCH_FREE_ARRAY(Q);\\n\\n  return 0;\\n}\\n", "gated/lu.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* lu.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"lu.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,N,N,n,n))\\n{\\n  int i, j;\\n\\n  #pragma omp target teams distribute parallel for if(target: ((double)(n) * ((1 + 6 * (double)(n)) - 49205) >= 2147483648) && n >= 1) map(tofrom: A[0:n][0:2000]) private(j)\\n  for (i = 0; i < n; i++)\\n    {\\n      for (j = 0; j <= i; j++)\\n\\tA[i][j] = (DATA_TYPE)(-j % n) / n + 1;\\n      for (j = i+1; j < n; j++) {\\n\\tA[i][j] = 0;\\n      }\\n      A[i][i] = 1;\\n    }\\n\\n  /* Make the matrix positive semi-definite. */\\n  /* not necessary for LU, but using same code as cholesky */\\n  int r,s,t;\\n  POLYBENCH_2D_ARRAY_DECL(B, DATA_TYPE, N, N, n, n);\\n  for (r = 0; r < n; ++r)\\n    for (s = 0; s < n; ++s)\\n      (POLYBENCH_ARRAY(B))[r][s] = 0;\\n  for (t = 0; t < n; ++t)\\n    for (r = 0; r < n; ++r)\\n      for (s = 0; s < n; ++s)\\n\\t(POLYBENCH_ARRAY(B))[r][s] += A[r][t] * A[s][t];\\n    for (r = 0; r < n; ++r)\\n      for (s = 0; s < n; ++s)\\n\\tA[r][s] = (POLYBENCH_ARRAY(B))[r][s];\\n  POLYBENCH_FREE_ARRAY(B);\\n\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,N,N,n,n))\\n\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"A\\");\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++) {\\n      if ((i * n + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"A\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_lu(int n,\\n\\t       DATA_TYPE POLYBENCH_2D(A,N,N,n,n))\\n{\\n  int i, j, k;\\n\\n#pragma scop\\n  for (i = 0; i < _PB_N; i++) {\\n    for (j = 0; j <i; j++) {\\n       for (k = 0; k < j; k++) {\\n          A[i][j] -= A[i][k] * A[k][j];\\n       }\\n        A[i][j] /= A[j][j];\\n    }\\n   #pragma omp target teams distribute parallel for if(target: ((double)(n) * ((3 * (double)(i)) - 98409) >= 2147483648) && (i < 0 ? i : 0) >= 0 && (-(__INTPTR_TYPE__)i + (__INTPTR_TYPE__)n) >= 1 && i >= 1) map(tofrom: A[(i < 0 ? i : 0):(i) - ((i < 0 ? i : 0)) + 1][0:2000]) private(k)\\n   for (j = i; j < _PB_N; j++) {\\n       for (k = 0; k < i; k++) {\\n          A[i][j] -= A[i][k] * A[k][j];\\n       }\\n    }\\n  }\\n#pragma endscop\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(A, DATA_TYPE, N, N, n, n);\\n\\n  /* Initialize array(s). */\\n  init_array (n, POLYBENCH_ARRAY(A));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_lu (n, POLYBENCH_ARRAY(A));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(A)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n\\n  return 0;\\n}\\n", "naive/lu.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* lu.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"lu.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,N,N,n,n))\\n{\\n  int i, j;\\n\\n  #pragma omp target teams distribute parallel for if(target: n >= 1) map(tofrom: A[0:n][0:2000]) private(j)\\n  for (i = 0; i < n; i++)\\n    {\\n      for (j = 0; j <= i; j++)\\n\\tA[i][j] = (DATA_TYPE)(-j % n) / n + 1;\\n      for (j = i+1; j < n; j++) {\\n\\tA[i][j] = 0;\\n      }\\n      A[i][i] = 1;\\n    }\\n\\n  /* Make the matrix positive semi-definite. */\\n  /* not necessary for LU, but using same code as cholesky */\\n  int r,s,t;\\n  POLYBENCH_2D_ARRAY_DECL(B, DATA_TYPE, N, N, n, n);\\n  for (r = 0; r < n; ++r)\\n    for (s = 0; s < n; ++s)\\n      (POLYBENCH_ARRAY(B))[r][s] = 0;\\n  for (t = 0; t < n; ++t)\\n    for (r = 0; r < n; ++r)\\n      for (s = 0; s < n; ++s)\\n\\t(POLYBENCH_ARRAY(B))[r][s] += A[r][t] * A[s][t];\\n    for (r = 0; r < n; ++r)\\n      for (s = 0; s < n; ++s)\\n\\tA[r][s] = (POLYBENCH_ARRAY(B))[r][s];\\n  POLYBENCH_FREE_ARRAY(B);\\n\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,N,N,n,n))\\n\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"A\\");\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++) {\\n      if ((i * n + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"A\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_lu(int n,\\n\\t       DATA_TYPE POLYBENCH_2D(A,N,N,n,n))\\n{\\n  int i, j, k;\\n\\n#pragma scop\\n  for (i = 0; i < _PB_N; i++) {\\n    for (j = 0; j <i; j++) {\\n       for (k = 0; k < j; k++) {\\n          A[i][j] -= A[i][k] * A[k][j];\\n       }\\n        A[i][j] /= A[j][j];\\n    }\\n   #pragma omp target teams distribute parallel for if(target: (i < 0 ? i : 0) >= 0 && (-(__INTPTR_TYPE__)i + (__INTPTR_TYPE__)n) >= 1 && i >= 1) map(tofrom: A[(i < 0 ? i : 0):(i) - ((i < 0 ? i : 0)) + 1][0:2000]) private(k)\\n   for (j = i; j < _PB_N; j++) {\\n       for (k = 0; k < i; k++) {\\n          A[i][j] -= A[i][k] * A[k][j];\\n       }\\n    }\\n  }\\n#pragma endscop\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(A, DATA_TYPE, N, N, n, n);\\n\\n  /* Initialize array(s). */\\n  init_array (n, POLYBENCH_ARRAY(A));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_lu (n, POLYBENCH_ARRAY(A));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(A)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n\\n  return 0;\\n}\\n", "gated/ludcmp.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* ludcmp.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"ludcmp.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,N,N,n,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(b,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(x,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(y,N,n))\\n{\\n  int i, j;\\n  DATA_TYPE fn = (DATA_TYPE)n;\\n\\n  #pragma omp parallel for if(parallel: (n >= 1048576) && ((__INTPTR_TYPE__)x + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)y + 0 || (__INTPTR_TYPE__)y + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)x + 0) && ((__INTPTR_TYPE__)x + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)b + 0 || (__INTPTR_TYPE__)b + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)x + 0) && ((__INTPTR_TYPE__)y + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)b + 0 || (__INTPTR_TYPE__)b + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)y + 0))\\n  for (i = 0; i < n; i++)\\n    {\\n      x[i] = 0;\\n      y[i] = 0;\\n      b[i] = (i+1)/fn/2.0 + 4;\\n    }\\n\\n  #pragma omp target teams distribute parallel for if(target: ((double)(n) * ((1 + 6 * (double)(n)) - 49205) >= 2147483648) && n >= 1) map(tofrom: A[0:n][0:2000]) private(j)\\n  for (i = 0; i < n; i++)\\n    {\\n      for (j = 0; j <= i; j++)\\n\\tA[i][j] = (DATA_TYPE)(-j % n) / n + 1;\\n      for (j = i+1; j < n; j++) {\\n\\tA[i][j] = 0;\\n      }\\n      A[i][i] = 1;\\n    }\\n\\n  /* Make the matrix positive semi-definite. */\\n  /* not necessary for LU, but using same code as cholesky */\\n  int r,s,t;\\n  POLYBENCH_2D_ARRAY_DECL(B, DATA_TYPE, N, N, n, n);\\n  for (r = 0; r < n; ++r)\\n    for (s = 0; s < n; ++s)\\n      (POLYBENCH_ARRAY(B))[r][s] = 0;\\n  for (t = 0; t < n; ++t)\\n    for (r = 0; r < n; ++r)\\n      for (s = 0; s < n; ++s)\\n\\t(POLYBENCH_ARRAY(B))[r][s] += A[r][t] * A[s][t];\\n    for (r = 0; r < n; ++r)\\n      for (s = 0; s < n; ++s)\\n\\tA[r][s] = (POLYBENCH_ARRAY(B))[r][s];\\n  POLYBENCH_FREE_ARRAY(B);\\n\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_1D(x,N,n))\\n\\n{\\n  int i;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"x\\");\\n  for (i = 0; i < n; i++) {\\n    if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, x[i]);\\n  }\\n  POLYBENCH_DUMP_END(\\"x\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_ludcmp(int n,\\n\\t\\t   DATA_TYPE POLYBENCH_2D(A,N,N,n,n),\\n\\t\\t   DATA_TYPE POLYBENCH_1D(b,N,n),\\n\\t\\t   DATA_TYPE POLYBENCH_1D(x,N,n),\\n\\t\\t   DATA_TYPE POLYBENCH_1D(y,N,n))\\n{\\n  int i, j, k;\\n\\n  DATA_TYPE w;\\n\\n#pragma scop\\n  for (i = 0; i < _PB_N; i++) {\\n    for (j = 0; j <i; j++) {\\n       w = A[i][j];\\n       for (k = 0; k < j; k++) {\\n          w -= A[i][k] * A[k][j];\\n       }\\n        A[i][j] = w / A[j][j];\\n    }\\n   #pragma omp target teams distribute parallel for if(target: ((double)(n) * ((3 * (double)(i)) - 98409) >= 2147483648) && (i < 0 ? i : 0) >= 0 && (-(__INTPTR_TYPE__)i + (__INTPTR_TYPE__)n) >= 1 && i >= 1) map(tofrom: A[(i < 0 ? i : 0):(i) - ((i < 0 ? i : 0)) + 1][0:2000]) private(w, k)\\n   for (j = i; j < _PB_N; j++) {\\n       w = A[i][j];\\n       for (k = 0; k < i; k++) {\\n          w -= A[i][k] * A[k][j];\\n       }\\n       A[i][j] = w;\\n    }\\n  }\\n\\n  for (i = 0; i < _PB_N; i++) {\\n     w = b[i];\\n     for (j = 0; j < i; j++)\\n        w -= A[i][j] * y[j];\\n     y[i] = w;\\n  }\\n\\n   for (i = _PB_N-1; i >=0; i--) {\\n     w = y[i];\\n     for (j = i+1; j < _PB_N; j++)\\n        w -= A[i][j] * x[j];\\n     x[i] = w / A[i][i];\\n  }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(A, DATA_TYPE, N, N, n, n);\\n  POLYBENCH_1D_ARRAY_DECL(b, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(x, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(y, DATA_TYPE, N, n);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (n,\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(b),\\n\\t      POLYBENCH_ARRAY(x),\\n\\t      POLYBENCH_ARRAY(y));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_ludcmp (n,\\n\\t\\t POLYBENCH_ARRAY(A),\\n\\t\\t POLYBENCH_ARRAY(b),\\n\\t\\t POLYBENCH_ARRAY(x),\\n\\t\\t POLYBENCH_ARRAY(y));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(x)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(b);\\n  POLYBENCH_FREE_ARRAY(x);\\n  POLYBENCH_FREE_ARRAY(y);\\n\\n  return 0;\\n}\\n", "naive/ludcmp.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* ludcmp.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"ludcmp.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,N,N,n,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(b,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(x,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(y,N,n))\\n{\\n  int i, j;\\n  DATA_TYPE fn = (DATA_TYPE)n;\\n\\n  #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)x + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)y + 0 || (__INTPTR_TYPE__)y + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)x + 0) && ((__INTPTR_TYPE__)x + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)b + 0 || (__INTPTR_TYPE__)b + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)x + 0) && ((__INTPTR_TYPE__)y + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)b + 0 || (__INTPTR_TYPE__)b + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)y + 0))\\n  for (i = 0; i < n; i++)\\n    {\\n      x[i] = 0;\\n      y[i] = 0;\\n      b[i] = (i+1)/fn/2.0 + 4;\\n    }\\n\\n  #pragma omp target teams distribute parallel for if(target: n >= 1) map(tofrom: A[0:n][0:2000]) private(j)\\n  for (i = 0; i < n; i++)\\n    {\\n      for (j = 0; j <= i; j++)\\n\\tA[i][j] = (DATA_TYPE)(-j % n) / n + 1;\\n      for (j = i+1; j < n; j++) {\\n\\tA[i][j] = 0;\\n      }\\n      A[i][i] = 1;\\n    }\\n\\n  /* Make the matrix positive semi-definite. */\\n  /* not necessary for LU, but using same code as cholesky */\\n  int r,s,t;\\n  POLYBENCH_2D_ARRAY_DECL(B, DATA_TYPE, N, N, n, n);\\n  for (r = 0; r < n; ++r)\\n    for (s = 0; s < n; ++s)\\n      (POLYBENCH_ARRAY(B))[r][s] = 0;\\n  for (t = 0; t < n; ++t)\\n    for (r = 0; r < n; ++r)\\n      for (s = 0; s < n; ++s)\\n\\t(POLYBENCH_ARRAY(B))[r][s] += A[r][t] * A[s][t];\\n    for (r = 0; r < n; ++r)\\n      for (s = 0; s < n; ++s)\\n\\tA[r][s] = (POLYBENCH_ARRAY(B))[r][s];\\n  POLYBENCH_FREE_ARRAY(B);\\n\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_1D(x,N,n))\\n\\n{\\n  int i;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"x\\");\\n  for (i = 0; i < n; i++) {\\n    if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, x[i]);\\n  }\\n  POLYBENCH_DUMP_END(\\"x\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_ludcmp(int n,\\n\\t\\t   DATA_TYPE POLYBENCH_2D(A,N,N,n,n),\\n\\t\\t   DATA_TYPE POLYBENCH_1D(b,N,n),\\n\\t\\t   DATA_TYPE POLYBENCH_1D(x,N,n),\\n\\t\\t   DATA_TYPE POLYBENCH_1D(y,N,n))\\n{\\n  int i, j, k;\\n\\n  DATA_TYPE w;\\n\\n#pragma scop\\n  for (i = 0; i < _PB_N; i++) {\\n    for (j = 0; j <i; j++) {\\n       w = A[i][j];\\n       for (k = 0; k < j; k++) {\\n          w -= A[i][k] * A[k][j];\\n       }\\n        A[i][j] = w / A[j][j];\\n    }\\n   #pragma omp target teams distribute parallel for if(target: (i < 0 ? i : 0) >= 0 && (-(__INTPTR_TYPE__)i + (__INTPTR_TYPE__)n) >= 1 && i >= 1) map(tofrom: A[(i < 0 ? i : 0):(i) - ((i < 0 ? i : 0)) + 1][0:2000]) private(w, k)\\n   for (j = i; j < _PB_N; j++) {\\n       w = A[i][j];\\n       for (k = 0; k < i; k++) {\\n          w -= A[i][k] * A[k][j];\\n       }\\n       A[i][j] = w;\\n    }\\n  }\\n\\n  for (i = 0; i < _PB_N; i++) {\\n     w = b[i];\\n     for (j = 0; j < i; j++)\\n        w -= A[i][j] * y[j];\\n     y[i] = w;\\n  }\\n\\n   for (i = _PB_N-1; i >=0; i--) {\\n     w = y[i];\\n     for (j = i+1; j < _PB_N; j++)\\n        w -= A[i][j] * x[j];\\n     x[i] = w / A[i][i];\\n  }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(A, DATA_TYPE, N, N, n, n);\\n  POLYBENCH_1D_ARRAY_DECL(b, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(x, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(y, DATA_TYPE, N, n);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (n,\\n\\t      POLYBENCH_ARRAY(A),\\n\\t      POLYBENCH_ARRAY(b),\\n\\t      POLYBENCH_ARRAY(x),\\n\\t      POLYBENCH_ARRAY(y));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_ludcmp (n,\\n\\t\\t POLYBENCH_ARRAY(A),\\n\\t\\t POLYBENCH_ARRAY(b),\\n\\t\\t POLYBENCH_ARRAY(x),\\n\\t\\t POLYBENCH_ARRAY(y));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(x)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(b);\\n  POLYBENCH_FREE_ARRAY(x);\\n  POLYBENCH_FREE_ARRAY(y);\\n\\n  return 0;\\n}\\n", "gated/trisolv.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* trisolv.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"trisolv.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int n,\\n\\t\\tDATA_TYPE POLYBENCH_2D(L,N,N,n,n),\\n\\t\\tDATA_TYPE POLYBENCH_1D(x,N,n),\\n\\t\\tDATA_TYPE POLYBENCH_1D(b,N,n))\\n{\\n  int i, j;\\n\\n  for (i = 0; i < n; i++)\\n    {\\n      x[i] = - 999;\\n      b[i] =  i ;\\n      #pragma omp target teams distribute parallel for if(target: (i >= 536870912) && i >= 0) map(tofrom: L[i:1][0:2000])\\n      for (j = 0; j <= i; j++)\\n\\tL[i][j] = (DATA_TYPE) (i+n-j+1)*2/n;\\n    }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_1D(x,N,n))\\n\\n{\\n  int i;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"x\\");\\n  for (i = 0; i < n; i++) {\\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, x[i]);\\n    if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n  }\\n  POLYBENCH_DUMP_END(\\"x\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_trisolv(int n,\\n\\t\\t    DATA_TYPE POLYBENCH_2D(L,N,N,n,n),\\n\\t\\t    DATA_TYPE POLYBENCH_1D(x,N,n),\\n\\t\\t    DATA_TYPE POLYBENCH_1D(b,N,n))\\n{\\n  int i, j;\\n\\n#pragma scop\\n  for (i = 0; i < _PB_N; i++)\\n    {\\n      x[i] = b[i];\\n      for (j = 0; j <i; j++)\\n        x[i] -= L[i][j] * x[j];\\n      x[i] = x[i] / L[i][i];\\n    }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(L, DATA_TYPE, N, N, n, n);\\n  POLYBENCH_1D_ARRAY_DECL(x, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(b, DATA_TYPE, N, n);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (n, POLYBENCH_ARRAY(L), POLYBENCH_ARRAY(x), POLYBENCH_ARRAY(b));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_trisolv (n, POLYBENCH_ARRAY(L), POLYBENCH_ARRAY(x), POLYBENCH_ARRAY(b));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(x)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(L);\\n  POLYBENCH_FREE_ARRAY(x);\\n  POLYBENCH_FREE_ARRAY(b);\\n\\n  return 0;\\n}\\n", "naive/trisolv.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* trisolv.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"trisolv.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array(int n,\\n\\t\\tDATA_TYPE POLYBENCH_2D(L,N,N,n,n),\\n\\t\\tDATA_TYPE POLYBENCH_1D(x,N,n),\\n\\t\\tDATA_TYPE POLYBENCH_1D(b,N,n))\\n{\\n  int i, j;\\n\\n  for (i = 0; i < n; i++)\\n    {\\n      x[i] = - 999;\\n      b[i] =  i ;\\n      #pragma omp target teams distribute parallel for if(target: i >= 0) map(tofrom: L[i:1][0:2000])\\n      for (j = 0; j <= i; j++)\\n\\tL[i][j] = (DATA_TYPE) (i+n-j+1)*2/n;\\n    }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_1D(x,N,n))\\n\\n{\\n  int i;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"x\\");\\n  for (i = 0; i < n; i++) {\\n    fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, x[i]);\\n    if (i % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n  }\\n  POLYBENCH_DUMP_END(\\"x\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_trisolv(int n,\\n\\t\\t    DATA_TYPE POLYBENCH_2D(L,N,N,n,n),\\n\\t\\t    DATA_TYPE POLYBENCH_1D(x,N,n),\\n\\t\\t    DATA_TYPE POLYBENCH_1D(b,N,n))\\n{\\n  int i, j;\\n\\n#pragma scop\\n  for (i = 0; i < _PB_N; i++)\\n    {\\n      x[i] = b[i];\\n      for (j = 0; j <i; j++)\\n        x[i] -= L[i][j] * x[j];\\n      x[i] = x[i] / L[i][i];\\n    }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(L, DATA_TYPE, N, N, n, n);\\n  POLYBENCH_1D_ARRAY_DECL(x, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(b, DATA_TYPE, N, n);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (n, POLYBENCH_ARRAY(L), POLYBENCH_ARRAY(x), POLYBENCH_ARRAY(b));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_trisolv (n, POLYBENCH_ARRAY(L), POLYBENCH_ARRAY(x), POLYBENCH_ARRAY(b));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(x)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(L);\\n  POLYBENCH_FREE_ARRAY(x);\\n  POLYBENCH_FREE_ARRAY(b);\\n\\n  return 0;\\n}\\n", "gated/deriche.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* deriche.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"deriche.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int w, int h, DATA_TYPE* alpha,\\n\\t\\t DATA_TYPE POLYBENCH_2D(imgIn,W,H,w,h),\\n\\t\\t DATA_TYPE POLYBENCH_2D(imgOut,W,H,w,h))\\n{\\n  int i, j;\\n\\n  *alpha=0.25; //parameter of the filter\\n\\n  //input should be between 0 and 1 (grayscale image pixel)\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ((double)(w) * ((6 * (double)(h)) - 26571) >= 4294967296) && w >= 1 && h >= 1) map(tofrom: imgIn[0:w][0:2160]) private(j)\\n  for (i = 0; i < w; i++)\\n     for (j = 0; j < h; j++)\\n\\timgIn[i][j] = (DATA_TYPE) ((313*i+991*j)%65536) / 65535.0f;\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int w, int h,\\n\\t\\t DATA_TYPE POLYBENCH_2D(imgOut,W,H,w,h))\\n\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"imgOut\\");\\n  for (i = 0; i < w; i++)\\n    for (j = 0; j < h; j++) {\\n      if ((i * h + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, imgOut[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"imgOut\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\n/* Original code provided by Gael Deest */\\nstatic\\nvoid kernel_deriche(int w, int h, DATA_TYPE alpha,\\n       DATA_TYPE POLYBENCH_2D(imgIn, W, H, w, h),\\n       DATA_TYPE POLYBENCH_2D(imgOut, W, H, w, h),\\n       DATA_TYPE POLYBENCH_2D(y1, W, H, w, h),\\n       DATA_TYPE POLYBENCH_2D(y2, W, H, w, h)) {\\n    int i,j;\\n    DATA_TYPE xm1, tm1, ym1, ym2;\\n    DATA_TYPE xp1, xp2;\\n    DATA_TYPE tp1, tp2;\\n    DATA_TYPE yp1, yp2;\\n\\n    DATA_TYPE k;\\n    DATA_TYPE a1, a2, a3, a4, a5, a6, a7, a8;\\n    DATA_TYPE b1, b2, c1, c2;\\n\\n#pragma scop\\n   k = (SCALAR_VAL(1.0)-EXP_FUN(-alpha))*(SCALAR_VAL(1.0)-EXP_FUN(-alpha))/(SCALAR_VAL(1.0)+SCALAR_VAL(2.0)*alpha*EXP_FUN(-alpha)-EXP_FUN(SCALAR_VAL(2.0)*alpha));\\n   a1 = a5 = k;\\n   a2 = a6 = k*EXP_FUN(-alpha)*(alpha-SCALAR_VAL(1.0));\\n   a3 = a7 = k*EXP_FUN(-alpha)*(alpha+SCALAR_VAL(1.0));\\n   a4 = a8 = -k*EXP_FUN(SCALAR_VAL(-2.0)*alpha);\\n   b1 =  POW_FUN(SCALAR_VAL(2.0),-alpha);\\n   b2 = -EXP_FUN(SCALAR_VAL(-2.0)*alpha);\\n   c1 = c2 = 1;\\n\\n   for (i=0; i<_PB_W; i++) {\\n        ym1 = SCALAR_VAL(0.0);\\n        ym2 = SCALAR_VAL(0.0);\\n        xm1 = SCALAR_VAL(0.0);\\n        for (j=0; j<_PB_H; j++) {\\n            y1[i][j] = a1*imgIn[i][j] + a2*xm1 + b1*ym1 + b2*ym2;\\n            xm1 = imgIn[i][j];\\n            ym2 = ym1;\\n            ym1 = y1[i][j];\\n        }\\n    }\\n\\n    for (i=0; i<_PB_W; i++) {\\n        yp1 = SCALAR_VAL(0.0);\\n        yp2 = SCALAR_VAL(0.0);\\n        xp1 = SCALAR_VAL(0.0);\\n        xp2 = SCALAR_VAL(0.0);\\n        for (j=_PB_H-1; j>=0; j--) {\\n            y2[i][j] = a3*xp1 + a4*xp2 + b1*yp1 + b2*yp2;\\n            xp2 = xp1;\\n            xp1 = imgIn[i][j];\\n            yp2 = yp1;\\n            yp1 = y2[i][j];\\n        }\\n    }\\n\\n    for (i=0; i<_PB_W; i++)\\n        #pragma omp parallel for if(parallel: (h >= 1048576) && ((__INTPTR_TYPE__)imgOut + 4 * (((__INTPTR_TYPE__)i) * 2160 + ((__INTPTR_TYPE__)h - 1) + 1) <= (__INTPTR_TYPE__)y1 + 4 * (((__INTPTR_TYPE__)i) * 2160 + 0) || (__INTPTR_TYPE__)y1 + 4 * (((__INTPTR_TYPE__)i) * 2160 + ((__INTPTR_TYPE__)h - 1) + 1) <= (__INTPTR_TYPE__)imgOut + 4 * (((__INTPTR_TYPE__)i) * 2160 + 0)) && ((__INTPTR_TYPE__)imgOut + 4 * (((__INTPTR_TYPE__)i) * 2160 + ((__INTPTR_TYPE__)h - 1) + 1) <= (__INTPTR_TYPE__)y2 + 4 * (((__INTPTR_TYPE__)i) * 2160 + 0) || (__INTPTR_TYPE__)y2 + 4 * (((__INTPTR_TYPE__)i) * 2160 + ((__INTPTR_TYPE__)h - 1) + 1) <= (__INTPTR_TYPE__)imgOut + 4 * (((__INTPTR_TYPE__)i) * 2160 + 0)))\\n        for (j=0; j<_PB_H; j++) {\\n            imgOut[i][j] = c1 * (y1[i][j] + y2[i][j]);\\n        }\\n\\n    for (j=0; j<_PB_H; j++) {\\n        tm1 = SCALAR_VAL(0.0);\\n        ym1 = SCALAR_VAL(0.0);\\n        ym2 = SCALAR_VAL(0.0);\\n        for (i=0; i<_PB_W; i++) {\\n            y1[i][j] = a5*imgOut[i][j] + a6*tm1 + b1*ym1 + b2*ym2;\\n            tm1 = imgOut[i][j];\\n            ym2 = ym1;\\n            ym1 = y1 [i][j];\\n        }\\n    }\\n\\n\\n    for (j=0; j<_PB_H; j++) {\\n        tp1 = SCALAR_VAL(0.0);\\n        tp2 = SCALAR_VAL(0.0);\\n        yp1 = SCALAR_VAL(0.0);\\n        yp2 = SCALAR_VAL(0.0);\\n        for (i=_PB_W-1; i>=0; i--) {\\n            y2[i][j] = a7*tp1 + a8*tp2 + b1*yp1 + b2*yp2;\\n            tp2 = tp1;\\n            tp1 = imgOut[i][j];\\n            yp2 = yp1;\\n            yp1 = y2[i][j];\\n        }\\n    }\\n\\n    for (i=0; i<_PB_W; i++)\\n        #pragma omp parallel for if(parallel: (h >= 1048576) && ((__INTPTR_TYPE__)imgOut + 4 * (((__INTPTR_TYPE__)i) * 2160 + ((__INTPTR_TYPE__)h - 1) + 1) <= (__INTPTR_TYPE__)y1 + 4 * (((__INTPTR_TYPE__)i) * 2160 + 0) || (__INTPTR_TYPE__)y1 + 4 * (((__INTPTR_TYPE__)i) * 2160 + ((__INTPTR_TYPE__)h - 1) + 1) <= (__INTPTR_TYPE__)imgOut + 4 * (((__INTPTR_TYPE__)i) * 2160 + 0)) && ((__INTPTR_TYPE__)imgOut + 4 * (((__INTPTR_TYPE__)i) * 2160 + ((__INTPTR_TYPE__)h - 1) + 1) <= (__INTPTR_TYPE__)y2 + 4 * (((__INTPTR_TYPE__)i) * 2160 + 0) || (__INTPTR_TYPE__)y2 + 4 * (((__INTPTR_TYPE__)i) * 2160 + ((__INTPTR_TYPE__)h - 1) + 1) <= (__INTPTR_TYPE__)imgOut + 4 * (((__INTPTR_TYPE__)i) * 2160 + 0)))\\n        for (j=0; j<_PB_H; j++)\\n            imgOut[i][j] = c2*(y1[i][j] + y2[i][j]);\\n\\n#pragma endscop\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int w = W;\\n  int h = H;\\n\\n  /* Variable declaration/allocation. */\\n  DATA_TYPE alpha;\\n  POLYBENCH_2D_ARRAY_DECL(imgIn, DATA_TYPE, W, H, w, h);\\n  POLYBENCH_2D_ARRAY_DECL(imgOut, DATA_TYPE, W, H, w, h);\\n  POLYBENCH_2D_ARRAY_DECL(y1, DATA_TYPE, W, H, w, h);\\n  POLYBENCH_2D_ARRAY_DECL(y2, DATA_TYPE, W, H, w, h);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (w, h, &alpha, POLYBENCH_ARRAY(imgIn), POLYBENCH_ARRAY(imgOut));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_deriche (w, h, alpha, POLYBENCH_ARRAY(imgIn), POLYBENCH_ARRAY(imgOut), POLYBENCH_ARRAY(y1), POLYBENCH_ARRAY(y2));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(w, h, POLYBENCH_ARRAY(imgOut)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(imgIn);\\n  POLYBENCH_FREE_ARRAY(imgOut);\\n  POLYBENCH_FREE_ARRAY(y1);\\n  POLYBENCH_FREE_ARRAY(y2);\\n\\n  return 0;\\n}\\n", "naive/deriche.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* deriche.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"deriche.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int w, int h, DATA_TYPE* alpha,\\n\\t\\t DATA_TYPE POLYBENCH_2D(imgIn,W,H,w,h),\\n\\t\\t DATA_TYPE POLYBENCH_2D(imgOut,W,H,w,h))\\n{\\n  int i, j;\\n\\n  *alpha=0.25; //parameter of the filter\\n\\n  //input should be between 0 and 1 (grayscale image pixel)\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: w >= 1 && h >= 1) map(tofrom: imgIn[0:w][0:2160]) private(j)\\n  for (i = 0; i < w; i++)\\n     for (j = 0; j < h; j++)\\n\\timgIn[i][j] = (DATA_TYPE) ((313*i+991*j)%65536) / 65535.0f;\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int w, int h,\\n\\t\\t DATA_TYPE POLYBENCH_2D(imgOut,W,H,w,h))\\n\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"imgOut\\");\\n  for (i = 0; i < w; i++)\\n    for (j = 0; j < h; j++) {\\n      if ((i * h + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, imgOut[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"imgOut\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\n/* Original code provided by Gael Deest */\\nstatic\\nvoid kernel_deriche(int w, int h, DATA_TYPE alpha,\\n       DATA_TYPE POLYBENCH_2D(imgIn, W, H, w, h),\\n       DATA_TYPE POLYBENCH_2D(imgOut, W, H, w, h),\\n       DATA_TYPE POLYBENCH_2D(y1, W, H, w, h),\\n       DATA_TYPE POLYBENCH_2D(y2, W, H, w, h)) {\\n    int i,j;\\n    DATA_TYPE xm1, tm1, ym1, ym2;\\n    DATA_TYPE xp1, xp2;\\n    DATA_TYPE tp1, tp2;\\n    DATA_TYPE yp1, yp2;\\n\\n    DATA_TYPE k;\\n    DATA_TYPE a1, a2, a3, a4, a5, a6, a7, a8;\\n    DATA_TYPE b1, b2, c1, c2;\\n\\n#pragma scop\\n   k = (SCALAR_VAL(1.0)-EXP_FUN(-alpha))*(SCALAR_VAL(1.0)-EXP_FUN(-alpha))/(SCALAR_VAL(1.0)+SCALAR_VAL(2.0)*alpha*EXP_FUN(-alpha)-EXP_FUN(SCALAR_VAL(2.0)*alpha));\\n   a1 = a5 = k;\\n   a2 = a6 = k*EXP_FUN(-alpha)*(alpha-SCALAR_VAL(1.0));\\n   a3 = a7 = k*EXP_FUN(-alpha)*(alpha+SCALAR_VAL(1.0));\\n   a4 = a8 = -k*EXP_FUN(SCALAR_VAL(-2.0)*alpha);\\n   b1 =  POW_FUN(SCALAR_VAL(2.0),-alpha);\\n   b2 = -EXP_FUN(SCALAR_VAL(-2.0)*alpha);\\n   c1 = c2 = 1;\\n\\n   for (i=0; i<_PB_W; i++) {\\n        ym1 = SCALAR_VAL(0.0);\\n        ym2 = SCALAR_VAL(0.0);\\n        xm1 = SCALAR_VAL(0.0);\\n        for (j=0; j<_PB_H; j++) {\\n            y1[i][j] = a1*imgIn[i][j] + a2*xm1 + b1*ym1 + b2*ym2;\\n            xm1 = imgIn[i][j];\\n            ym2 = ym1;\\n            ym1 = y1[i][j];\\n        }\\n    }\\n\\n    for (i=0; i<_PB_W; i++) {\\n        yp1 = SCALAR_VAL(0.0);\\n        yp2 = SCALAR_VAL(0.0);\\n        xp1 = SCALAR_VAL(0.0);\\n        xp2 = SCALAR_VAL(0.0);\\n        for (j=_PB_H-1; j>=0; j--) {\\n            y2[i][j] = a3*xp1 + a4*xp2 + b1*yp1 + b2*yp2;\\n            xp2 = xp1;\\n            xp1 = imgIn[i][j];\\n            yp2 = yp1;\\n            yp1 = y2[i][j];\\n        }\\n    }\\n\\n    for (i=0; i<_PB_W; i++)\\n        #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)imgOut + 4 * (((__INTPTR_TYPE__)i) * 2160 + ((__INTPTR_TYPE__)h - 1) + 1) <= (__INTPTR_TYPE__)y1 + 4 * (((__INTPTR_TYPE__)i) * 2160 + 0) || (__INTPTR_TYPE__)y1 + 4 * (((__INTPTR_TYPE__)i) * 2160 + ((__INTPTR_TYPE__)h - 1) + 1) <= (__INTPTR_TYPE__)imgOut + 4 * (((__INTPTR_TYPE__)i) * 2160 + 0)) && ((__INTPTR_TYPE__)imgOut + 4 * (((__INTPTR_TYPE__)i) * 2160 + ((__INTPTR_TYPE__)h - 1) + 1) <= (__INTPTR_TYPE__)y2 + 4 * (((__INTPTR_TYPE__)i) * 2160 + 0) || (__INTPTR_TYPE__)y2 + 4 * (((__INTPTR_TYPE__)i) * 2160 + ((__INTPTR_TYPE__)h - 1) + 1) <= (__INTPTR_TYPE__)imgOut + 4 * (((__INTPTR_TYPE__)i) * 2160 + 0)))\\n        for (j=0; j<_PB_H; j++) {\\n            imgOut[i][j] = c1 * (y1[i][j] + y2[i][j]);\\n        }\\n\\n    for (j=0; j<_PB_H; j++) {\\n        tm1 = SCALAR_VAL(0.0);\\n        ym1 = SCALAR_VAL(0.0);\\n        ym2 = SCALAR_VAL(0.0);\\n        for (i=0; i<_PB_W; i++) {\\n            y1[i][j] = a5*imgOut[i][j] + a6*tm1 + b1*ym1 + b2*ym2;\\n            tm1 = imgOut[i][j];\\n            ym2 = ym1;\\n            ym1 = y1 [i][j];\\n        }\\n    }\\n\\n\\n    for (j=0; j<_PB_H; j++) {\\n        tp1 = SCALAR_VAL(0.0);\\n        tp2 = SCALAR_VAL(0.0);\\n        yp1 = SCALAR_VAL(0.0);\\n        yp2 = SCALAR_VAL(0.0);\\n        for (i=_PB_W-1; i>=0; i--) {\\n            y2[i][j] = a7*tp1 + a8*tp2 + b1*yp1 + b2*yp2;\\n            tp2 = tp1;\\n            tp1 = imgOut[i][j];\\n            yp2 = yp1;\\n            yp1 = y2[i][j];\\n        }\\n    }\\n\\n    for (i=0; i<_PB_W; i++)\\n        #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)imgOut + 4 * (((__INTPTR_TYPE__)i) * 2160 + ((__INTPTR_TYPE__)h - 1) + 1) <= (__INTPTR_TYPE__)y1 + 4 * (((__INTPTR_TYPE__)i) * 2160 + 0) || (__INTPTR_TYPE__)y1 + 4 * (((__INTPTR_TYPE__)i) * 2160 + ((__INTPTR_TYPE__)h - 1) + 1) <= (__INTPTR_TYPE__)imgOut + 4 * (((__INTPTR_TYPE__)i) * 2160 + 0)) && ((__INTPTR_TYPE__)imgOut + 4 * (((__INTPTR_TYPE__)i) * 2160 + ((__INTPTR_TYPE__)h - 1) + 1) <= (__INTPTR_TYPE__)y2 + 4 * (((__INTPTR_TYPE__)i) * 2160 + 0) || (__INTPTR_TYPE__)y2 + 4 * (((__INTPTR_TYPE__)i) * 2160 + ((__INTPTR_TYPE__)h - 1) + 1) <= (__INTPTR_TYPE__)imgOut + 4 * (((__INTPTR_TYPE__)i) * 2160 + 0)))\\n        for (j=0; j<_PB_H; j++)\\n            imgOut[i][j] = c2*(y1[i][j] + y2[i][j]);\\n\\n#pragma endscop\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int w = W;\\n  int h = H;\\n\\n  /* Variable declaration/allocation. */\\n  DATA_TYPE alpha;\\n  POLYBENCH_2D_ARRAY_DECL(imgIn, DATA_TYPE, W, H, w, h);\\n  POLYBENCH_2D_ARRAY_DECL(imgOut, DATA_TYPE, W, H, w, h);\\n  POLYBENCH_2D_ARRAY_DECL(y1, DATA_TYPE, W, H, w, h);\\n  POLYBENCH_2D_ARRAY_DECL(y2, DATA_TYPE, W, H, w, h);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (w, h, &alpha, POLYBENCH_ARRAY(imgIn), POLYBENCH_ARRAY(imgOut));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_deriche (w, h, alpha, POLYBENCH_ARRAY(imgIn), POLYBENCH_ARRAY(imgOut), POLYBENCH_ARRAY(y1), POLYBENCH_ARRAY(y2));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(w, h, POLYBENCH_ARRAY(imgOut)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(imgIn);\\n  POLYBENCH_FREE_ARRAY(imgOut);\\n  POLYBENCH_FREE_ARRAY(y1);\\n  POLYBENCH_FREE_ARRAY(y2);\\n\\n  return 0;\\n}\\n", "gated/floyd-warshall.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* floyd-warshall.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"floyd-warshall.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(path,N,N,n,n))\\n{\\n  int i, j;\\n\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ((double)(n) * ((10 * (double)(n)) - 34443) >= 4294967296) && n >= 1) map(tofrom: path[0:n][0:2800]) private(j)\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++) {\\n      path[i][j] = i*j%7+1;\\n      if ((i+j)%13 == 0 || (i+j)%7==0 || (i+j)%11 == 0)\\n         path[i][j] = 999;\\n    }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(path,N,N,n,n))\\n\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"path\\");\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++) {\\n      if ((i * n + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, path[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"path\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_floyd_warshall(int n,\\n\\t\\t\\t   DATA_TYPE POLYBENCH_2D(path,N,N,n,n))\\n{\\n  int i, j, k;\\n\\n#pragma scop\\n  for (k = 0; k < _PB_N; k++)\\n    {\\n      for(i = 0; i < _PB_N; i++)\\n\\tfor (j = 0; j < _PB_N; j++)\\n\\t  path[i][j] = path[i][j] < path[i][k] + path[k][j] ?\\n\\t    path[i][j] : path[i][k] + path[k][j];\\n    }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(path, DATA_TYPE, N, N, n, n);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (n, POLYBENCH_ARRAY(path));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_floyd_warshall (n, POLYBENCH_ARRAY(path));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(path)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(path);\\n\\n  return 0;\\n}\\n", "naive/floyd-warshall.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* floyd-warshall.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"floyd-warshall.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(path,N,N,n,n))\\n{\\n  int i, j;\\n\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: n >= 1) map(tofrom: path[0:n][0:2800]) private(j)\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++) {\\n      path[i][j] = i*j%7+1;\\n      if ((i+j)%13 == 0 || (i+j)%7==0 || (i+j)%11 == 0)\\n         path[i][j] = 999;\\n    }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(path,N,N,n,n))\\n\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"path\\");\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++) {\\n      if ((i * n + j) % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, path[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"path\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_floyd_warshall(int n,\\n\\t\\t\\t   DATA_TYPE POLYBENCH_2D(path,N,N,n,n))\\n{\\n  int i, j, k;\\n\\n#pragma scop\\n  for (k = 0; k < _PB_N; k++)\\n    {\\n      for(i = 0; i < _PB_N; i++)\\n\\tfor (j = 0; j < _PB_N; j++)\\n\\t  path[i][j] = path[i][j] < path[i][k] + path[k][j] ?\\n\\t    path[i][j] : path[i][k] + path[k][j];\\n    }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(path, DATA_TYPE, N, N, n, n);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (n, POLYBENCH_ARRAY(path));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_floyd_warshall (n, POLYBENCH_ARRAY(path));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(path)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(path);\\n\\n  return 0;\\n}\\n", "gated/nussinov.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* nussinov.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"nussinov.h\\"\\n\\n/* RNA bases represented as chars, range is [0,3] */\\ntypedef char base;\\n\\n#define match(b1, b2) (((b1)+(b2)) == 3 ? 1 : 0)\\n#define max_score(s1, s2) ((s1 >= s2) ? s1 : s2)\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int n,\\n                 base POLYBENCH_1D(seq,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_2D(table,N,N,n,n))\\n{\\n  int i, j;\\n\\n  //base is AGCT/0..3\\n  for (i=0; i <n; i++) {\\n     seq[i] = (base)((i+1)%4);\\n  }\\n\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ((double)(n) * ((1 * (double)(n)) - 30753) >= 4294967296) && n >= 1) map(tofrom: table[0:n][0:2500]) private(j)\\n  for (i=0; i <n; i++)\\n     for (j=0; j <n; j++)\\n       table[i][j] = 0;\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(table,N,N,n,n))\\n\\n{\\n  int i, j;\\n  int t = 0;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"table\\");\\n  for (i = 0; i < n; i++) {\\n    for (j = i; j < n; j++) {\\n      if (t % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, table[i][j]);\\n      t++;\\n    }\\n  }\\n  POLYBENCH_DUMP_END(\\"table\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\n/*\\n  Original version by Dave Wonnacott at Haverford College <davew@cs.haverford.edu>,\\n  with help from Allison Lake, Ting Zhou, and Tian Jin,\\n  based on algorithm by Nussinov, described in Allison Lake\'s senior thesis.\\n*/\\nstatic\\nvoid kernel_nussinov(int n, base POLYBENCH_1D(seq,N,n),\\n\\t\\t\\t   DATA_TYPE POLYBENCH_2D(table,N,N,n,n))\\n{\\n  int i, j, k;\\n\\n#pragma scop\\n for (i = _PB_N-1; i >= 0; i--) {\\n  for (j=i+1; j<_PB_N; j++) {\\n\\n   if (j-1>=0)\\n      table[i][j] = max_score(table[i][j], table[i][j-1]);\\n   if (i+1<_PB_N)\\n      table[i][j] = max_score(table[i][j], table[i+1][j]);\\n\\n   if (j-1>=0 && i+1<_PB_N) {\\n     /* don\'t allow adjacent elements to bond */\\n     if (i<j-1)\\n        table[i][j] = max_score(table[i][j], table[i+1][j-1]+match(seq[i], seq[j]));\\n     else\\n        table[i][j] = max_score(table[i][j], table[i+1][j-1]);\\n   }\\n\\n   for (k=i+1; k<j; k++) {\\n      table[i][j] = max_score(table[i][j], table[i][k] + table[k+1][j]);\\n   }\\n  }\\n }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_1D_ARRAY_DECL(seq, base, N, n);\\n  POLYBENCH_2D_ARRAY_DECL(table, DATA_TYPE, N, N, n, n);\\n\\n  /* Initialize array(s). */\\n  init_array (n, POLYBENCH_ARRAY(seq), POLYBENCH_ARRAY(table));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_nussinov (n, POLYBENCH_ARRAY(seq), POLYBENCH_ARRAY(table));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(table)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(seq);\\n  POLYBENCH_FREE_ARRAY(table);\\n\\n  return 0;\\n}\\n", "naive/nussinov.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* nussinov.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"nussinov.h\\"\\n\\n/* RNA bases represented as chars, range is [0,3] */\\ntypedef char base;\\n\\n#define match(b1, b2) (((b1)+(b2)) == 3 ? 1 : 0)\\n#define max_score(s1, s2) ((s1 >= s2) ? s1 : s2)\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int n,\\n                 base POLYBENCH_1D(seq,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_2D(table,N,N,n,n))\\n{\\n  int i, j;\\n\\n  //base is AGCT/0..3\\n  #pragma omp target teams distribute parallel for if(target: n >= 1) map(from: seq[0:n])\\n  for (i=0; i <n; i++) {\\n     seq[i] = (base)((i+1)%4);\\n  }\\n\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: n >= 1) map(tofrom: table[0:n][0:2500]) private(j)\\n  for (i=0; i <n; i++)\\n     for (j=0; j <n; j++)\\n       table[i][j] = 0;\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(table,N,N,n,n))\\n\\n{\\n  int i, j;\\n  int t = 0;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"table\\");\\n  for (i = 0; i < n; i++) {\\n    for (j = i; j < n; j++) {\\n      if (t % 20 == 0) fprintf (POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, table[i][j]);\\n      t++;\\n    }\\n  }\\n  POLYBENCH_DUMP_END(\\"table\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\n/*\\n  Original version by Dave Wonnacott at Haverford College <davew@cs.haverford.edu>,\\n  with help from Allison Lake, Ting Zhou, and Tian Jin,\\n  based on algorithm by Nussinov, described in Allison Lake\'s senior thesis.\\n*/\\nstatic\\nvoid kernel_nussinov(int n, base POLYBENCH_1D(seq,N,n),\\n\\t\\t\\t   DATA_TYPE POLYBENCH_2D(table,N,N,n,n))\\n{\\n  int i, j, k;\\n\\n#pragma scop\\n for (i = _PB_N-1; i >= 0; i--) {\\n  for (j=i+1; j<_PB_N; j++) {\\n\\n   if (j-1>=0)\\n      table[i][j] = max_score(table[i][j], table[i][j-1]);\\n   if (i+1<_PB_N)\\n      table[i][j] = max_score(table[i][j], table[i+1][j]);\\n\\n   if (j-1>=0 && i+1<_PB_N) {\\n     /* don\'t allow adjacent elements to bond */\\n     if (i<j-1)\\n        table[i][j] = max_score(table[i][j], table[i+1][j-1]+match(seq[i], seq[j]));\\n     else\\n        table[i][j] = max_score(table[i][j], table[i+1][j-1]);\\n   }\\n\\n   for (k=i+1; k<j; k++) {\\n      table[i][j] = max_score(table[i][j], table[i][k] + table[k+1][j]);\\n   }\\n  }\\n }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_1D_ARRAY_DECL(seq, base, N, n);\\n  POLYBENCH_2D_ARRAY_DECL(table, DATA_TYPE, N, N, n, n);\\n\\n  /* Initialize array(s). */\\n  init_array (n, POLYBENCH_ARRAY(seq), POLYBENCH_ARRAY(table));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_nussinov (n, POLYBENCH_ARRAY(seq), POLYBENCH_ARRAY(table));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(table)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(seq);\\n  POLYBENCH_FREE_ARRAY(table);\\n\\n  return 0;\\n}\\n", "gated/adi.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* adi.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"adi.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(u,N,N,n,n))\\n{\\n  int i, j;\\n\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ((double)(n) * ((4 * (double)(n)) - 24603) >= 4294967296) && n >= 1) map(tofrom: u[0:n][0:1000]) private(j)\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++)\\n      {\\n\\tu[i][j] =  (DATA_TYPE)(i + n-j) / n;\\n      }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(u,N,N,n,n))\\n\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"u\\");\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++) {\\n      if ((i * n + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, u[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"u\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\n/* Based on a Fortran code fragment from Figure 5 of\\n * \\"Automatic Data and Computation Decomposition on Distributed Memory Parallel Computers\\"\\n * by Peizong Lee and Zvi Meir Kedem, TOPLAS, 2002\\n */\\nstatic\\nvoid kernel_adi(int tsteps, int n,\\n\\t\\tDATA_TYPE POLYBENCH_2D(u,N,N,n,n),\\n\\t\\tDATA_TYPE POLYBENCH_2D(v,N,N,n,n),\\n\\t\\tDATA_TYPE POLYBENCH_2D(p,N,N,n,n),\\n\\t\\tDATA_TYPE POLYBENCH_2D(q,N,N,n,n))\\n{\\n  int t, i, j;\\n  DATA_TYPE DX, DY, DT;\\n  DATA_TYPE B1, B2;\\n  DATA_TYPE mul1, mul2;\\n  DATA_TYPE a, b, c, d, e, f;\\n\\n#pragma scop\\n\\n  DX = SCALAR_VAL(1.0)/(DATA_TYPE)_PB_N;\\n  DY = SCALAR_VAL(1.0)/(DATA_TYPE)_PB_N;\\n  DT = SCALAR_VAL(1.0)/(DATA_TYPE)_PB_TSTEPS;\\n  B1 = SCALAR_VAL(2.0);\\n  B2 = SCALAR_VAL(1.0);\\n  mul1 = B1 * DT / (DX * DX);\\n  mul2 = B2 * DT / (DY * DY);\\n\\n  a = -mul1 /  SCALAR_VAL(2.0);\\n  b = SCALAR_VAL(1.0)+mul1;\\n  c = a;\\n  d = -mul2 / SCALAR_VAL(2.0);\\n  e = SCALAR_VAL(1.0)+mul2;\\n  f = d;\\n\\n for (t=1; t<=_PB_TSTEPS; t++) {\\n    //Column Sweep\\n    for (i=1; i<_PB_N-1; i++) {\\n      v[0][i] = SCALAR_VAL(1.0);\\n      p[i][0] = SCALAR_VAL(0.0);\\n      q[i][0] = v[0][i];\\n      for (j=1; j<_PB_N-1; j++) {\\n        p[i][j] = -c / (a*p[i][j-1]+b);\\n        q[i][j] = (-d*u[j][i-1]+(SCALAR_VAL(1.0)+SCALAR_VAL(2.0)*d)*u[j][i] - f*u[j][i+1]-a*q[i][j-1])/(a*p[i][j-1]+b);\\n      }\\n\\n      v[_PB_N-1][i] = SCALAR_VAL(1.0);\\n      for (j=_PB_N-2; j>=1; j--) {\\n        v[j][i] = p[i][j] * v[j+1][i] + q[i][j];\\n      }\\n    }\\n    //Row Sweep\\n    for (i=1; i<_PB_N-1; i++) {\\n      u[i][0] = SCALAR_VAL(1.0);\\n      p[i][0] = SCALAR_VAL(0.0);\\n      q[i][0] = u[i][0];\\n      for (j=1; j<_PB_N-1; j++) {\\n        p[i][j] = -f / (d*p[i][j-1]+e);\\n        q[i][j] = (-a*v[i-1][j]+(SCALAR_VAL(1.0)+SCALAR_VAL(2.0)*a)*v[i][j] - c*v[i+1][j]-d*q[i][j-1])/(d*p[i][j-1]+e);\\n      }\\n      u[i][_PB_N-1] = SCALAR_VAL(1.0);\\n      for (j=_PB_N-2; j>=1; j--) {\\n        u[i][j] = p[i][j] * u[i][j+1] + q[i][j];\\n      }\\n    }\\n  }\\n#pragma endscop\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n  int tsteps = TSTEPS;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(u, DATA_TYPE, N, N, n, n);\\n  POLYBENCH_2D_ARRAY_DECL(v, DATA_TYPE, N, N, n, n);\\n  POLYBENCH_2D_ARRAY_DECL(p, DATA_TYPE, N, N, n, n);\\n  POLYBENCH_2D_ARRAY_DECL(q, DATA_TYPE, N, N, n, n);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (n, POLYBENCH_ARRAY(u));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_adi (tsteps, n, POLYBENCH_ARRAY(u), POLYBENCH_ARRAY(v), POLYBENCH_ARRAY(p), POLYBENCH_ARRAY(q));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(u)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(u);\\n  POLYBENCH_FREE_ARRAY(v);\\n  POLYBENCH_FREE_ARRAY(p);\\n  POLYBENCH_FREE_ARRAY(q);\\n\\n  return 0;\\n}\\n", "naive/adi.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* adi.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"adi.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(u,N,N,n,n))\\n{\\n  int i, j;\\n\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: n >= 1) map(tofrom: u[0:n][0:1000]) private(j)\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++)\\n      {\\n\\tu[i][j] =  (DATA_TYPE)(i + n-j) / n;\\n      }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(u,N,N,n,n))\\n\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"u\\");\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++) {\\n      if ((i * n + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf (POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, u[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"u\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\n/* Based on a Fortran code fragment from Figure 5 of\\n * \\"Automatic Data and Computation Decomposition on Distributed Memory Parallel Computers\\"\\n * by Peizong Lee and Zvi Meir Kedem, TOPLAS, 2002\\n */\\nstatic\\nvoid kernel_adi(int tsteps, int n,\\n\\t\\tDATA_TYPE POLYBENCH_2D(u,N,N,n,n),\\n\\t\\tDATA_TYPE POLYBENCH_2D(v,N,N,n,n),\\n\\t\\tDATA_TYPE POLYBENCH_2D(p,N,N,n,n),\\n\\t\\tDATA_TYPE POLYBENCH_2D(q,N,N,n,n))\\n{\\n  int t, i, j;\\n  DATA_TYPE DX, DY, DT;\\n  DATA_TYPE B1, B2;\\n  DATA_TYPE mul1, mul2;\\n  DATA_TYPE a, b, c, d, e, f;\\n\\n#pragma scop\\n\\n  DX = SCALAR_VAL(1.0)/(DATA_TYPE)_PB_N;\\n  DY = SCALAR_VAL(1.0)/(DATA_TYPE)_PB_N;\\n  DT = SCALAR_VAL(1.0)/(DATA_TYPE)_PB_TSTEPS;\\n  B1 = SCALAR_VAL(2.0);\\n  B2 = SCALAR_VAL(1.0);\\n  mul1 = B1 * DT / (DX * DX);\\n  mul2 = B2 * DT / (DY * DY);\\n\\n  a = -mul1 /  SCALAR_VAL(2.0);\\n  b = SCALAR_VAL(1.0)+mul1;\\n  c = a;\\n  d = -mul2 / SCALAR_VAL(2.0);\\n  e = SCALAR_VAL(1.0)+mul2;\\n  f = d;\\n\\n for (t=1; t<=_PB_TSTEPS; t++) {\\n    //Column Sweep\\n    for (i=1; i<_PB_N-1; i++) {\\n      v[0][i] = SCALAR_VAL(1.0);\\n      p[i][0] = SCALAR_VAL(0.0);\\n      q[i][0] = v[0][i];\\n      for (j=1; j<_PB_N-1; j++) {\\n        p[i][j] = -c / (a*p[i][j-1]+b);\\n        q[i][j] = (-d*u[j][i-1]+(SCALAR_VAL(1.0)+SCALAR_VAL(2.0)*d)*u[j][i] - f*u[j][i+1]-a*q[i][j-1])/(a*p[i][j-1]+b);\\n      }\\n\\n      v[_PB_N-1][i] = SCALAR_VAL(1.0);\\n      for (j=_PB_N-2; j>=1; j--) {\\n        v[j][i] = p[i][j] * v[j+1][i] + q[i][j];\\n      }\\n    }\\n    //Row Sweep\\n    for (i=1; i<_PB_N-1; i++) {\\n      u[i][0] = SCALAR_VAL(1.0);\\n      p[i][0] = SCALAR_VAL(0.0);\\n      q[i][0] = u[i][0];\\n      for (j=1; j<_PB_N-1; j++) {\\n        p[i][j] = -f / (d*p[i][j-1]+e);\\n        q[i][j] = (-a*v[i-1][j]+(SCALAR_VAL(1.0)+SCALAR_VAL(2.0)*a)*v[i][j] - c*v[i+1][j]-d*q[i][j-1])/(d*p[i][j-1]+e);\\n      }\\n      u[i][_PB_N-1] = SCALAR_VAL(1.0);\\n      for (j=_PB_N-2; j>=1; j--) {\\n        u[i][j] = p[i][j] * u[i][j+1] + q[i][j];\\n      }\\n    }\\n  }\\n#pragma endscop\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n  int tsteps = TSTEPS;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(u, DATA_TYPE, N, N, n, n);\\n  POLYBENCH_2D_ARRAY_DECL(v, DATA_TYPE, N, N, n, n);\\n  POLYBENCH_2D_ARRAY_DECL(p, DATA_TYPE, N, N, n, n);\\n  POLYBENCH_2D_ARRAY_DECL(q, DATA_TYPE, N, N, n, n);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (n, POLYBENCH_ARRAY(u));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_adi (tsteps, n, POLYBENCH_ARRAY(u), POLYBENCH_ARRAY(v), POLYBENCH_ARRAY(p), POLYBENCH_ARRAY(q));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(u)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(u);\\n  POLYBENCH_FREE_ARRAY(v);\\n  POLYBENCH_FREE_ARRAY(p);\\n  POLYBENCH_FREE_ARRAY(q);\\n\\n  return 0;\\n}\\n", "gated/fdtd-2d.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* fdtd-2d.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"fdtd-2d.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int tmax,\\n\\t\\t int nx,\\n\\t\\t int ny,\\n\\t\\t DATA_TYPE POLYBENCH_2D(ex,NX,NY,nx,ny),\\n\\t\\t DATA_TYPE POLYBENCH_2D(ey,NX,NY,nx,ny),\\n\\t\\t DATA_TYPE POLYBENCH_2D(hz,NX,NY,nx,ny),\\n\\t\\t DATA_TYPE POLYBENCH_1D(_fict_,TMAX,tmax))\\n{\\n  int i, j;\\n\\n  #pragma omp parallel for if(parallel: tmax >= 524288)\\n  for (i = 0; i < tmax; i++)\\n    _fict_[i] = (DATA_TYPE) i;\\n  for (i = 0; i < nx; i++)\\n    #pragma omp parallel for if(parallel: (ny >= 131072) && ((__INTPTR_TYPE__)ex + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) <= (__INTPTR_TYPE__)ey + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0) || (__INTPTR_TYPE__)ey + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) <= (__INTPTR_TYPE__)ex + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)) && ((__INTPTR_TYPE__)ex + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) <= (__INTPTR_TYPE__)hz + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0) || (__INTPTR_TYPE__)hz + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) <= (__INTPTR_TYPE__)ex + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)) && ((__INTPTR_TYPE__)ey + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) <= (__INTPTR_TYPE__)hz + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0) || (__INTPTR_TYPE__)hz + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) <= (__INTPTR_TYPE__)ey + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)))\\n    for (j = 0; j < ny; j++)\\n      {\\n\\tex[i][j] = ((DATA_TYPE) i*(j+1)) / nx;\\n\\tey[i][j] = ((DATA_TYPE) i*(j+2)) / ny;\\n\\thz[i][j] = ((DATA_TYPE) i*(j+3)) / nx;\\n      }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int nx,\\n\\t\\t int ny,\\n\\t\\t DATA_TYPE POLYBENCH_2D(ex,NX,NY,nx,ny),\\n\\t\\t DATA_TYPE POLYBENCH_2D(ey,NX,NY,nx,ny),\\n\\t\\t DATA_TYPE POLYBENCH_2D(hz,NX,NY,nx,ny))\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"ex\\");\\n  for (i = 0; i < nx; i++)\\n    for (j = 0; j < ny; j++) {\\n      if ((i * nx + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, ex[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"ex\\");\\n  POLYBENCH_DUMP_FINISH;\\n\\n  POLYBENCH_DUMP_BEGIN(\\"ey\\");\\n  for (i = 0; i < nx; i++)\\n    for (j = 0; j < ny; j++) {\\n      if ((i * nx + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, ey[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"ey\\");\\n\\n  POLYBENCH_DUMP_BEGIN(\\"hz\\");\\n  for (i = 0; i < nx; i++)\\n    for (j = 0; j < ny; j++) {\\n      if ((i * nx + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, hz[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"hz\\");\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_fdtd_2d(int tmax,\\n\\t\\t    int nx,\\n\\t\\t    int ny,\\n\\t\\t    DATA_TYPE POLYBENCH_2D(ex,NX,NY,nx,ny),\\n\\t\\t    DATA_TYPE POLYBENCH_2D(ey,NX,NY,nx,ny),\\n\\t\\t    DATA_TYPE POLYBENCH_2D(hz,NX,NY,nx,ny),\\n\\t\\t    DATA_TYPE POLYBENCH_1D(_fict_,TMAX,tmax))\\n{\\n  int t, i, j;\\n\\n#pragma scop\\n\\n  for(t = 0; t < _PB_TMAX; t++)\\n    {\\n      #pragma omp parallel for if(parallel: (ny >= 262144) && ((__INTPTR_TYPE__)ey + 8 * ((0) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) <= (__INTPTR_TYPE__)_fict_ + 8 * (__INTPTR_TYPE__)t || (__INTPTR_TYPE__)_fict_ + 8 * ((__INTPTR_TYPE__)t + 1) <= (__INTPTR_TYPE__)ey + 8 * ((0) * 1200 + 0)))\\n      for (j = 0; j < _PB_NY; j++)\\n\\tey[0][j] = _fict_[t];\\n      for (i = 1; i < _PB_NX; i++)\\n\\t#pragma omp parallel for if(parallel: (ny >= 131072) && ((__INTPTR_TYPE__)ey + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) <= (__INTPTR_TYPE__)hz + (8 * (((__INTPTR_TYPE__)i) * 1200 + 0) < 8 * ((((__INTPTR_TYPE__)i - 1)) * 1200 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1200 + 0) : 8 * ((((__INTPTR_TYPE__)i - 1)) * 1200 + 0)) || (__INTPTR_TYPE__)hz + (8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) > 8 * ((((__INTPTR_TYPE__)i - 1)) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) : 8 * ((((__INTPTR_TYPE__)i - 1)) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1)) <= (__INTPTR_TYPE__)ey + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)))\\n for (j = 0; j < _PB_NY; j++)\\n\\t  ey[i][j] = ey[i][j] - SCALAR_VAL(0.5)*(hz[i][j]-hz[i-1][j]);\\n      for (i = 0; i < _PB_NX; i++)\\n\\t#pragma omp parallel for if(parallel: (ny >= 131072) && ((__INTPTR_TYPE__)ex + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) <= (__INTPTR_TYPE__)hz + (8 * (((__INTPTR_TYPE__)i) * 1200 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1200 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1200 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)) || (__INTPTR_TYPE__)hz + (8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 2) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 2) + 1)) <= (__INTPTR_TYPE__)ex + 8 * (((__INTPTR_TYPE__)i) * 1200 + 1)))\\n for (j = 1; j < _PB_NY; j++)\\n\\t  ex[i][j] = ex[i][j] - SCALAR_VAL(0.5)*(hz[i][j]-hz[i][j-1]);\\n      for (i = 0; i < _PB_NX - 1; i++)\\n\\t#pragma omp parallel for if(parallel: (_PB_NY - 1 >= 131072) && ((__INTPTR_TYPE__)hz + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 2) + 1) <= (__INTPTR_TYPE__)ex + (8 * (((__INTPTR_TYPE__)i) * 1200 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1200 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1200 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)) || (__INTPTR_TYPE__)ex + (8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 2) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 2) + 1)) <= (__INTPTR_TYPE__)hz + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)) && ((__INTPTR_TYPE__)hz + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 2) + 1) <= (__INTPTR_TYPE__)ey + (8 * ((((__INTPTR_TYPE__)i + 1)) * 1200 + 0) < 8 * (((__INTPTR_TYPE__)i) * 1200 + 0) ? 8 * ((((__INTPTR_TYPE__)i + 1)) * 1200 + 0) : 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)) || (__INTPTR_TYPE__)ey + (8 * ((((__INTPTR_TYPE__)i + 1)) * 1200 + ((__INTPTR_TYPE__)ny - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 2) + 1) ? 8 * ((((__INTPTR_TYPE__)i + 1)) * 1200 + ((__INTPTR_TYPE__)ny - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 2) + 1)) <= (__INTPTR_TYPE__)hz + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)))\\n for (j = 0; j < _PB_NY - 1; j++)\\n\\t  hz[i][j] = hz[i][j] - SCALAR_VAL(0.7)*  (ex[i][j+1] - ex[i][j] +\\n\\t\\t\\t\\t       ey[i+1][j] - ey[i][j]);\\n    }\\n\\n#pragma endscop\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int tmax = TMAX;\\n  int nx = NX;\\n  int ny = NY;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(ex,DATA_TYPE,NX,NY,nx,ny);\\n  POLYBENCH_2D_ARRAY_DECL(ey,DATA_TYPE,NX,NY,nx,ny);\\n  POLYBENCH_2D_ARRAY_DECL(hz,DATA_TYPE,NX,NY,nx,ny);\\n  POLYBENCH_1D_ARRAY_DECL(_fict_,DATA_TYPE,TMAX,tmax);\\n\\n  /* Initialize array(s). */\\n  init_array (tmax, nx, ny,\\n\\t      POLYBENCH_ARRAY(ex),\\n\\t      POLYBENCH_ARRAY(ey),\\n\\t      POLYBENCH_ARRAY(hz),\\n\\t      POLYBENCH_ARRAY(_fict_));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_fdtd_2d (tmax, nx, ny,\\n\\t\\t  POLYBENCH_ARRAY(ex),\\n\\t\\t  POLYBENCH_ARRAY(ey),\\n\\t\\t  POLYBENCH_ARRAY(hz),\\n\\t\\t  POLYBENCH_ARRAY(_fict_));\\n\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(nx, ny, POLYBENCH_ARRAY(ex),\\n\\t\\t\\t\\t    POLYBENCH_ARRAY(ey),\\n\\t\\t\\t\\t    POLYBENCH_ARRAY(hz)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(ex);\\n  POLYBENCH_FREE_ARRAY(ey);\\n  POLYBENCH_FREE_ARRAY(hz);\\n  POLYBENCH_FREE_ARRAY(_fict_);\\n\\n  return 0;\\n}\\n", "naive/fdtd-2d.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* fdtd-2d.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"fdtd-2d.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int tmax,\\n\\t\\t int nx,\\n\\t\\t int ny,\\n\\t\\t DATA_TYPE POLYBENCH_2D(ex,NX,NY,nx,ny),\\n\\t\\t DATA_TYPE POLYBENCH_2D(ey,NX,NY,nx,ny),\\n\\t\\t DATA_TYPE POLYBENCH_2D(hz,NX,NY,nx,ny),\\n\\t\\t DATA_TYPE POLYBENCH_1D(_fict_,TMAX,tmax))\\n{\\n  int i, j;\\n\\n  #pragma omp target teams distribute parallel for if(target: tmax >= 1) map(from: _fict_[0:tmax])\\n  for (i = 0; i < tmax; i++)\\n    _fict_[i] = (DATA_TYPE) i;\\n  for (i = 0; i < nx; i++)\\n    #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)ex + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) <= (__INTPTR_TYPE__)ey + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0) || (__INTPTR_TYPE__)ey + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) <= (__INTPTR_TYPE__)ex + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)) && ((__INTPTR_TYPE__)ex + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) <= (__INTPTR_TYPE__)hz + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0) || (__INTPTR_TYPE__)hz + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) <= (__INTPTR_TYPE__)ex + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)) && ((__INTPTR_TYPE__)ey + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) <= (__INTPTR_TYPE__)hz + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0) || (__INTPTR_TYPE__)hz + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) <= (__INTPTR_TYPE__)ey + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)))\\n    for (j = 0; j < ny; j++)\\n      {\\n\\tex[i][j] = ((DATA_TYPE) i*(j+1)) / nx;\\n\\tey[i][j] = ((DATA_TYPE) i*(j+2)) / ny;\\n\\thz[i][j] = ((DATA_TYPE) i*(j+3)) / nx;\\n      }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int nx,\\n\\t\\t int ny,\\n\\t\\t DATA_TYPE POLYBENCH_2D(ex,NX,NY,nx,ny),\\n\\t\\t DATA_TYPE POLYBENCH_2D(ey,NX,NY,nx,ny),\\n\\t\\t DATA_TYPE POLYBENCH_2D(hz,NX,NY,nx,ny))\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"ex\\");\\n  for (i = 0; i < nx; i++)\\n    for (j = 0; j < ny; j++) {\\n      if ((i * nx + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, ex[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"ex\\");\\n  POLYBENCH_DUMP_FINISH;\\n\\n  POLYBENCH_DUMP_BEGIN(\\"ey\\");\\n  for (i = 0; i < nx; i++)\\n    for (j = 0; j < ny; j++) {\\n      if ((i * nx + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, ey[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"ey\\");\\n\\n  POLYBENCH_DUMP_BEGIN(\\"hz\\");\\n  for (i = 0; i < nx; i++)\\n    for (j = 0; j < ny; j++) {\\n      if ((i * nx + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, hz[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"hz\\");\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_fdtd_2d(int tmax,\\n\\t\\t    int nx,\\n\\t\\t    int ny,\\n\\t\\t    DATA_TYPE POLYBENCH_2D(ex,NX,NY,nx,ny),\\n\\t\\t    DATA_TYPE POLYBENCH_2D(ey,NX,NY,nx,ny),\\n\\t\\t    DATA_TYPE POLYBENCH_2D(hz,NX,NY,nx,ny),\\n\\t\\t    DATA_TYPE POLYBENCH_1D(_fict_,TMAX,tmax))\\n{\\n  int t, i, j;\\n\\n#pragma scop\\n\\n  for(t = 0; t < _PB_TMAX; t++)\\n    {\\n      #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)ey + 8 * ((0) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) <= (__INTPTR_TYPE__)_fict_ + 8 * (__INTPTR_TYPE__)t || (__INTPTR_TYPE__)_fict_ + 8 * ((__INTPTR_TYPE__)t + 1) <= (__INTPTR_TYPE__)ey + 8 * ((0) * 1200 + 0)))\\n      for (j = 0; j < _PB_NY; j++)\\n\\tey[0][j] = _fict_[t];\\n      for (i = 1; i < _PB_NX; i++)\\n\\t#pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)ey + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) <= (__INTPTR_TYPE__)hz + (8 * (((__INTPTR_TYPE__)i) * 1200 + 0) < 8 * ((((__INTPTR_TYPE__)i - 1)) * 1200 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1200 + 0) : 8 * ((((__INTPTR_TYPE__)i - 1)) * 1200 + 0)) || (__INTPTR_TYPE__)hz + (8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) > 8 * ((((__INTPTR_TYPE__)i - 1)) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) : 8 * ((((__INTPTR_TYPE__)i - 1)) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1)) <= (__INTPTR_TYPE__)ey + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)))\\n for (j = 0; j < _PB_NY; j++)\\n\\t  ey[i][j] = ey[i][j] - SCALAR_VAL(0.5)*(hz[i][j]-hz[i-1][j]);\\n      for (i = 0; i < _PB_NX; i++)\\n\\t#pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)ex + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) <= (__INTPTR_TYPE__)hz + (8 * (((__INTPTR_TYPE__)i) * 1200 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1200 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1200 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)) || (__INTPTR_TYPE__)hz + (8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 2) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 2) + 1)) <= (__INTPTR_TYPE__)ex + 8 * (((__INTPTR_TYPE__)i) * 1200 + 1)))\\n for (j = 1; j < _PB_NY; j++)\\n\\t  ex[i][j] = ex[i][j] - SCALAR_VAL(0.5)*(hz[i][j]-hz[i][j-1]);\\n      for (i = 0; i < _PB_NX - 1; i++)\\n\\t#pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)hz + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 2) + 1) <= (__INTPTR_TYPE__)ex + (8 * (((__INTPTR_TYPE__)i) * 1200 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1200 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1200 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)) || (__INTPTR_TYPE__)ex + (8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 2) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 1) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 2) + 1)) <= (__INTPTR_TYPE__)hz + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)) && ((__INTPTR_TYPE__)hz + 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 2) + 1) <= (__INTPTR_TYPE__)ey + (8 * ((((__INTPTR_TYPE__)i + 1)) * 1200 + 0) < 8 * (((__INTPTR_TYPE__)i) * 1200 + 0) ? 8 * ((((__INTPTR_TYPE__)i + 1)) * 1200 + 0) : 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)) || (__INTPTR_TYPE__)ey + (8 * ((((__INTPTR_TYPE__)i + 1)) * 1200 + ((__INTPTR_TYPE__)ny - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 2) + 1) ? 8 * ((((__INTPTR_TYPE__)i + 1)) * 1200 + ((__INTPTR_TYPE__)ny - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1200 + ((__INTPTR_TYPE__)ny - 2) + 1)) <= (__INTPTR_TYPE__)hz + 8 * (((__INTPTR_TYPE__)i) * 1200 + 0)))\\n for (j = 0; j < _PB_NY - 1; j++)\\n\\t  hz[i][j] = hz[i][j] - SCALAR_VAL(0.7)*  (ex[i][j+1] - ex[i][j] +\\n\\t\\t\\t\\t       ey[i+1][j] - ey[i][j]);\\n    }\\n\\n#pragma endscop\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int tmax = TMAX;\\n  int nx = NX;\\n  int ny = NY;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(ex,DATA_TYPE,NX,NY,nx,ny);\\n  POLYBENCH_2D_ARRAY_DECL(ey,DATA_TYPE,NX,NY,nx,ny);\\n  POLYBENCH_2D_ARRAY_DECL(hz,DATA_TYPE,NX,NY,nx,ny);\\n  POLYBENCH_1D_ARRAY_DECL(_fict_,DATA_TYPE,TMAX,tmax);\\n\\n  /* Initialize array(s). */\\n  init_array (tmax, nx, ny,\\n\\t      POLYBENCH_ARRAY(ex),\\n\\t      POLYBENCH_ARRAY(ey),\\n\\t      POLYBENCH_ARRAY(hz),\\n\\t      POLYBENCH_ARRAY(_fict_));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_fdtd_2d (tmax, nx, ny,\\n\\t\\t  POLYBENCH_ARRAY(ex),\\n\\t\\t  POLYBENCH_ARRAY(ey),\\n\\t\\t  POLYBENCH_ARRAY(hz),\\n\\t\\t  POLYBENCH_ARRAY(_fict_));\\n\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(nx, ny, POLYBENCH_ARRAY(ex),\\n\\t\\t\\t\\t    POLYBENCH_ARRAY(ey),\\n\\t\\t\\t\\t    POLYBENCH_ARRAY(hz)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(ex);\\n  POLYBENCH_FREE_ARRAY(ey);\\n  POLYBENCH_FREE_ARRAY(hz);\\n  POLYBENCH_FREE_ARRAY(_fict_);\\n\\n  return 0;\\n}\\n", "gated/heat-3d.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* heat-3d.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"heat-3d.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int n,\\n\\t\\t DATA_TYPE POLYBENCH_3D(A,N,N,N,n,n,n),\\n\\t\\t DATA_TYPE POLYBENCH_3D(B,N,N,N,n,n,n))\\n{\\n  int i, j, k;\\n\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++)\\n      #pragma omp parallel for if(parallel: (n >= 1048576) && ((__INTPTR_TYPE__)A + 8 * ((((__INTPTR_TYPE__)i) * 120 + (__INTPTR_TYPE__)j) * 120 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)B + 8 * ((((__INTPTR_TYPE__)i) * 120 + (__INTPTR_TYPE__)j) * 120 + 0) || (__INTPTR_TYPE__)B + 8 * ((((__INTPTR_TYPE__)i) * 120 + (__INTPTR_TYPE__)j) * 120 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)A + 8 * ((((__INTPTR_TYPE__)i) * 120 + (__INTPTR_TYPE__)j) * 120 + 0)))\\n      for (k = 0; k < n; k++)\\n        A[i][j][k] = B[i][j][k] = (DATA_TYPE) (i + j + (n-k))* 10 / (n);\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_3D(A,N,N,N,n,n,n))\\n\\n{\\n  int i, j, k;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"A\\");\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++)\\n      for (k = 0; k < n; k++) {\\n         if ((i * n * n + j * n + k) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n         fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j][k]);\\n      }\\n  POLYBENCH_DUMP_END(\\"A\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_heat_3d(int tsteps,\\n\\t\\t      int n,\\n\\t\\t      DATA_TYPE POLYBENCH_3D(A,N,N,N,n,n,n),\\n\\t\\t      DATA_TYPE POLYBENCH_3D(B,N,N,N,n,n,n))\\n{\\n  int t, i, j, k;\\n\\n#pragma scop\\n    for (t = 1; t <= TSTEPS; t++) {\\n        for (i = 1; i < _PB_N-1; i++) {\\n            for (j = 1; j < _PB_N-1; j++) {\\n                for (k = 1; k < _PB_N-1; k++) {\\n                    B[i][j][k] =   SCALAR_VAL(0.125) * (A[i+1][j][k] - SCALAR_VAL(2.0) * A[i][j][k] + A[i-1][j][k])\\n                                 + SCALAR_VAL(0.125) * (A[i][j+1][k] - SCALAR_VAL(2.0) * A[i][j][k] + A[i][j-1][k])\\n                                 + SCALAR_VAL(0.125) * (A[i][j][k+1] - SCALAR_VAL(2.0) * A[i][j][k] + A[i][j][k-1])\\n                                 + A[i][j][k];\\n                }\\n            }\\n        }\\n        for (i = 1; i < _PB_N-1; i++) {\\n           for (j = 1; j < _PB_N-1; j++) {\\n               for (k = 1; k < _PB_N-1; k++) {\\n                   A[i][j][k] =   SCALAR_VAL(0.125) * (B[i+1][j][k] - SCALAR_VAL(2.0) * B[i][j][k] + B[i-1][j][k])\\n                                + SCALAR_VAL(0.125) * (B[i][j+1][k] - SCALAR_VAL(2.0) * B[i][j][k] + B[i][j-1][k])\\n                                + SCALAR_VAL(0.125) * (B[i][j][k+1] - SCALAR_VAL(2.0) * B[i][j][k] + B[i][j][k-1])\\n                                + B[i][j][k];\\n               }\\n           }\\n       }\\n    }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n  int tsteps = TSTEPS;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_3D_ARRAY_DECL(A, DATA_TYPE, N, N, N, n, n, n);\\n  POLYBENCH_3D_ARRAY_DECL(B, DATA_TYPE, N, N, N, n, n, n);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (n, POLYBENCH_ARRAY(A), POLYBENCH_ARRAY(B));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_heat_3d (tsteps, n, POLYBENCH_ARRAY(A), POLYBENCH_ARRAY(B));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(A)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n\\n  return 0;\\n}\\n", "naive/heat-3d.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* heat-3d.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"heat-3d.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int n,\\n\\t\\t DATA_TYPE POLYBENCH_3D(A,N,N,N,n,n,n),\\n\\t\\t DATA_TYPE POLYBENCH_3D(B,N,N,N,n,n,n))\\n{\\n  int i, j, k;\\n\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++)\\n      #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)A + 8 * ((((__INTPTR_TYPE__)i) * 120 + (__INTPTR_TYPE__)j) * 120 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)B + 8 * ((((__INTPTR_TYPE__)i) * 120 + (__INTPTR_TYPE__)j) * 120 + 0) || (__INTPTR_TYPE__)B + 8 * ((((__INTPTR_TYPE__)i) * 120 + (__INTPTR_TYPE__)j) * 120 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)A + 8 * ((((__INTPTR_TYPE__)i) * 120 + (__INTPTR_TYPE__)j) * 120 + 0)))\\n      for (k = 0; k < n; k++)\\n        A[i][j][k] = B[i][j][k] = (DATA_TYPE) (i + j + (n-k))* 10 / (n);\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_3D(A,N,N,N,n,n,n))\\n\\n{\\n  int i, j, k;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"A\\");\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++)\\n      for (k = 0; k < n; k++) {\\n         if ((i * n * n + j * n + k) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n         fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j][k]);\\n      }\\n  POLYBENCH_DUMP_END(\\"A\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_heat_3d(int tsteps,\\n\\t\\t      int n,\\n\\t\\t      DATA_TYPE POLYBENCH_3D(A,N,N,N,n,n,n),\\n\\t\\t      DATA_TYPE POLYBENCH_3D(B,N,N,N,n,n,n))\\n{\\n  int t, i, j, k;\\n\\n#pragma scop\\n    for (t = 1; t <= TSTEPS; t++) {\\n        for (i = 1; i < _PB_N-1; i++) {\\n            for (j = 1; j < _PB_N-1; j++) {\\n                for (k = 1; k < _PB_N-1; k++) {\\n                    B[i][j][k] =   SCALAR_VAL(0.125) * (A[i+1][j][k] - SCALAR_VAL(2.0) * A[i][j][k] + A[i-1][j][k])\\n                                 + SCALAR_VAL(0.125) * (A[i][j+1][k] - SCALAR_VAL(2.0) * A[i][j][k] + A[i][j-1][k])\\n                                 + SCALAR_VAL(0.125) * (A[i][j][k+1] - SCALAR_VAL(2.0) * A[i][j][k] + A[i][j][k-1])\\n                                 + A[i][j][k];\\n                }\\n            }\\n        }\\n        for (i = 1; i < _PB_N-1; i++) {\\n           for (j = 1; j < _PB_N-1; j++) {\\n               for (k = 1; k < _PB_N-1; k++) {\\n                   A[i][j][k] =   SCALAR_VAL(0.125) * (B[i+1][j][k] - SCALAR_VAL(2.0) * B[i][j][k] + B[i-1][j][k])\\n                                + SCALAR_VAL(0.125) * (B[i][j+1][k] - SCALAR_VAL(2.0) * B[i][j][k] + B[i][j-1][k])\\n                                + SCALAR_VAL(0.125) * (B[i][j][k+1] - SCALAR_VAL(2.0) * B[i][j][k] + B[i][j][k-1])\\n                                + B[i][j][k];\\n               }\\n           }\\n       }\\n    }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n  int tsteps = TSTEPS;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_3D_ARRAY_DECL(A, DATA_TYPE, N, N, N, n, n, n);\\n  POLYBENCH_3D_ARRAY_DECL(B, DATA_TYPE, N, N, N, n, n, n);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (n, POLYBENCH_ARRAY(A), POLYBENCH_ARRAY(B));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_heat_3d (tsteps, n, POLYBENCH_ARRAY(A), POLYBENCH_ARRAY(B));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(A)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n\\n  return 0;\\n}\\n", "gated/jacobi-1d.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* jacobi-1d.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"jacobi-1d.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int n,\\n\\t\\t DATA_TYPE POLYBENCH_1D(A,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(B,N,n))\\n{\\n  int i;\\n\\n  #pragma omp parallel for if(parallel: (n >= 524288) && ((__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)B + 0 || (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)A + 0))\\n  for (i = 0; i < n; i++)\\n      {\\n\\tA[i] = ((DATA_TYPE) i+ 2) / n;\\n\\tB[i] = ((DATA_TYPE) i+ 3) / n;\\n      }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_1D(A,N,n))\\n\\n{\\n  int i;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"A\\");\\n  for (i = 0; i < n; i++)\\n    {\\n      if (i % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i]);\\n    }\\n  POLYBENCH_DUMP_END(\\"A\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_jacobi_1d(int tsteps,\\n\\t\\t\\t    int n,\\n\\t\\t\\t    DATA_TYPE POLYBENCH_1D(A,N,n),\\n\\t\\t\\t    DATA_TYPE POLYBENCH_1D(B,N,n))\\n{\\n  int t, i;\\n\\n#pragma scop\\n  for (t = 0; t < _PB_TSTEPS; t++)\\n    {\\n      #pragma omp parallel for if(parallel: (_PB_N - 1 >= 524288) && ((__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)n - 2) + 1) <= (__INTPTR_TYPE__)A + 0 || (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)B + 8))\\n      for (i = 1; i < _PB_N - 1; i++)\\n\\tB[i] = 0.33333 * (A[i-1] + A[i] + A[i + 1]);\\n      #pragma omp parallel for if(parallel: (_PB_N - 1 >= 524288) && ((__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)n - 2) + 1) <= (__INTPTR_TYPE__)B + 0 || (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)A + 8))\\n      for (i = 1; i < _PB_N - 1; i++)\\n\\tA[i] = 0.33333 * (B[i-1] + B[i] + B[i + 1]);\\n    }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n  int tsteps = TSTEPS;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_1D_ARRAY_DECL(A, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(B, DATA_TYPE, N, n);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (n, POLYBENCH_ARRAY(A), POLYBENCH_ARRAY(B));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_jacobi_1d(tsteps, n, POLYBENCH_ARRAY(A), POLYBENCH_ARRAY(B));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(A)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(B);\\n\\n  return 0;\\n}\\n", "naive/jacobi-1d.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* jacobi-1d.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"jacobi-1d.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int n,\\n\\t\\t DATA_TYPE POLYBENCH_1D(A,N,n),\\n\\t\\t DATA_TYPE POLYBENCH_1D(B,N,n))\\n{\\n  int i;\\n\\n  #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)B + 0 || (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)A + 0))\\n  for (i = 0; i < n; i++)\\n      {\\n\\tA[i] = ((DATA_TYPE) i+ 2) / n;\\n\\tB[i] = ((DATA_TYPE) i+ 3) / n;\\n      }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_1D(A,N,n))\\n\\n{\\n  int i;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"A\\");\\n  for (i = 0; i < n; i++)\\n    {\\n      if (i % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i]);\\n    }\\n  POLYBENCH_DUMP_END(\\"A\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_jacobi_1d(int tsteps,\\n\\t\\t\\t    int n,\\n\\t\\t\\t    DATA_TYPE POLYBENCH_1D(A,N,n),\\n\\t\\t\\t    DATA_TYPE POLYBENCH_1D(B,N,n))\\n{\\n  int t, i;\\n\\n#pragma scop\\n  for (t = 0; t < _PB_TSTEPS; t++)\\n    {\\n      #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)n - 2) + 1) <= (__INTPTR_TYPE__)A + 0 || (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)B + 8))\\n      for (i = 1; i < _PB_N - 1; i++)\\n\\tB[i] = 0.33333 * (A[i-1] + A[i] + A[i + 1]);\\n      #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)n - 2) + 1) <= (__INTPTR_TYPE__)B + 0 || (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)A + 8))\\n      for (i = 1; i < _PB_N - 1; i++)\\n\\tA[i] = 0.33333 * (B[i-1] + B[i] + B[i + 1]);\\n    }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n  int tsteps = TSTEPS;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_1D_ARRAY_DECL(A, DATA_TYPE, N, n);\\n  POLYBENCH_1D_ARRAY_DECL(B, DATA_TYPE, N, n);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (n, POLYBENCH_ARRAY(A), POLYBENCH_ARRAY(B));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_jacobi_1d(tsteps, n, POLYBENCH_ARRAY(A), POLYBENCH_ARRAY(B));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(A)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(B);\\n\\n  return 0;\\n}\\n", "gated/jacobi-2d.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* jacobi-2d.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"jacobi-2d.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,N,N,n,n),\\n\\t\\t DATA_TYPE POLYBENCH_2D(B,N,N,n,n))\\n{\\n  int i, j;\\n\\n  for (i = 0; i < n; i++)\\n    #pragma omp parallel for if(parallel: (n >= 524288) && ((__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) || (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)))\\n    for (j = 0; j < n; j++)\\n      {\\n\\tA[i][j] = ((DATA_TYPE) i*(j+2) + 2) / n;\\n\\tB[i][j] = ((DATA_TYPE) i*(j+3) + 3) / n;\\n      }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,N,N,n,n))\\n\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"A\\");\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++) {\\n      if ((i * n + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"A\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_jacobi_2d(int tsteps,\\n\\t\\t\\t    int n,\\n\\t\\t\\t    DATA_TYPE POLYBENCH_2D(A,N,N,n,n),\\n\\t\\t\\t    DATA_TYPE POLYBENCH_2D(B,N,N,n,n))\\n{\\n  int t, i, j;\\n\\n#pragma scop\\n  for (t = 0; t < _PB_TSTEPS; t++)\\n    {\\n      for (i = 1; i < _PB_N - 1; i++)\\n\\t#pragma omp parallel for if(parallel: (_PB_N - 1 >= 524288) && ((__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) <= (__INTPTR_TYPE__)A + ((((8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 2) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 2)) < 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + 1) ? ((8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 2) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 2)) : 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + 1)) < 8 * ((((__INTPTR_TYPE__)i - 1)) * 1300 + 1) ? (((8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 2) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 2)) < 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + 1) ? ((8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 2) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 2)) : 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + 1)) : 8 * ((((__INTPTR_TYPE__)i - 1)) * 1300 + 1)) || (__INTPTR_TYPE__)A + ((((8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1)) > 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) ? ((8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1)) : 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1)) > 8 * ((((__INTPTR_TYPE__)i - 1)) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) ? (((8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1)) > 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) ? ((8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1)) : 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1)) : 8 * ((((__INTPTR_TYPE__)i - 1)) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1)) <= (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)i) * 1300 + 1)))\\n for (j = 1; j < _PB_N - 1; j++)\\n\\t  B[i][j] = SCALAR_VAL(0.2) * (A[i][j] + A[i][j-1] + A[i][1+j] + A[1+i][j] + A[i-1][j]);\\n      for (i = 1; i < _PB_N - 1; i++)\\n\\t#pragma omp parallel for if(parallel: (_PB_N - 1 >= 524288) && ((__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) <= (__INTPTR_TYPE__)B + ((((8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 2) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 2)) < 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + 1) ? ((8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 2) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 2)) : 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + 1)) < 8 * ((((__INTPTR_TYPE__)i - 1)) * 1300 + 1) ? (((8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 2) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 2)) < 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + 1) ? ((8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 2) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 2)) : 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + 1)) : 8 * ((((__INTPTR_TYPE__)i - 1)) * 1300 + 1)) || (__INTPTR_TYPE__)B + ((((8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1)) > 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) ? ((8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1)) : 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1)) > 8 * ((((__INTPTR_TYPE__)i - 1)) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) ? (((8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1)) > 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) ? ((8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1)) : 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1)) : 8 * ((((__INTPTR_TYPE__)i - 1)) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1)) <= (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1300 + 1)))\\n for (j = 1; j < _PB_N - 1; j++)\\n\\t  A[i][j] = SCALAR_VAL(0.2) * (B[i][j] + B[i][j-1] + B[i][1+j] + B[1+i][j] + B[i-1][j]);\\n    }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n  int tsteps = TSTEPS;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(A, DATA_TYPE, N, N, n, n);\\n  POLYBENCH_2D_ARRAY_DECL(B, DATA_TYPE, N, N, n, n);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (n, POLYBENCH_ARRAY(A), POLYBENCH_ARRAY(B));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_jacobi_2d(tsteps, n, POLYBENCH_ARRAY(A), POLYBENCH_ARRAY(B));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(A)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(B);\\n\\n  return 0;\\n}\\n", "naive/jacobi-2d.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* jacobi-2d.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"jacobi-2d.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,N,N,n,n),\\n\\t\\t DATA_TYPE POLYBENCH_2D(B,N,N,n,n))\\n{\\n  int i, j;\\n\\n  for (i = 0; i < n; i++)\\n    #pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) || (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1) <= (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)))\\n    for (j = 0; j < n; j++)\\n      {\\n\\tA[i][j] = ((DATA_TYPE) i*(j+2) + 2) / n;\\n\\tB[i][j] = ((DATA_TYPE) i*(j+3) + 3) / n;\\n      }\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,N,N,n,n))\\n\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"A\\");\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++) {\\n      if ((i * n + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"A\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_jacobi_2d(int tsteps,\\n\\t\\t\\t    int n,\\n\\t\\t\\t    DATA_TYPE POLYBENCH_2D(A,N,N,n,n),\\n\\t\\t\\t    DATA_TYPE POLYBENCH_2D(B,N,N,n,n))\\n{\\n  int t, i, j;\\n\\n#pragma scop\\n  for (t = 0; t < _PB_TSTEPS; t++)\\n    {\\n      for (i = 1; i < _PB_N - 1; i++)\\n\\t#pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) <= (__INTPTR_TYPE__)A + ((((8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 2) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 2)) < 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + 1) ? ((8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 2) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 2)) : 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + 1)) < 8 * ((((__INTPTR_TYPE__)i - 1)) * 1300 + 1) ? (((8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 2) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 2)) < 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + 1) ? ((8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 2) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 2)) : 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + 1)) : 8 * ((((__INTPTR_TYPE__)i - 1)) * 1300 + 1)) || (__INTPTR_TYPE__)A + ((((8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1)) > 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) ? ((8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1)) : 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1)) > 8 * ((((__INTPTR_TYPE__)i - 1)) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) ? (((8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1)) > 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) ? ((8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1)) : 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1)) : 8 * ((((__INTPTR_TYPE__)i - 1)) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1)) <= (__INTPTR_TYPE__)B + 8 * (((__INTPTR_TYPE__)i) * 1300 + 1)))\\n for (j = 1; j < _PB_N - 1; j++)\\n\\t  B[i][j] = SCALAR_VAL(0.2) * (A[i][j] + A[i][j-1] + A[i][1+j] + A[1+i][j] + A[i-1][j]);\\n      for (i = 1; i < _PB_N - 1; i++)\\n\\t#pragma omp parallel for if(parallel: ((__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) <= (__INTPTR_TYPE__)B + ((((8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 2) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 2)) < 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + 1) ? ((8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 2) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 2)) : 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + 1)) < 8 * ((((__INTPTR_TYPE__)i - 1)) * 1300 + 1) ? (((8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 2) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 2)) < 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + 1) ? ((8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 2) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + 1) < 8 * (((__INTPTR_TYPE__)i) * 1300 + 0) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 0)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + 2)) : 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + 1)) : 8 * ((((__INTPTR_TYPE__)i - 1)) * 1300 + 1)) || (__INTPTR_TYPE__)B + ((((8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1)) > 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) ? ((8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1)) : 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1)) > 8 * ((((__INTPTR_TYPE__)i - 1)) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) ? (((8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1)) > 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) ? ((8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1) ? (8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) > 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1) ? 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 3) + 1)) : 8 * (((__INTPTR_TYPE__)i) * 1300 + ((__INTPTR_TYPE__)n - 1) + 1)) : 8 * ((((__INTPTR_TYPE__)i + 1)) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1)) : 8 * ((((__INTPTR_TYPE__)i - 1)) * 1300 + ((__INTPTR_TYPE__)n - 2) + 1)) <= (__INTPTR_TYPE__)A + 8 * (((__INTPTR_TYPE__)i) * 1300 + 1)))\\n for (j = 1; j < _PB_N - 1; j++)\\n\\t  A[i][j] = SCALAR_VAL(0.2) * (B[i][j] + B[i][j-1] + B[i][1+j] + B[1+i][j] + B[i-1][j]);\\n    }\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n  int tsteps = TSTEPS;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(A, DATA_TYPE, N, N, n, n);\\n  POLYBENCH_2D_ARRAY_DECL(B, DATA_TYPE, N, N, n, n);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (n, POLYBENCH_ARRAY(A), POLYBENCH_ARRAY(B));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_jacobi_2d(tsteps, n, POLYBENCH_ARRAY(A), POLYBENCH_ARRAY(B));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(A)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n  POLYBENCH_FREE_ARRAY(B);\\n\\n  return 0;\\n}\\n", "gated/seidel-2d.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* seidel-2d.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"seidel-2d.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,N,N,n,n))\\n{\\n  int i, j;\\n\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: ((double)(n) * ((5 * (double)(n)) - 49205) >= 4294967296) && n >= 1) map(tofrom: A[0:n][0:2000]) private(j)\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++)\\n      A[i][j] = ((DATA_TYPE) i*(j+2) + 2) / n;\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,N,N,n,n))\\n\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"A\\");\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++) {\\n      if ((i * n + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"A\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_seidel_2d(int tsteps,\\n\\t\\t      int n,\\n\\t\\t      DATA_TYPE POLYBENCH_2D(A,N,N,n,n))\\n{\\n  int t, i, j;\\n\\n#pragma scop\\n  for (t = 0; t <= _PB_TSTEPS - 1; t++)\\n    for (i = 1; i<= _PB_N - 2; i++)\\n      for (j = 1; j <= _PB_N - 2; j++)\\n\\tA[i][j] = (A[i-1][j-1] + A[i-1][j] + A[i-1][j+1]\\n\\t\\t   + A[i][j-1] + A[i][j] + A[i][j+1]\\n\\t\\t   + A[i+1][j-1] + A[i+1][j] + A[i+1][j+1])/SCALAR_VAL(9.0);\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n  int tsteps = TSTEPS;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(A, DATA_TYPE, N, N, n, n);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (n, POLYBENCH_ARRAY(A));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_seidel_2d (tsteps, n, POLYBENCH_ARRAY(A));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(A)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n\\n  return 0;\\n}\\n", "naive/seidel-2d.c": "/**\\n * This version is stamped on May 10, 2016\\n *\\n * Contact:\\n *   Louis-Noel Pouchet <pouchet.ohio-state.edu>\\n *   Tomofumi Yuki <tomofumi.yuki.fr>\\n *\\n * Web address: http://polybench.sourceforge.net\\n */\\n/* seidel-2d.c: this file is part of PolyBench/C */\\n\\n#include <stdio.h>\\n#include <unistd.h>\\n#include <string.h>\\n#include <math.h>\\n\\n/* Include polybench common header. */\\n#include <polybench.h>\\n\\n/* Include benchmark-specific header. */\\n#include \\"seidel-2d.h\\"\\n\\n\\n/* Array initialization. */\\nstatic\\nvoid init_array (int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,N,N,n,n))\\n{\\n  int i, j;\\n\\n  #pragma omp target teams distribute parallel for collapse(2) if(target: n >= 1) map(tofrom: A[0:n][0:2000]) private(j)\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++)\\n      A[i][j] = ((DATA_TYPE) i*(j+2) + 2) / n;\\n}\\n\\n\\n/* DCE code. Must scan the entire live-out data.\\n   Can be used also to check the correctness of the output. */\\nstatic\\nvoid print_array(int n,\\n\\t\\t DATA_TYPE POLYBENCH_2D(A,N,N,n,n))\\n\\n{\\n  int i, j;\\n\\n  POLYBENCH_DUMP_START;\\n  POLYBENCH_DUMP_BEGIN(\\"A\\");\\n  for (i = 0; i < n; i++)\\n    for (j = 0; j < n; j++) {\\n      if ((i * n + j) % 20 == 0) fprintf(POLYBENCH_DUMP_TARGET, \\"\\\\n\\");\\n      fprintf(POLYBENCH_DUMP_TARGET, DATA_PRINTF_MODIFIER, A[i][j]);\\n    }\\n  POLYBENCH_DUMP_END(\\"A\\");\\n  POLYBENCH_DUMP_FINISH;\\n}\\n\\n\\n/* Main computational kernel. The whole function will be timed,\\n   including the call and return. */\\nstatic\\nvoid kernel_seidel_2d(int tsteps,\\n\\t\\t      int n,\\n\\t\\t      DATA_TYPE POLYBENCH_2D(A,N,N,n,n))\\n{\\n  int t, i, j;\\n\\n#pragma scop\\n  for (t = 0; t <= _PB_TSTEPS - 1; t++)\\n    for (i = 1; i<= _PB_N - 2; i++)\\n      for (j = 1; j <= _PB_N - 2; j++)\\n\\tA[i][j] = (A[i-1][j-1] + A[i-1][j] + A[i-1][j+1]\\n\\t\\t   + A[i][j-1] + A[i][j] + A[i][j+1]\\n\\t\\t   + A[i+1][j-1] + A[i+1][j] + A[i+1][j+1])/SCALAR_VAL(9.0);\\n#pragma endscop\\n\\n}\\n\\n\\nint main(int argc, char** argv)\\n{\\n  /* Retrieve problem size. */\\n  int n = N;\\n  int tsteps = TSTEPS;\\n\\n  /* Variable declaration/allocation. */\\n  POLYBENCH_2D_ARRAY_DECL(A, DATA_TYPE, N, N, n, n);\\n\\n\\n  /* Initialize array(s). */\\n  init_array (n, POLYBENCH_ARRAY(A));\\n\\n  /* Start timer. */\\n  polybench_start_instruments;\\n\\n  /* Run kernel. */\\n  kernel_seidel_2d (tsteps, n, POLYBENCH_ARRAY(A));\\n\\n  /* Stop and print timer. */\\n  polybench_stop_instruments;\\n  polybench_print_instruments;\\n\\n  /* Prevent dead-code elimination. All live-out data must be printed\\n     by the function call in argument. */\\n  polybench_prevent_dce(print_array(n, POLYBENCH_ARRAY(A)));\\n\\n  /* Be clean. */\\n  POLYBENCH_FREE_ARRAY(A);\\n\\n  return 0;\\n}\\n"}')
for p, text in FILES.items():
    os.makedirs(os.path.dirname(os.path.join(ROOT, 'rewrites', p)), exist_ok=True)
    open(os.path.join(ROOT, 'rewrites', p), 'w').write(text)
print(len(FILES), 'files')


In [ ]:
%%bash
cd "$P05_ROOT"
NVC=$(ls -d /opt/nvidia/hpc_sdk/Linux_x86_64/*/compilers/bin/nvc | sort -V | tail -1)
CC=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d .)
$NVC --version | grep -v '^$' | head -1
python3 gpu_run.py --pb polybench --src rewrites --cc $NVC --omp-flags "-mp=gpu -gpu=cc$CC" \
  --threads $(nproc) --runs 3 --dataset LARGE_DATASET --csv polybench-t4.csv


In [ ]:
import csv, math, os
CSV = os.path.join(os.environ['P05_ROOT'], 'polybench-t4.csv')
rows = list(csv.DictReader(open(CSV)))
t = {(r['kernel'], r['variant']): float(r['time_s']) for r in rows if r['status'] == 'ok' and r['time_s']}
bad = [(r['kernel'], r['variant'], r['status'], r['correct']) for r in rows
       if r['status'] != 'ok' or r['correct'] not in ('', 'exact', 'tolerance')]
print('not ok / not correct:', bad or 'none')
for v in ('p05', 'p05-naive', 'p05-warm', 'p05-naive-warm'):
    xs = [t[(k, 'seq')] / t[(k, v)] for (k, vv) in t if vv == v and (k, 'seq') in t]
    if xs:
        print(f"{v:15} geomean speedup over seq {math.exp(sum(map(math.log, xs)) / len(xs)):.2f} ({len(xs)} kernels)")
try:
    from google.colab import files
    files.download(CSV)
except ImportError:  # Kaggle: files in /kaggle/working are the version's Output
    from IPython.display import FileLink, display
    print('CSV at', CSV)
    display(FileLink(os.path.relpath(CSV)))
